# SQL

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook)*



*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

SQL is the shared language of every warehouse and lakehouse engine, and the most predictable part of a data-engineering interview. The questions below cover the query patterns real pipelines use - window functions, CTEs, joins, dedupe, anti-joins and the plan reading that makes performance explainable - with runnable examples against an in-memory SQLite database.

## Contents

<details>
<summary>43 sections + practice appendix - click to expand</summary>

- [Write sql query to get the second highest salary among all employees?](#write-sql-query-to-get-the-second-highest-salary-among-all-employees)
- [What is a pseudo-column?](#what-is-a-pseudo-column)
- [What are the reasons for denormalizing the data?](#what-are-the-reasons-for-denormalizing-the-data)
- [What is the feature in sql for writing if and else statements?](#what-is-the-feature-in-sql-for-writing-if-and-else-statements)
- [What is the difference between delete and truncate in sql?](#what-is-the-difference-between-delete-and-truncate-in-sql)
- [What is the difference between ddl and dml commands in sql?](#what-is-the-difference-between-ddl-and-dml-commands-in-sql)
- [Why do we use escape characters in sql queries?](#why-do-we-use-escape-characters-in-sql-queries)
- [What is the difference between primary key and unique key in sql?](#what-is-the-difference-between-primary-key-and-unique-key-in-sql)
- [What is the difference between inner join and outer join in sql?](#what-is-the-difference-between-inner-join-and-outer-join-in-sql)
- [What is the difference between where clause and having clause?](#what-is-the-difference-between-where-clause-and-having-clause)
- [How will you limit a mysql query to display only top 10 rows?](#how-will-you-limit-a-mysql-query-to-display-only-top-10-rows)
- [How do you upsert with MERGE in SQL?](#how-do-you-upsert-with-merge-in-sql)
- [What is sharding in sql?](#what-is-sharding-in-sql)
- [You wrote search engine that should retrieve 10 results at a time but at the same time you do like to know how many rows there are total.](#you-wrote-search-engine-that-should-retrieve-10-results-at-a-time-but-at-the-same-time-you-do-like-to-know-how-many-rows-there-are-total)
- [How would you write a query to select all teams that won either 2 and 4 and 6 or 8 games?](#how-would-you-write-a-query-to-select-all-teams-that-won-either-2-and-4-and-6-or-8-games)
- [How would you select all users whose phone number is null?](#how-would-you-select-all-users-whose-phone-number-is-null)
- [What is index?](#what-is-index)
- [What is union?](#what-is-union)
- [What is transaction?](#what-is-transaction)
- [What are the different types of sql commands?](#what-are-the-different-types-of-sql-commands)
- [Explain what is a subquery ?](#explain-what-is-a-subquery-)
- [How do you write a CTE and a recursive CTE in SQL?](#how-do-you-write-a-cte-and-a-recursive-cte-in-sql)
- [What are the constraints available in sql?](#what-are-the-constraints-available-in-sql)
- [What is a null value?](#what-is-a-null-value)
- [What is normalization?](#what-is-normalization)
- [What is stored procedure?](#what-is-stored-procedure)
- [List out the acid properties and explain?](#list-out-the-acid-properties-and-explain)
- [What is sql injection?](#what-is-sql-injection)
- [What is the difference between clustered and non-clustered indexes?](#what-is-the-difference-between-clustered-and-non-clustered-indexes)
- [What is collation?](#what-is-collation)
- [What are the advantages of views?](#what-are-the-advantages-of-views)
- [What is schema?](#what-is-schema)
- [What are the steps to take to improve performance of a poor performing query?](#what-are-the-steps-to-take-to-improve-performance-of-a-poor-performing-query)
- [What is a deadlock and what is a live lock?](#what-is-a-deadlock-and-what-is-a-live-lock)
- [How do you read a query plan with EXPLAIN?](#how-do-you-read-a-query-plan-with-explain)
- [How can you know that statistics should be updated?](#how-can-you-know-that-statistics-should-be-updated)
- [What is write ahead log?](#what-is-write-ahead-log)
- [What is the use of cascade constraints?](#what-is-the-use-of-cascade-constraints)
- [How do you implement one-to-one, one-to-many and many-to-many relationships while designing tables?](#how-do-you-implement-one-to-one-one-to-many-and-many-to-many-relationships-while-designing-tables)
- [Define candidate key and alternate key and composite key.](#define-candidate-key-and-alternate-key-and-composite-key)
- [Explain different isolation levels?](#explain-different-isolation-levels)
- [What are statistics under what circumstances they go out of date and how do you update them?](#what-are-statistics-under-what-circumstances-they-go-out-of-date-and-how-do-you-update-them)
- [How do window functions help in data engineering SQL?](#how-do-window-functions-help-in-data-engineering-sql)

- *Practice appendix*: SQL interview questions, SQLite setup, exercises (merged below)

</details>

Tip: the links open the sections on GitHub; in Jupyter or Colab, use the notebook's outline panel in the sidebar instead.

## Write sql query to get the second highest salary among all employees?

Given Employee Table with two columns
ID, Salary 10, 2000
11, 5000
12, 3000

The modern answer uses a window function, which also handles ties and works when salaries repeat.

```sql
select distinct salary
from (
  select salary,
         dense_rank() over (order by salary desc) as rnk
  from employee
) ranked
where rnk = 2;
```

Trade-off: `dense_rank` returns the second distinct salary value, so equal salaries do not consume a rank. `row_number` would return the second row even when it ties the first, and `rank` would skip ranks after a tie. State which definition of "second highest" you are using before writing the query.

## There are multiple ways to get the second highest salary among all employees.

Option 1: Use Subquery
SELECT MAX(Salary) FROM Employee WHERE Salary NOT IN (SELECT MAX(Salary) FROM Employee );
In this approach, we are getting the maximum salary in a subquery and then excluding this from the rest of the resultset.
Option 2: Use Not equals
select MAX(Salary) from Employee WHERE Salary <> (select MAX(Salary) from Employee )
This is same as option 1 but we are using <> instead of NOT IN.

## How can we retrieve alternate records from a table in oracle?

We can use rownum and MOD function to retrieve the alternate records from a table.
To get Even number records:
SELECT * FROM (SELECT rownum, ID, Name FROM Employee) WHERE MOD(rownum,2)=0
To get Odd number records: SELECT * FROM (SELECT rownum, ID, Name FROM Employee)
WHERE MOD(rownum,2)=1

## Write sql query to find max salary and department name from each department.

Given Employee table with three columns
ID, Salary, DeptID
10, 1000, 2
20, 5000, 3
30, 3000, 2

Department table with two columns: ID, DeptName
1, Marketing
2, IT
3, Finance

This is a trick question. There can be some department without any employee. So we have to ask interviewer if they expect the name of such Department also in result.
If yes then we have to join Department table with Employee table by using foreign key DeptID. We have to use LEFT OUTER JOIN to print all the departments.
Query would be like as follows:
SELECT d.DeptName, MAX(e.Salary) FROM Department d LEFT OUTER JOIN Employee e ON e.DeptId = d.ID GROUP BY DeptName

## Write sql query to find records in table a that are not in table b without using not in operator.

Consider two tables
Table_A

10
20
30

Table_B
15
30
45
We can use MINUS operator in this case for Oracle and EXCEPT for SQL Server.
Query will be as follows:
SELECT * FROM Table_A MINUS SELECT * FROM Table_B

The same "rows in A with no match in B" result is called an anti-join.

```sql
-- null-safe and usually the fastest
select a.*
from table_a a
where not exists (select 1 from table_b b where b.id = a.id);

-- left anti-join
select a.*
from table_a a
left join table_b b on b.id = a.id
where b.id is null;

-- set operators (deduplicate as a side effect)
select id from table_a
except
select id from table_b;          -- Oracle spells this minus
```

Trade-off: `not exists` and the left anti-join compile to the same plan on most engines, while `except`/`minus` also removes duplicate rows. Avoid `not in` whenever the subquery column can be null, because a single null makes the predicate unknown and the query returns no rows.

## What is the result of following query?

SELECT CASE WHEN null = null THEN "True" ELSE "False" END AS Result;
Answer: In SQL null can not be compared with itself. There fore null = null is not true. We can compare null with a non-null value to check whether a value is not null.
Therefore the result of above query is False.
The correct way to check for null is to use IS NULL clause.
Following query will give result True. SELECT CASE WHEN null IS NULL THEN "True" ELSE "False" END AS Result;

```sql
select case when null = null then 'True' else 'False' end as result;  -- False
select case when null is null then 'True' else 'False' end as result; -- True
select case when null <> null then 'True' else 'False' end as result; -- False
```

`null` is the absence of a value, so every comparison with it evaluates to `unknown`, and `where`/`case` treat `unknown` as not true. Use `is null`, `is not null`, `coalesce`, or `nullif` for null-safe logic, and remember that `not (null = null)` is also `unknown`.

## Write sql query to find employees that have same name and email.

Employee table:
ID NAME EMAIL
10 John jbaldwin
20 George gadams
30 John jsmith

This is a simple question with one trick. The trick here is to use Group by on two columns Name and Email.
Query would be as follows: SELECT name, email, COUNT(*) FROM Employee GROUP BY name, email HAVING COUNT(*) > 1

## Write sql query to find max salary from each department.

Given Employee table with three columns
ID, Salary, DeptID 10, 1000, 2
20, 5000, 3
30, 3000, 2

We can first use group by DeptID on Employee table and then get the Max salary from each Dept group.
SELECT DeptID, MAX(salary) FROM Employee GROUP BY DeptID

```sql
select dept_id, max(salary) as max_salary
from employee
group by dept_id;
```

To return the whole row of the top earner per department, rank instead of aggregating:

```sql
select dept_id, id, name, salary
from (
  select e.*,
         row_number() over (partition by dept_id order by salary desc) as rn
  from employee e
) t
where rn = 1;
```

Trade-off: `max()` gives one number per group and cannot return the employee name without a second join; the window version returns the full row and controls tie-breaking with `row_number` versus `rank`.

## Write sql query to get the nth highest salary among all employees.

Given Employee Table with two columns
ID, Salary 10, 2000
11, 5000
12, 3000

Option 1: Use Subquery We can use following sub query approach for this:
SELECT * FROM Employee emp1 WHERE (N-1) = ( SELECT COUNT(DISTINCT(emp2.salary)) FROM Employee emp2 WHERE emp2.salary > emp1.salary)
Option 2: Using Rownum in Oracle
SELECT * FROM (SELECT emp.*,row_number() OVER (ORDER BY salary DESC) rnum FROM Employee emp) WHERE rnum = n;

Ranking functions are the cleanest answer. `row_number` skips ties, `rank` leaves gaps, and `dense_rank` does not, which changes what the "nth highest" means when salaries tie.

```sql
-- nth distinct salary
select distinct salary
from (
  select salary, dense_rank() over (order by salary desc) as rnk
  from employee
) ranked
where rnk = :n;

-- nth row by salary
select salary
from (
  select salary, row_number() over (order by salary desc) as rn
  from employee
) ranked
where rn = :n;
```

Trade-off: `dense_rank` is the usual interview answer because duplicate salaries should not consume a rank. The correlated-subquery form `(select count(distinct salary) ...)` still works but rescans the table per row.

## How can you find 10 employees with odd number as employee id?

In Oracle we can use Top to limit the number of records. We can also use Rownum < 11 to get the only 10 or less number of records.To find the Odd number Employee ID, we can use % function.
Sample Query with TOP:
SELECT TOP 10 ID FROM Employee WHERE ID % 2 = 1;
Sample Query with ROWNUM:
SELECT ID FROM Employee WHERE ID % 2 = 1 AND ROWNUM < 11;

## Write sql query to get the names of employees whose date of birth is between 01/01/1990 to 31/12/2000.

This SQL query appears a bit tricky. We can use BETWEEN clause to get all the employees whose date of birth lies between two given dates.
Query will be as follows:
SELECT EmpName FROM Employees WHERE birth_date BETWEEN "01/01/1990" AND "31/12/2000"
Remember BETWEEN is always inclusive of both the dates.

## Write sql query to get the quarter from date.

Answer: We can use to_char function with "Q" option for quarter to get quarter from a date.
Use TO_CHAR with option "Q" for Quarter
SELECT TO_CHAR(TO_DATE("3/31/2016", "MM/DD/YYYY"), "Q") AS quarter FROM DUAL

```sql
-- Oracle
select to_char(date '2026-03-31', 'Q') as quarter from dual;

-- portable
select extract(quarter from date '2026-03-31') as quarter;        -- Postgres, Snowflake, BigQuery
select date_trunc('quarter', date '2026-03-31') as quarter_start; -- first day of the quarter
select quarter('2026-03-31') as quarter;                          -- MySQL
```

Trade-off: `extract` returns a number, which is easy to compare but awkward to label; `date_trunc('quarter', ...)` returns a date and is usually what a date-dimension join wants. Use single quotes for string and date literals — double quotes are identifiers in ANSI SQL and Oracle.

## Write query to find employees with duplicate email.

Employee table:
ID NAME EMAIL
10 John jsmith
20 George gadams
30 Jane jsmith
We can use Group by clause on the column in which we want to find duplicate values.
Query would be as follows:
SELECT name, COUNT(email) FROM Employee GROUP BY email HAVING ( COUNT(email) > 1 )

```sql
select email, count(*) as occurrences
from employees
group by email
having count(*) > 1
order by occurrences desc;
```

To keep the newest row and delete the rest:

```sql
delete from employees
where id in (
  select id from (
    select id,
           row_number() over (partition by email order by updated_at desc) as rn
    from employees
  ) ranked
  where rn > 1
);
```

Trade-off: the window-function delete is portable; MySQL refuses to delete from a table that a subquery also selects from unless you wrap it in a derived table (as above) or stage the ids in a temporary table.

## Write a query to find all employee whose name contains the word "rich", regardless of case.

E.g. Rich, RICH, rich.
We can use UPPER function for comparing the both sides with uppercase.
SELECT * FROM Employees WHERE UPPER(emp_name) like "%RICH%"

## What is a pseudo-column?

A Pseudocolumn is like a table column, but it is not stored in the same table. We can select from a Pseudocolumn, but we can not insert, update or delete on a Pseudocolumn. A Pseudocolumn is like a function with no arguments. Two most popular Pseudocolumns in Oracle are ROWID and ROWNUM. NEXTVAL and CURRVAL are also pseudo columns.

## What are the reasons for denormalizing the data?

We de-normalize data when we need better performance. Sometimes there are many joins in a query due to highly normalized data. In that case, for faster data retrieval it becomes essential to de-normalize data.

## What is the feature in sql for writing if and else statements?

In SQL, we can use CASE statements to write If/Else statements. We can also use DECODE function in Oracle SQL for writing simple If/Else logic.

```sql
select
  order_id,
  case
    when amount >= 1000 then 'high'
    when amount >= 100  then 'medium'
    else 'low'
  end as amount_band,
  case status when 'C' then 'completed'
              when 'X' then 'cancelled'
              else 'other' end as status_label
from orders;
```

Trade-off: searched `case` (with conditions) is portable and readable; Oracle's `decode` is shorter for equality-only logic but Oracle-specific. `case` works in `select`, `where`, `order by`, and inside aggregates, which is how you build pivot-style columns such as `sum(case when status = 'paid' then amount else 0 end)`.

**CASE is SQL's if/else and it works row by row.** It evaluates conditions in order and returns the first matching result, so put the most specific condition first. Without an `ELSE`, the default is `NULL`.

```sql
SELECT name, salary,
       CASE
           WHEN salary >= 150000 THEN 'high'
           WHEN salary >= 100000 THEN 'mid'
           ELSE 'standard'
       END AS band
FROM employees;
```

Order matters: listing `salary >= 100000` before `salary >= 150000` would label everyone as `mid`. `CASE` also composes with aggregation to pivot, and it short-circuits, so a guard branch can protect a division from zero. The shorthand `CASE status WHEN 'a' THEN 1 END` is fine when every branch tests equality on one expression.

## What is the difference between delete and truncate in sql?

Main differences between DELETE and TRUNCATE commands are:
DML vs. DDL: DELETE is a Data Manipulation Language (DML) command. TRUNCATE is a Data Definition Language (DDL) command.

Number of Rows: We can use DELETE command to remove one or more rows from a table. TRUNCATE command will remove all the rows from a table.
WHERE clause: DELETE command provides support for WHERE clause that can be used to filter the data that we want to delete. TRUNCATE command can only delete all the rows. There is no WHERE clause in TRUNCATE command.
Commit: After DELETE command we have to issue COMMIT or ROLLBACK command to confirm our changes. After TRUNCATE command there is no need to run COMMIT. Changes done by TRUNCATE command can not be rolled back.

```sql
delete from orders where order_date < date '2020-01-01';  -- row by row, logged, filterable
truncate table orders;                                    -- all rows, minimal logging
```

Trade-off: `delete` fires triggers, honors `where`, and can be rolled back; `truncate` is far faster on large tables but deletes everything, resets identity values in MySQL, and in SQL Server fails when a foreign key references the table.

## What is the difference between ddl and dml commands in sql?

Main differences between Data Definition Language (DDL) and Data Manipulation Language (DML) commands are:
+ DDL vs. DML: DDL statements are used for creating and defining the Database structure. DML statements are used for managing data within Database. Sample Statements: DDL statements are CREATE, ALTER, DROP, TRUNCATE, RENAME etc. DML statements are SELECT, INSERT, DELETE, UPDATE, MERGE, CALL etc.
+ Number of Rows: DDL statements work on whole table. CREATE will a create a new table. DROP will remove the whole table. TRUNCATE will delete all records in a table. DML statements can work on one or more rows. INSERT can insert one or more rows. DELETE can remove one or more rows.
+ WHERE clause: DDL statements do not have a WHERE clause to filter the data. Most of DML statements support filtering the data by WHERE clause.
+ Commit: Changes done by a DDL statement can not be rolled back. So there is no need to issue a COMMIT or ROLLBACK command after DDL statement. We need to run COMMIT or ROLLBACK to confirm our changed after running a DML statement.
+ Transaction: Since each DDL statement is permanent, we can not run multiple DDL statements in a group like Transaction. DML statements can be run in a Transaction. Then we can COMMIT or ROLLBACK this group as a transaction. E.g. We can insert data in two tables and commit it together in a transaction.
+ Triggers: After DDL statements no triggers are fired. But after DML statements relevant triggers can be fired.

## Why do we use escape characters in sql queries?

In SQL, there are certain special characters and words that are reserved for special purpose. E.g. & is a reserved character. When we want to use these special characters in the context of our data, we have to use Escape characters to pass the message to database to interpret these as non Special / non Reserved characters.

## What is the difference between primary key and unique key in sql?

Main differences between Primary key and Unique key in SQL are:
+ Number: There can be only one Primary key in a table. There can be more than one Unique key in a table.
+ Null value: In some DBMS Primary key cannot be NULL. E.g. MySQL adds NOT NULL to Primary key. A Unique key can have null values.
+ Unique Identifier: Primary Key is a unique identifier of a record in database table. Unique key can be null and we may not be able to identify a record in a unique way by a unique key
+ Changes: It is not recommended to change a Primary key. A Unique key can be changed much easily.
+ Usage: Primary Key is used to identify a row in a table. A Unique key is used to prevent duplicate non-null values in a column.

**Keys are constraints first, indexes second.** A primary key is `NOT NULL` plus `UNIQUE`, one per table, and in InnoDB and SQL Server it is usually the clustered index that defines physical row order. A unique key allows `NULL` (one or more per engine) and exists to enforce a rule on an alternate key.

```sql
CREATE TABLE customers (
    customer_id   BIGINT PRIMARY KEY,
    email         VARCHAR(255) NOT NULL,
    tax_id        VARCHAR(50),
    CONSTRAINT uq_customers_email UNIQUE (email),
    CONSTRAINT uq_customers_tax   UNIQUE (tax_id)
);
```

Common mistake: treating a unique index as free. Every constraint is maintained on write and consumes space, and a natural key such as an email changes, which forces updates to every child row. Prefer a stable surrogate key, and use a separate unique constraint for the business key.

## What is the difference between inner join and outer join in sql?

Let say we have two tables X and Y.
The result of an INNER JOIN of X and Y is X intersect. It is the INNER overlapping intersection part of a Venn diagram. The result of an OUTER JOIN of X and Y is X union Y. It is the OUTER parts of a Venn diagram. E.g. Consider following two tables, with just one column x and y:

x | y
- -|- -
  10 | 30
  20 | 40
  30 | 50
  40 | 60
  In above tables (10,20) are unique to table X, (30,40) are common, and (50,60) are unique to table Y.
  INNER JOIN An INNER JOIN by using following query will give the intersection of the two tables X and Y. The intersection is the common data between these tables.
  select * from X INNER JOIN Y on X.x =Y.y;
  x | y
  --+--
  30 | 30
  40 | 40

OUTER JOIN
A full OUTER JOIN by using following query will us the union of X and Y. It will have all the rows in X and all the rows in Y. If some row in X has not corresponding value in Y, then Y side will be null, and vice versa.
select * from X FULL OUTER JOIN Y on X.x = Y.y;

x| y
----- + -----
10 | null
20 | null

30 | 30
40 | 40
null | 60
null | 50

```sql
-- sample tables: customers(id, name), orders(id, customer_id, amount)
select c.name, o.amount
from customers c
inner join orders o on o.customer_id = c.id;      -- only matching pairs

select c.name, o.amount
from customers c
left join orders o on o.customer_id = c.id;       -- every customer, nulls when no order

select c.name, o.amount
from customers c
full outer join orders o on o.customer_id = c.id; -- every customer and every order
```

Trade-off: `inner join` lets the optimizer discard non-matching rows early and is the cheapest. A `full outer join` usually forces a sort or hash of both inputs. MySQL has no `full outer join`, so it is emulated with `left join union all right join ... where not exists`; Postgres, Snowflake, and BigQuery support it directly. State whether unmatched rows must be preserved before choosing the join.

## What is the difference between left outer join and right outer join?

Let say we have two tables X and Y.
The result of an LEFT OUTER JOIN of X and Y is all rows of X and common rows between X and Y.
The result of an RIGHT OUTER JOIN of X and Y is all rows of Y and common rows between X and Y.
E.g.
Consider following two tables, with just one column x and y:

x| y
- - -|- - 10 | 30
    20 | 40
    30 | 50
    40 | 60

In above tables (10,20) are unique to table X, (30,40) are common, and (50,60) are unique to table Y.
LEFT OUTER JOIN
A left OUTER JOIN by using following query will give us all rows in X and common rows in X and Y.
select * from X LEFT OUTER JOIN Y on X.x = Y.y;
x| y
-- -+-----
10 | null
20 | null
30 |30
40 |40
RIGHT OUTER JOIN
A right OUTER JOIN by using following query will give all rows in Y and common rows in X and Y.
select * from X RIGHT OUTER JOIN Y on X.x = Y.y;
x| y
----- +----
30 | 30
40 | 40
null | 50
null | 60

```sql
-- left join: the left table is preserved
select c.name, count(o.id) as orders
from customers c
left join orders o on o.customer_id = c.id
group by c.name;

-- right join: the right table is preserved (same as swapping and using left join)
select c.name, o.id
from orders o
right join customers c on o.customer_id = c.id;
```

Trade-off: a `right join` is a `left join` with the tables swapped. Most teams standardize on `left join` because the preserved table comes first, which makes "did this filter drop rows?" easier to answer in review. `count(o.id)` counts only real matches, while `count(*)` would count the null-padded rows as one.

## What is the difference between where clause and having clause?

We use where clause to filter elements based on some criteria on individual records of a table.
E.g. We can select only employees with first name as John.
SELECT ID, Name FROM Employee WHERE name = "John"
We use having clause to filter the groups based on the values of aggregate functions.
E.g. We can group by department and only select departments that have more than 10 employees.
SELECT deptId, count(1) FROM Employee GROUP BY deptId HAVING count(*) > 10.

```sql
select dept_id, count(*) as employees, avg(salary) as avg_salary
from employees
where hire_date >= date '2020-01-01'   -- filters rows before grouping
group by dept_id
having count(*) > 10                   -- filters groups after aggregation
order by avg_salary desc;
```

Trade-off: `where` runs before `group by` and can use indexes; `having` runs after aggregation and works on aggregate results or group keys. Pushing every non-aggregate predicate into `where` lets the engine aggregate fewer rows, which is the most common single-query optimization.

**The execution order is the whole story.** SQL runs `FROM` and `JOIN`, then `WHERE`, then `GROUP BY`, then `HAVING`, then `SELECT`, then `ORDER BY`, then `LIMIT`. Because `WHERE` runs before grouping, it filters individual rows and can use an index. Because `HAVING` runs after grouping, it filters whole groups and can reference aggregates.

```sql
SELECT dept_id, COUNT(*) AS headcount, AVG(salary) AS avg_salary
FROM employees
WHERE hire_date >= DATE '2020-01-01'   -- rows, before grouping
GROUP BY dept_id
HAVING COUNT(*) > 3;                   -- groups, after aggregation
```

Common mistake: filtering on an aggregate in `WHERE`, which errors. The other: moving a row filter into `HAVING`, which still produces the right answer but aggregates far more rows than necessary and cannot use an index.

## How will you limit a mysql query to display only top 10 rows?

We can use LIMIT clause in MySQL to limit a query to a range of rows.
Following query will give top 10 rows from the table with table_name:
SELECT * FROM table_name LIMIT 0,10;
Following query will give 6 rows starting from the 4th row in table with table_name:
SELECT * FROM table_name LIMIT 3,6;

## How do you upsert with MERGE in SQL?

```sql
merge into analytics.customers as t
using staging.customers as s
  on t.customer_id = s.customer_id
when matched then update set
  t.name = s.name,
  t.email = s.email,
  t.updated_at = s.updated_at
when not matched then insert (customer_id, name, email, updated_at)
values (s.customer_id, s.name, s.email, s.updated_at);
```

Dialect notes: Postgres 15+ and SQL Server support `merge`; MySQL uses `insert ... on duplicate key update`; SQLite uses `insert ... on conflict do update`; BigQuery and Snowflake support `merge` but require the source to be unique per key or the statement fails. Always deduplicate `staging` first. Trade-off: `merge` is atomic and self-documenting, but an update-heavy merge rewrites many rows; for append-only facts a plain insert is cheaper.

**An upsert inserts or updates in one atomic statement.** The portable spelling is `MERGE`; Postgres and SQLite offer the shorter `INSERT ... ON CONFLICT`. Doing it in one statement avoids the read-then-write race that a `SELECT` followed by an `INSERT` or `UPDATE` leaves open.

```sql
-- Postgres / SQLite
INSERT INTO customers(customer_id, email, updated_at)
VALUES (1, 'ada@example.com', now())
ON CONFLICT (customer_id)
DO UPDATE SET email = EXCLUDED.email, updated_at = EXCLUDED.updated_at;

-- Portable MERGE
MERGE INTO customers AS t
USING staging AS s ON t.customer_id = s.customer_id
WHEN MATCHED THEN UPDATE SET t.email = s.email
WHEN NOT MATCHED THEN INSERT (customer_id, email) VALUES (s.customer_id, s.email);
```

Common mistake: a `MERGE` source that has duplicate keys, which makes the statement ambiguous and errors in some engines or updates twice in others. Deduplicate the source first, and wrap it in a transaction so the whole batch commits or none of it does.

## What is sharding in sql?

The process of breaking up large tables into smaller chunks (called shards) that are spread across multiple servers is called Sharding.
The advantage of Sharding is that since the sharded database is generally much smaller than the original; queries, maintenance, and all other tasks are much faster.

## You wrote search engine that should retrieve 10 results at a time but at the same time you do like to know how many rows there are total.

SELECT SQL_CALC_FOUND_ROWS page_title FROM web_pages LIMIT 1,10; SELECT FOUND_ROWS(); The second query (not that COUNT() is never used) will tell you how many results there're total, so you can display a phrase "Found 13,450,600 results, displaying 1-10". Note that FOUND_ROWS does not pay attention to the LIMITs you specified and always returns the total number of rows affected by query.

```sql
select count(*) over () as total_rows,
       book_id, title, score
from search_results
order by score desc
limit 10 offset 20;
```

A window `count(*) over ()` returns the total alongside each page in one query. Trade-off: on very large result sets the window forces the engine to scan the full result, so a separate `select count(*)` (or an approximate count in BigQuery/Spark) can be cheaper. Keyset pagination (`where score < :last_score`) avoids the growing cost of a large `offset`.

## How would you write a query to select all teams that won either 2 and 4 and 6 or 8 games?

SELECT team_name FROM teams WHERE team_won IN (2, 4, 6, 8)

## How would you select all users whose phone number is null?

SELECT user_name FROM users WHERE ISNULL(user_phonenumber);

```sql
select * from users where phone_number is null;
-- never: where phone_number = null   (always unknown, returns no rows)

-- treat empty strings as missing too
select * from users where phone_number is null or phone_number = '';
```

Trade-off: `is null` can use a B-tree index in Postgres and Oracle; `= null` is always unknown. For a "missing or blank" rule, normalize blank strings to null at load time so every query can use the simple `is null` form.

**NULL is unknown, not empty.** Every comparison with `NULL` returns `UNKNOWN`, and `WHERE` keeps only `TRUE`, so `= NULL` and `<> NULL` both return nothing. Use `IS NULL` and `IS NOT NULL`.

```sql
WHERE phone_number IS NULL          -- correct
WHERE phone_number = NULL           -- always empty
WHERE COALESCE(phone_number, '') = ''  -- both NULL and blank, not sargable
```

Three-valued logic propagates: `TRUE OR NULL` is `TRUE`, `FALSE AND NULL` is `FALSE`, and everything else that touches `NULL` is `UNKNOWN`. Two traps follow. `NOT IN (SELECT ...)` returns no rows if the subquery returns a single `NULL`, so prefer `NOT EXISTS`. And `COUNT(*)` counts rows while `COUNT(column)` skips `NULL`, which is why the two disagree by exactly the number of missing values.

## What is index?

An index is a single column or multiple columns defined to have values pre-sorted to speed up data retrieval speed.

**An index is a sorted side structure.** The default B-tree maps key values to row locations, so an equality lookup is a few page reads instead of a full scan, and a range or `ORDER BY` becomes a walk along linked leaves. Hash indexes serve equality only; bitmap indexes suit low-cardinality columns in analytical stores.

```sql
CREATE INDEX idx_orders_customer_date ON orders(customer_id, order_date);
```

The costs are real: each index consumes disk and slows every insert, update, and delete because it must be maintained. A composite index is usable only through a leftmost prefix, so `(customer_id, order_date)` helps a query on `customer_id` but not one on `order_date` alone. Index the columns you filter and join on, keep the number small, and drop indexes nothing uses.

## What is join?

Join is data retrieval operation that combines rows from multiple tables under certain matching conditions to form a single row.

A join combines rows from two inputs using a predicate. Beyond the textbook inner and outer joins, data engineers use semi-joins, anti-joins, and self-joins.

```sql
-- semi-join: customers with at least one order, without duplicating customers
select c.*
from customers c
where exists (select 1 from orders o where o.customer_id = c.id);

-- anti-join: customers with no orders
select c.*
from customers c
where not exists (select 1 from orders o where o.customer_id = c.id);

-- self-join: employee and manager from the same table
select e.name as employee, m.name as manager
from employees e
left join employees m on m.id = e.manager_id;
```

Trade-off: `exists` expresses "at least one match" without multiplying rows the way a plain join can. A self-join needs an alias on both sides and an index on the join key, or the table is scanned twice.

**Mechanics of each join type, in one place.** An inner join keeps only matching pairs. A left join keeps every left row and pads the right side with `NULL` where there is no match; a right join is the mirror image. A full outer join keeps unmatched rows from both sides. A cross join pairs everything with everything and is almost always a mistake outside a deliberate dimension expansion.

```sql
SELECT c.customer_id, o.order_id
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE o.order_id IS NULL;   -- anti-join: customers with no orders
```

The common mistake is adding a right-side filter to the `WHERE` clause of a left join, which silently turns it back into an inner join. Put the condition in the `ON` clause to keep the unmatched left rows:

```sql
-- inner join in disguise: NULL status rows are dropped
LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE o.status = 'completed'

-- true left join: filter applies only to matches
LEFT JOIN orders o ON o.customer_id = c.customer_id AND o.status = 'completed'
```

## What is union?

Join is data retrieval operation that combines multiple query outputs of the same structure into a single output. By default the MySQL UNION removes all duplicate rows from the result set even if you don't explicit using DISTINCT after the keyword UNION.
SELECT customerNumber id, contactLastname name FROM customers UNION SELECT employeeNurrber id, firstname name FROM employees

id name
103 Schmitt
112 King
114 Ferguson
119 Labrune
121 Bergulfsen

**Set operations combine result sets, not tables.** The shapes must match: same number of columns and compatible types. `UNION` removes duplicates with a sort or hash; `UNION ALL` concatenates without work; `INTERSECT` keeps rows in both inputs; `EXCEPT` (or `MINUS` in Oracle) subtracts.

```sql
SELECT email FROM customers
UNION            -- deduplicated
SELECT email FROM leads;

SELECT email FROM customers
UNION ALL        -- keeps duplicates, no sort
SELECT email FROM leads;
```

Common mistake: using `UNION` in ETL where the inputs are already disjoint, paying for a useless deduplication over millions of rows. `UNION ALL` is almost always the right default. Also note that `UNION` deduplicates the whole row, so it can hide a duplicated key that you would rather catch.

## What is transaction?

A transaction is a logical unit of work requested by a user to be applied to the database objects. MySQL server introduces the transaction concept to allow users to group one or more SQL statements into a single transaction, so that the effects of all the SQL statements in a transaction can be either all committed (applied to the database) or all rolled back (undone from the database).

## What is commit?

Commit is a way to terminate a transaction with all database changes to be saved permanently to the database server.

## What is rollback?

Rollback is a way to terminate a transaction with all database changes not saving to the database server.

## What are the different types of sql commands?

SQL commands are segregated into following types:
+ DDL – Data Definition Language
+ DML – Data Manipulation Language
+ DQL – Data Query Language
+ DCL – Data Control Language
+ TCL – Transaction Control Language

## Explain what is a subquery ?

A Subquery is a SQL query within another query. It is a subset of a Select statement whose return values are used in filtering the conditions of the main query.

```sql
-- scalar subquery
select (select max(amount) from orders) as max_amount;

-- correlated subquery: conceptually runs once per outer row
select o.*
from orders o
where o.amount > (select avg(amount) from orders x where x.customer_id = o.customer_id);

-- same logic with a window function, usually one pass
select *
from (
  select o.*, avg(amount) over (partition by customer_id) as customer_avg
  from orders o
) t
where amount > customer_avg;
```

Trade-off: correlated subqueries are readable but can execute once per row; a window function or a join to a pre-aggregated CTE lets the engine do one pass. Use `exists` for existence tests because it can stop at the first match.

**Subqueries and joins are two spellings of the same thing.** A scalar subquery returns one value and can appear anywhere an expression can. A correlated subquery references the outer row and is conceptually evaluated per row, which is why it can be slow; the optimiser often rewrites it into a join or a semi-join anyway.

```sql
-- scalar subquery
SELECT (SELECT MAX(amount) FROM orders) AS max_amount;

-- correlated: employees above their own department average
SELECT name FROM employees e
WHERE salary > (SELECT AVG(salary) FROM employees x WHERE x.dept_id = e.dept_id);

-- the same idea as a join to a pre-aggregated CTE, usually faster
WITH dept_avg AS (SELECT dept_id, AVG(salary) AS avg_salary FROM employees GROUP BY dept_id)
SELECT e.name FROM employees e
JOIN dept_avg d ON d.dept_id = e.dept_id
WHERE e.salary > d.avg_salary;
```

Use `EXISTS` rather than `IN` when the subquery can return `NULL`s, because `NOT IN` with a `NULL` returns nothing. Push the filter into the subquery when only a few rows matter, and check the plan: a correlated subquery that the optimiser cannot decorrelate runs once per outer row.

## How do you write a CTE and a recursive CTE in SQL?

```sql
-- plain CTE: stage a heavy aggregation, then join to it
with daily_revenue as (
  select order_date, customer_id, sum(amount) as revenue
  from orders
  where status = 'completed'
  group by order_date, customer_id
)
select order_date, sum(revenue) as revenue
from daily_revenue
group by order_date
order by order_date;

-- recursive CTE: walk the management hierarchy from the top
with recursive org as (
  select id, name, manager_id, 0 as depth
  from employees
  where manager_id is null
  union all
  select e.id, e.name, e.manager_id, o.depth + 1
  from employees e
  join org o on e.manager_id = o.id
)
select * from org order by depth, name;
```

The `recursive` keyword is required in Postgres, Snowflake, and BigQuery; SQL Server omits it. A recursive CTE can loop forever on cyclic data, so cap it with a `depth < 10` predicate or carry a visited-path array. Trade-off: a CTE is mainly a readability and reuse tool — most optimizers inline it, so on some engines it is not a performance boundary at all.

**A CTE is a named subquery, not a guarantee.** Whether it is materialised once or inlined into each reference is an engine decision. Postgres 12+ inlines non-recursive CTEs by default; add `MATERIALIZED` to force one evaluation and reuse, or `NOT MATERIALIZED` to let predicates push in.

```sql
WITH heavy AS MATERIALIZED (
    SELECT customer_id, SUM(amount) AS revenue FROM orders GROUP BY customer_id
)
SELECT * FROM heavy WHERE revenue > 10000;
```

Recursion turns a CTE into a loop. The anchor query runs once, then the recursive step runs repeatedly over the rows produced so far.

```sql
WITH RECURSIVE org AS (
    SELECT emp_id, manager_id, 1 AS depth FROM employees WHERE manager_id IS NULL
    UNION ALL
    SELECT e.emp_id, e.manager_id, org.depth + 1
    FROM employees e JOIN org ON e.manager_id = org.emp_id
)
SELECT * FROM org;
```

Common mistake: an unguarded cycle, which loops forever, and a recursive join key with no index, which makes every level a full scan. Add a `depth < n` guard and index the join key.

## What are the constraints available in sql?

Some constraints in SQL are : Primary Key, Foreign Key, Unique Key, SQL Not Null, Default, Check and Index constraint.

```sql
create table orders (
  id           bigint primary key,
  order_number varchar(32) not null unique,
  customer_id  bigint not null references customers(id) on delete cascade,
  amount       numeric(12,2) not null check (amount >= 0),
  status       varchar(16) not null default 'new'
               check (status in ('new','paid','shipped','cancelled')),
  created_at   timestamp not null default current_timestamp
);
```

- `primary key`: unique and not null, one per table.
- `unique`: uniqueness rule, nulls allowed in most engines.
- `foreign key`: referential integrity with `on delete`/`on update` actions.
- `check`: a row-level boolean rule.
- `not null` and `default`: presence and fallback rules.

Trade-off: constraints catch bad data at the source and give the optimizer facts it can trust, but validating a check on a huge existing table can scan it. Oracle and Postgres let you add a constraint `not valid` first and validate later.

## What is a unique key and primary key and foreign key?

+ A UNIQUE KEY constraint is used to ensure that there are no duplication values in the field/column.
+ A PRIMARY KEY constraint uniquely identifies each record in a database table. All columns participating in a primary key constraint must not contain NULL values.
+ A FOREIGN KEY is a key used to link two tables together. A FOREIGN KEY in a table is linked with the PRIMARY KEY of another table.

```sql
create table orders (
  order_id     bigint primary key,                               -- row identity
  order_number varchar(32) unique not null,                      -- alternate unique key
  customer_id  bigint not null references customers(customer_id) -- foreign key
);
```

- Primary key: uniquely identifies the row, cannot be null, one per table.
- Unique key: enforces uniqueness, nulls allowed in most engines.
- Foreign key: a value that must exist in the referenced table, with `on delete`/`on update` behavior.

All three are enforced by the database and backed by an index, so they also speed up joins and lookups.

## What is a null value?

A field with a NULL value is a field with no value. A NULL value is different from a zero value or a field that contains spaces. A field with a NULL value is one that has been left blank during record creation. Assume, there is a field in a table is optional and it is possible to insert a record without adding a value to the optional field then the field will be saved with a NULL value.

```sql
select
  count(*)                                                  as total_rows,
  count(phone_number)                                       as non_null_phones,
  sum(case when phone_number is null then 1 else 0 end)     as missing_phones
from users;
```

`null` means "unknown", not zero or blank: `null = null` is unknown, `null + 1` is null, and `not (null = null)` is also unknown. `count(*)` counts rows while `count(column)` skips nulls, which is the usual cause of totals that do not add up in a report.

## What is normalization?

Normalization is the process of table design to minimize the data redundancy.
There are different types of Noramalization forms in SQL:-
+ First Normal Form (1NF): It removes all duplicate columns from the table. Creates table for related data and identifies unique column values
+ First Normal Form (2NF): Follows 1NF and creates and places data subsets in an individual table and defines relationship between tables using primary key
+ Third Normal Form (3NF): Follows 2NF and removes those columns which are not related through primary key
+ Fourth Normal Form (4NF): Follows 3NF and do not define multi-valued dependencies. 4NF also known as BCNF

```sql
-- 1NF/2NF/3NF: every fact in exactly one place
create table customers (customer_id bigint primary key, name text, email text);
create table products  (product_id bigint primary key, name text, price numeric(12,2));
create table orders    (order_id bigint primary key, customer_id bigint references customers);
create table order_items (
  order_id   bigint references orders,
  product_id bigint references products,
  quantity   int not null,
  primary key (order_id, product_id)
);
```

1NF removes repeating groups, 2NF removes partial dependencies on part of a composite key, and 3NF removes transitive dependencies (a non-key column determined by another non-key column). Trade-off: normalization prevents update anomalies and saves storage, but analytics then needs more joins, so warehouses deliberately denormalize into star schemas for read performance.

**Normal forms remove redundancy; each one fixes a specific anomaly.** 1NF: atomic values, no repeating groups. 2NF: no partial dependency on part of a composite key. 3NF: no transitive dependency - a non-key column must not depend on another non-key column. BCNF is the stricter version of 3NF for overlapping candidate keys.

```sql
-- Violates 2NF: course_name depends only on course_id, not on the whole key
CREATE TABLE enrollments (
    student_id BIGINT,
    course_id  BIGINT,
    course_name VARCHAR(200),
    grade      VARCHAR(2),
    PRIMARY KEY (student_id, course_id)
);

-- 2NF: move the partial dependency out
CREATE TABLE courses (course_id BIGINT PRIMARY KEY, course_name VARCHAR(200));
CREATE TABLE enrollments (
    student_id BIGINT, course_id BIGINT, grade VARCHAR(2),
    PRIMARY KEY (student_id, course_id)
);
```

Normalise to 3NF for OLTP, where writes must be correct. Denormalise deliberately for reads - star schemas, materialized views, pre-aggregated marts - because a join is often cheaper than an inconsistent copy.

## What is stored procedure?

A Stored Procedure is a collection of SQL statements that have been created and stored in the database to perform a particular task. The stored procedure accepts input parameters and processes them and returns a single value such as a number or text value or a result set (set of rows).

```sql
delimiter //
create procedure credit_account(in p_account_id bigint, in p_amount numeric(12,2))
begin
  update accounts set balance = balance + p_amount where id = p_account_id;
  insert into account_ledger (account_id, amount, created_at)
  values (p_account_id, p_amount, current_timestamp);
end //
delimiter ;

call credit_account(42, 100.00);
```

Syntax differs by engine: Postgres uses `create function ... language plpgsql`, SQL Server uses `create procedure ... as begin ... end`. Trade-off: procedures keep logic close to the data and cut round trips, but they are harder to version, test, and review than SQL models in dbt or Spark; most modern data teams keep procedural logic for operational tasks only.

**A stored procedure is database-side logic, executed where the data lives.** It can take parameters, run several statements, branch, loop, and manage transactions, which makes it a fit for bulk maintenance and tight loops that would otherwise move data to the client.

```sql
CREATE PROCEDURE credit_account(IN p_account BIGINT, IN p_amount NUMERIC)
LANGUAGE plpgsql AS $$
BEGIN
    UPDATE accounts SET balance = balance + p_amount WHERE account_id = p_account;
    INSERT INTO account_audit(account_id, amount, changed_at)
    VALUES (p_account, p_amount, now());
END;
$$;
```

The trade-offs: logic in the database is harder to version, test, and review than application code, and each engine's procedural language is different. Prefer a plain SQL function or a view when the logic is a single query, and reserve procedures for multi-statement operations that must run atomically near the data.

## List out the acid properties and explain?

Following are the four properties of ACID. These guarantees that the database transactions are processed reliably.
+ Atomicity
+ Consistency
+ Isolation
+ Durability

```sql
begin;
update accounts set balance = balance - 100 where id = 1;
update accounts set balance = balance + 100 where id = 2;
commit;   -- both updates land, or rollback undoes both
```

- Atomic: all statements in the transaction take effect or none do.
- Consistent: constraints hold before and after, so the transfer cannot create or destroy money.
- Isolated: concurrent transactions do not see each other's uncommitted changes (isolation level controls how strict this is).
- Durable: once committed, the change survives a crash, usually because a write-ahead log was flushed first.

In practice isolation is the property teams relax for throughput; durability is the one you never relax for money movement.

## What is sql injection?

SQL Injection is a type of database attack technique where malicious SQL statements are inserted into an entry field of database such that once it is executed the database is opened for an attacker. This technique is usually used for attacking Data-Driven Applications to have an access to sensitive data and perform administrative tasks on databases.

In [1]:
import sqlite3

# A tiny in-memory table so the illustration is runnable.
connection = sqlite3.connect(":memory:")
cursor = connection.cursor()
cursor.execute("create table users (email text)")
email = "alice@example.com"

# vulnerable: user input is concatenated into the SQL text
cursor.execute("select * from users where email = '" + email + "'")

# safe: a bound parameter travels separately from the SQL statement
# (the placeholder differs by driver: %s for psycopg, ? for sqlite3)
cursor.execute("select * from users where email = ?", (email,))

print("both statements ran; only the second one is safe")


both statements ran; only the second one is safe


Parameter binding is the only reliable defense; escaping and blacklists are not. Also:
- grant the application role the least privilege it needs, so an injected statement cannot read other tables;
- never build table or column names from user input (they cannot be parameterized) — validate them against an allowlist;
- use an ORM or query builder that binds parameters by default.

## What is the difference between clustered and non-clustered indexes?

One table can have only one clustered index but multiple nonclustered indexes.
Clustered indexes can be read rapidly rather than non-clustered indexes.
Clustered indexes store data physically in the table or view and non-clustered indexes do not store data in table as it has separate structure from data row

## What is collation?

Collation is set of rules that check how the data is sorted by comparing it. Such as Character data is stored using correct character sequence along with case sensitivity, type, and accent.

## What are the advantages of views?

Advantages of Views:
Views restrict access to the data because the view can display selective columns from the table.
Views can be used to make simple queries to retrieve the results of complicated queries. For example, views can be used to query information from multiple tables without the user knowing.

```sql
-- expose only non-sensitive columns to the analyst role
create view v_customers_public as
select customer_id, segment, country, created_at
from customers;     -- intentionally omits email and phone

grant select on v_customers_public to analyst;
```

Trade-off: views simplify access control and freeze a stable interface, so a schema change lands in one place. But deeply nested views hide the real work from the optimizer and can block predicate pushdown, which is why warehouses often flatten them into tables or materialized views.

## What is schema?

A schema is a collection of database objects of a User.

## What are the steps to take to improve performance of a poor performing query?

Maximum use of indexes, stored procures should be done.
Avoid excessive use of complicated joins and cursors.
Avoid using conditional operators using columns of different tables.
Make use of computed columns and rewriting the query.

## What is a deadlock and what is a live lock?

Deadlock occur in interconnection n/w when group of process are unable to act because of waiting each other to release some resource.
live lock packets continue to move through n/w, but does not advance towards destination.

## What is blocking and how would you troubleshoot it?

Blocking occurs when two or more rows are locked by one SQL connection and a second connection to the SQL server requires a conflicting on lock on those rows. This results in the second connection to wait until the first lock is released.
Troubleshooting blocking:

SQL scripts can be written that constantly monitor the state of locking and blocking on SQL Server
The common blocking scenarios must be identified and resolved.
The scripts output must be checked constantly.
The SQL profilers data must be examined regularly to detect blocking.

## How do you read a query plan with EXPLAIN?

```sql
explain (analyze, buffers)          -- PostgreSQL
select c.name, sum(o.amount) as revenue
from customers c
join orders o on o.customer_id = c.id
where o.order_date >= date '2026-01-01'
group by c.name;
```

Read the plan from the innermost node out. Look for a sequential scan where you expected an index scan, an estimated row count far from the actual count (stale statistics), and `Sort`/`Hash`/`Shuffle` nodes that spill to disk. Engine notes: Postgres uses `explain (analyze, buffers)`, MySQL uses `explain format=json` or `explain analyze`, BigQuery exposes the execution plan in the console, and Snowflake shows the Query Profile. The plan is the evidence; rewriting without it is guesswork.

## How can you know that statistics should be updated?

Statistics represent the uniqueness for the indexes that are being used for selecting the records. This can make the query execution pretty efficient. The tables that we are dealing with if truncated and repopulated, there is a good chance that the indexes and statistics are out of sync and this is when we have to update the statistics. There are also other situations like when the table has been modified and lot of rows have been added recently or like when a server has been updated with different version of software. These also give us the reason to use the UPDATE_STATISTICS, DBCC SHOW_STATISTICS etc to update it accordingly.

## What is write ahead log?

The write-ahead log is the logging system that just updates the buffer cache of the database for the transactions and updates the logs and only then the actual changes are incorporated in the actual database. This is the reason why it is called “write ahead”. This helps in maintaining the consistency in the database. This can also be useful in getting the actual database values even in case of failures.

## What is the use of check points in the transaction logs?

The check points are restoration points that indicate the specific state of the database. When there is some failure in the database that is occurring before the next check point, the database can be reverted back to the previous check point and thus the database would still be consistent.

## What is the use of cascade constraints?

cascading is used for maintaining referencial integrity rules, which says that foreign key attributes values should be either subset of primary key values or null.

```sql
create table order_items (
  order_id   bigint not null,
  product_id bigint not null,
  constraint fk_order foreign key (order_id)
    references orders(id) on delete cascade
);

delete from orders where id = 42;   -- removes its order_items rows too
```

Trade-off: `on delete cascade` keeps child tables consistent automatically but can delete far more than intended, so use it only when the child cannot exist without the parent. Many teams prefer `on delete restrict` plus an explicit, auditable cleanup job.

## How do you implement one-to-one, one-to-many and many-to-many relationships while designing tables?

+ One-to-One relationship can be implemented as a single table and rarely as two tables with primary and foreign key relationships.
+ One-to-Many relationships are implemented by splitting the data into two tables with primary key and foreign key relationships.
+ Many-to-Many relationships are implemented using a junction table with the keys from both the tables forming the composite primary key of the junction table.

```sql
-- one-to-one: the child primary key is also the foreign key
create table user_profile (
  user_id    bigint primary key references users(id),
  bio        text,
  updated_at timestamp
);

-- one-to-many: the many side holds the foreign key
create table orders (
  id          bigint primary key,
  customer_id bigint not null references customers(id),
  amount      numeric(12,2) not null
);

-- many-to-many: a junction table with a composite primary key
create table order_products (
  order_id   bigint not null references orders(id),
  product_id bigint not null references products(id),
  quantity   int not null default 1,
  primary key (order_id, product_id)
);
```

Trade-off: the junction table's composite primary key prevents duplicate pairs for free; add an index on `product_id` as well if the relationship is queried from both directions.

## Define candidate key and alternate key and composite key.

A candidate key is one that can identify each row of a table uniquely. Generally a candidate key becomes the primary key of the table. If the table has more than one candidate key, one of them will become the primary key, and the rest are called alternate keys.
A key formed by combining at least two or more columns is called composite key.

```sql
create table employees (
  employee_id bigint primary key,             -- the chosen candidate key
  email       varchar(255) unique not null,   -- candidate key -> alternate key
  first_name  varchar(100) not null,
  last_name   varchar(100) not null,
  constraint ux_name unique (first_name, last_name, email)   -- composite key
);
```

A candidate key is any minimal set of columns that uniquely identifies a row. The primary key is the candidate you choose; the other candidates are alternate keys. A composite key is a key made of more than one column, and it is often also the primary key of a junction table.

## Explain different isolation levels?

An isolation level determines the degree of isolation of data between concurrent transactions. The default SQL Server isolation level is Read Committed. Here are the other isolation levels (in the ascending order of isolation): Read Uncommitted, Read Committed, Repeatable Read, Serializable. See SQL Server books online for an explanation of the isolation levels. Be sure to read about SET TRANSACTION ISOLATION LEVEL, which lets you customize the isolation level at the connection level.
CREATE INDEX myIndex ON myTable(myColumn)

| Level | Dirty read | Non-repeatable read | Phantom read |
|---|---|---|---|
| read uncommitted | yes | yes | yes |
| read committed | no | yes | yes |
| repeatable read | no | no | mostly prevented (InnoDB) |
| serializable | no | no | no |

```sql
begin transaction isolation level repeatable read;
select balance from accounts where id = 1;
-- another transaction commits an update here
select balance from accounts where id = 1;   -- same value: repeatable read
commit;
```

Trade-off: higher isolation prevents more anomalies but increases locking and retries. Postgres defaults to `read committed`, MySQL/InnoDB to `repeatable read`, and warehouses such as Snowflake and BigQuery give snapshot isolation without exposing the same knobs.

## What are statistics under what circumstances they go out of date and how do you update them?

Statistics determine the selectivity of the indexes. If an indexed column has unique values then the selectivity of that index is more, as opposed to an index with non-unique values. Query optimizer uses these indexes in determining whether to choose an index or not while executing a query.
Some situations under which you should update statistics:

## If there is significant change in the key values in the index?

If a large amount of data in an indexed column has been added, changed, or removed (that is, if the distribution of key values has changed), or the table has been truncated using the TRUNCATE TABLE statement and then repopulated
Database is upgraded from a previous version
Look up SQL Server books online for the following commands: UPDATE STATISTICS, STATS_DATE, DBCC SHOW_STATISTICS, CREATE STATISTICS, DROP STATISTICS, sp_autostats, sp_createstats, sp_updatestats.

## What is database replication and what are the different types of replication you can set up in sql server?

Replication is the process of copying/moving data between databases on the same or different servers. SQL Server supports the following types of replication scenarios:
+ Snapshot replication
+ Transactional replication (with immediate updating subscribers, with queued updating subscribers)
+ Merge replication

## How do window functions help in data engineering SQL?

Window functions calculate values across related rows without collapsing the result set like `group by` does. They are useful for deduplication, latest-record selection, running totals, ranking, sessionization, and change detection.

Example: keep the latest event per user.

```sql
select user_id, event_type, event_time
from (
  select
    user_id,
    event_type,
    event_time,
    row_number() over (
      partition by user_id
      order by event_time desc
    ) as rn
  from events
) ranked
where rn = 1;
```

In interviews, explain the three parts: `partition by` defines the group, `order by` defines sequence inside the group, and the function (`row_number`, `lag`, `sum`, etc.) computes the value.

**The frame is the part interviewers probe.** A window spec has three pieces: `PARTITION BY` (the groups), `ORDER BY` (the sequence inside each group), and the frame (which rows each output row sees). With an `ORDER BY` and no explicit frame, the default is `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`, which includes *peer* rows with the same ordering value. Write `ROWS` when you mean one physical row at a time.

```sql
SELECT order_id, order_date, amount,
       SUM(amount) OVER (PARTITION BY customer_id ORDER BY order_date
                         ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total,
       ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC) AS amount_rank,
       LAG(amount) OVER (PARTITION BY customer_id ORDER BY order_date) AS previous_amount
FROM orders;
```

`ROW_NUMBER` breaks ties arbitrarily and never repeats, `RANK` repeats on ties and skips numbers, and `DENSE_RANK` repeats without skipping. Use `ROW_NUMBER` for deduplication and `RANK` or `DENSE_RANK` when ties carry meaning. A frame of `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` gives a three-row moving average; forgetting the frame silently gives a running total instead.

## How do you deduplicate data with SQL?

Deduplication usually means defining the business key, ordering duplicate records by trust or recency, and keeping one winner. Avoid `select distinct` unless every column must match exactly, because real duplicates often differ in ingestion time, source file, or metadata.

Example:

```sql
with ranked as (
  select
    *,
    row_number() over (
      partition by order_id
      order by updated_at desc, ingested_at desc
    ) as rn
  from raw_orders
)
select *
from ranked
where rn = 1;
```

For production pipelines, write rejected duplicates to an audit table or track duplicate counts as a data quality metric.

## How do you write incremental SQL models?

An incremental model processes only new or changed records instead of rebuilding the whole table. The model needs a reliable watermark, such as `updated_at`, ingestion timestamp, source offset, or partition date. The target is usually updated with `merge`, partition overwrite, or append-only inserts depending on the table design.

Example:

```sql
merge into analytics.orders as target
using (
  select *
  from staging.orders
  where updated_at > (
    select coalesce(max(updated_at), timestamp '1900-01-01')
    from analytics.orders
  )
) as source
on target.order_id = source.order_id
when matched then update set
  status = source.status,
  amount = source.amount,
  updated_at = source.updated_at
when not matched then insert (
  order_id, status, amount, updated_at
) values (
  source.order_id, source.status, source.amount, source.updated_at
);
```

The main risk is late-arriving data. Many teams use a lookback window, for example reprocess the last 2 or 7 days, then deduplicate by key.

## How do you optimize analytical SQL queries?

Start by reducing scanned data, then reducing shuffle/join work. Select only needed columns, filter early, use partition pruning, avoid exploding joins, and aggregate before joining when possible. Check the query plan to see whether the engine is scanning too much data or doing expensive repartitions.

Example improvement:

```sql
with daily_orders as (
  select customer_id, order_date, sum(amount) as revenue
  from fact_orders
  where order_date >= date '2026-01-01'
  group by customer_id, order_date
)
select c.customer_segment, d.order_date, sum(d.revenue) as revenue
from daily_orders d
join dim_customer c
  on d.customer_id = c.customer_id
group by c.customer_segment, d.order_date;
```

This aggregates the large fact table before joining to the dimension, which can be much cheaper than joining every raw order row first.

## How do you query JSON in SQL?

Modern engines store semi-structured data in a column and query into it without a separate document store. The mechanics are the same everywhere: parse the document, navigate a path, and return a typed scalar - then index the path you filter on most.

```sql
-- Postgres: jsonb with the -> (JSON) and ->> (text) operators
SELECT payload ->> 'event_type' AS event_type,
       (payload -> 'amount')::numeric AS amount
FROM events
WHERE payload ->> 'country' = 'US';

-- Snowflake: the colon path accessor, cast with ::
SELECT payload:event_type::string AS event_type,
       payload:amount::number AS amount
FROM events
WHERE payload:country::string = 'US';

-- BigQuery: JSON functions on a STRING or JSON column
SELECT JSON_VALUE(payload, '$.event_type') AS event_type
FROM events
WHERE JSON_VALUE(payload, '$.country') = 'US';
```

The gotcha is cost. Path extraction is a function on the column, so it is not automatically sargable, and engines differ on whether they can push it into an index. Querying a wide JSON column also reads the whole document, not just the path. The fix is to extract hot paths at load time into typed columns, or to build an expression index on the path. Store JSON for flexibility, not for fields you filter and join on every day.

## How do SQL dialects differ?

The concepts transfer between engines; the syntax and defaults do not. Interviewers use dialect questions to check whether you have actually written SQL on more than one platform.

| Concern | Postgres | MySQL | SQL Server | Snowflake | BigQuery |
| --- | --- | --- | --- | --- | --- |
| Limit rows | `LIMIT n` | `LIMIT n` | `TOP n` / `OFFSET FETCH` | `LIMIT n` | `LIMIT n` |
| Concatenate | `\|\|` | `CONCAT` | `+` or `CONCAT` | `\|\|` | `CONCAT` or `\|\|` |
| Current time | `now()` | `now()` | `getdate()` | `current_timestamp()` | `current_timestamp()` |
| Date truncation | `date_trunc` | `date_format` | `dateadd`/`format` | `date_trunc` | `date_trunc` |
| Null-safe equal | `IS NOT DISTINCT FROM` | `<=>` | none | `EQUAL_NULL` | `IS NOT DISTINCT FROM` |
| Upsert | `INSERT ... ON CONFLICT` | `ON DUPLICATE KEY` | `MERGE` | `MERGE` | `MERGE` |
| String quotes | `'...'`, `"id"` | `` `id` `` | `[id]` | `"id"` | `` `id` `` |
| Grouping strictness | strict | permissive | strict | strict | strict |

Three differences that cause real bugs. First, identifier quoting: `"id"` is an identifier in Postgres and a string in MySQL, and backticks are the reverse. Second, `GROUP BY` strictness: MySQL will silently accept a non-aggregated column, while Postgres errors, so a query that passes locally can fail in production. Third, case sensitivity of identifiers: Postgres folds unquoted names to lower case, while Snowflake folds them to upper case. Write portable SQL with unquoted lower-case names and explicit casts.

## How do you read a query plan?

Written SQL says *what* you want. The query plan says *how* the engine will get it. When a query is slow, the SQL text is rarely the whole story - the plan is. Reading plans is the single highest-value tuning skill because it turns guessing into evidence.

The mental model: a plan is a tree of operators. The root produces the rows you see. Its children feed it, their children feed them, and the leaves read tables, indexes, or partitions. Rows flow upward one operator at a time, and each operator either reduces rows, multiplies them, or reshapes them.

### The operators you will see most often

- **Scan** (also called a table access or Seq Scan): read every row of a table. Cheap per row, expensive in total on a big table.
- **Search / Index Seek**: jump into a B-tree by key and read only the matching range. The plan text usually says it used an index.
- **Filter**: drop rows that fail a predicate after they were read.
- **Join**: combine two inputs. The interesting part is *which* algorithm and *which side* is the outer input.
- **Aggregate**: group rows. Implemented as a hash table or as a sorted run.
- **Sort**: order rows. The usual source of memory use and spills.
- **Limit**: stop early. It only helps if the operator below it can also stop early.
- **Exchange / Shuffle**: move rows between workers in a distributed engine such as Spark, Snowflake, or BigQuery.

### EXPLAIN versus EXPLAIN ANALYZE

`EXPLAIN` asks the optimiser to show the plan it *would* use. It does not execute the query. That makes it safe, instant, and based purely on statistics.

`EXPLAIN ANALYZE` (Postgres, MySQL 8, SQL Server's actual plan) actually runs the statement and adds real elapsed time and real row counts next to the estimates. Note that some engines also call the statistics command `ANALYZE` - different word, same spelling, do not confuse them.

```sql
-- Postgres: show the chosen plan, do not run it
EXPLAIN
SELECT c.customer_id, SUM(o.amount)
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
WHERE o.order_date >= DATE '2023-01-01'
GROUP BY c.customer_id;

-- Postgres: run it and report estimates next to actuals
EXPLAIN (ANALYZE, BUFFERS)
SELECT c.customer_id, SUM(o.amount)
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
WHERE o.order_date >= DATE '2023-01-01'
GROUP BY c.customer_id;
```

```sql
-- SQLite
EXPLAIN QUERY PLAN
SELECT customer_id, SUM(amount) FROM orders GROUP BY customer_id;

-- MySQL 8 / SQL Server are close cousins
-- MySQL:   EXPLAIN ANALYZE SELECT ...
-- SQL Server: SET STATISTICS IO ON; SET SHOWPLAN_ALL ON;
```

Gotcha: `EXPLAIN ANALYZE` runs the statement. On an `UPDATE`, `DELETE`, or `INSERT ... SELECT` it will change data. Run it inside a transaction and roll back, or restrict it to `SELECT`.

### Estimated rows versus actual rows

Every operator carries two numbers: the estimated number of rows and, under `ANALYZE`, the actual number. The estimate is what the optimiser used to pick the algorithm. The actual is the truth.

When they are close, the plan is probably the best one available. When they diverge by orders of magnitude, the optimiser made its choice blind and you should not trust the rest of the plan.

A concrete example: a table has 1,000,000 rows. A `WHERE country = 'US'` predicate is estimated at 1 row because the statistics are stale and think `country` is almost unique. The optimiser therefore chooses a nested loop with an index seek for each of the 1 rows it expects. At runtime 400,000 rows match, so it performs 400,000 index seeks - a classic slow query.

Gotcha: estimates are built from statistics, and statistics lie in predictable ways. Uniform-distribution assumptions break on skewed data. Per-column statistics break on correlated columns, such as `city` and `state`. Sampling breaks on rare values. Functions and casts on the filtered column hide the value from the estimator entirely.

### Generic plans versus custom plans

A prepared statement can be planned once with placeholders (a generic plan) or replanned for each concrete parameter value (a custom plan). Postgres uses custom plans for the first five executions, then switches to a generic plan if it looks no more expensive. SQL Server always caches a plan and reuses it, which is what parameter sniffing means.

```sql
-- Postgres: inspect and steer
PREPARE q(int) AS SELECT * FROM orders WHERE customer_id = $1;
EXPLAIN EXECUTE q(42);
SET plan_cache_mode = force_custom_plan;   -- or force_generic_plan
```

Gotcha: the same generic plan has to work for every value. A plan tuned for `customer_id = 1` (a whale with 100,000 orders) is terrible for `customer_id = 2` (three orders). Symptoms are a query that is fast for one parameter and slow for another. Fixes are a `RECOMPILE` hint, forcing a custom plan, or splitting the query so the heavy case is handled separately.

The example below builds a tiny table in SQLite and watches the plan change when an index appears. SQLite's `EXPLAIN QUERY PLAN` is a simplified, readable version of the same idea: `SCAN` means read everything, `SEARCH ... USING INDEX` means the index restricted the rows.

In [2]:
import sqlite3

con = sqlite3.connect(":memory:")
con.executescript(
    '''
    CREATE TABLE orders (
        order_id INTEGER PRIMARY KEY,
        customer_id INTEGER,
        order_date TEXT,
        amount REAL
    );
    '''
)
con.executemany(
    "INSERT INTO orders(order_id, customer_id, order_date, amount) VALUES (?, ?, ?, ?)",
    [(i, i % 5000, "2023-01-%02d" % (i % 28 + 1), float(i % 100)) for i in range(1, 50001)],
)


def plan(sql):
    return [row[3] for row in con.execute("EXPLAIN QUERY PLAN " + sql)]


query = "SELECT order_id FROM orders WHERE customer_id = 11"
before = plan(query)
con.execute("CREATE INDEX idx_orders_customer ON orders(customer_id)")
after = plan(query)

print("before index:", before)
print("after index: ", after)
assert any("SCAN" in step for step in before)
assert any("SEARCH" in step and "INDEX" in step for step in after)
print("the plan switched from a full scan to an index search")

before index: ['SCAN orders']
after index:  ['SEARCH orders USING COVERING INDEX idx_orders_customer (customer_id=?)']
the plan switched from a full scan to an index search


### What to look for, in order

1. Start at the top and find the operator that takes the most time, not the first one that looks scary.
2. Compare estimated rows with actual rows at that operator. A big gap explains bad choices downstream.
3. Look for full scans on large tables where a predicate should have restricted the rows.
4. Look at the join algorithms and join order. A nested loop over a large inner input is the classic blow-up.
5. Look for sorts and hashes that spilled to disk rather than staying in memory.
6. Check whether the rows you discard are discarded early or late. Filtering after a join is far more expensive than filtering before it.

### A worked example

Suppose an `events` table has 500 million rows, partitioned by day, and you run a report for one day. The plan should prune to one partition and read roughly ten million rows. If the plan shows a scan across every partition, the filter is written in a way the engine cannot push down - often a function applied to the partition column.

```sql
-- Cannot prune: the partition column is wrapped in a function
SELECT COUNT(*) FROM events
WHERE CAST(event_time AS DATE) = DATE '2023-06-01';

-- Can prune: the column is compared directly to a range
SELECT COUNT(*) FROM events
WHERE event_time >= TIMESTAMP '2023-06-01 00:00:00'
  AND event_time <  TIMESTAMP '2023-06-02 00:00:00';
```

The two queries return the same count, but only the second lets the optimiser use the partitioning metadata and the index on `event_time`. This is the idea of a *sargable* predicate, covered in detail in the index section later in this chapter.

Gotcha: `EXPLAIN` is not a benchmark. The cost numbers are internal units, not milliseconds, and they are comparable only within one plan on one engine. Two plans whose costs differ by 2x may run at the same speed; two plans whose estimated rows differ by 1000x will usually differ a lot in reality.

### Common plan smells

- **Seq Scan on a big table** with a selective predicate: a missing or unusable index.
- **Nested Loop** where the inner side is large: the estimator under-counted the outer side.
- **Sort** with a disk/spill note: reduce the row count before sorting, or add the sort key to an index.
- **Hash Join** with a large build side: the smaller side should have been the build side.
- **Rows Removed by Filter** that is much larger than the rows returned: the predicate is not pushed down.
- **Recheck Cond** on an index scan: the index is only partially selective.
- **Estimate 1, actual 1000000**: stale or missing statistics, or a function on the column.

In an interview, narrate the plan rather than reading it line by line. Say what the engine does first, where the expensive step is, why it chose that step, and what you would change. That structure shows you can tune rather than memorise.

### Reading the plan text, engine by engine

The shape is universal, the formatting is not. Postgres prints an indented tree with costs and rows. MySQL prints a table with an `id`, a `type`, a `key`, and a `rows` column. SQL Server prints a graphical or XML plan. Snowflake and BigQuery show a profile with operators and a percentage of total time. Learn one, and you can read them all.

```text
-- Postgres EXPLAIN ANALYZE (abridged)

HashAggregate  (cost=1840..1860 rows=200 width=16) (actual time=42.1..42.2 rows=198 loops=1)
  Group Key: orders.customer_id
  ->  Hash Join  (cost=120..1600 rows=40000 width=16) (actual time=0.9..35.0 rows=41000 loops=1)
        Hash Cond: (orders.customer_id = c.customer_id)
        ->  Seq Scan on orders  (cost=0..1200 rows=40000 width=16) (actual time=0.01..12.0 rows=41000 loops=1)
              Filter: (order_date >= '2023-01-01'::date)
              Rows Removed by Filter: 159000
        ->  Hash  (cost=100..100 rows=200 width=8)
              ->  Seq Scan on customers  (cost=0..100 rows=200 width=8) (actual time=0.01..1.2 rows=200 loops=1)
```

Read it bottom-up. `customers` is scanned into a hash table. `orders` is scanned with a filter that removes 159,000 rows. The hash join produces 41,000 rows, and the hash aggregate collapses them to 198 groups. The estimates were 40,000 and 200; the actuals are 41,000 and 198, close enough. Nothing here is obviously wrong, so this query is already near its natural cost.

### Buffers and I/O

`EXPLAIN (ANALYZE, BUFFERS)` adds page counts: how many blocks came from the shared cache and how many were read from disk. A query that is slow with high disk reads is I/O-bound. A query that is slow with cache hits is CPU-bound or spending time in sorts and hashes. That distinction decides whether an index, more memory, or a rewrite is the right fix.

### Scan versus seek cost

A sequential scan reads pages one after another, so each page read is cheap and the cost is roughly linear in pages. An index seek jumps between pages in key order, so each row can touch a different page. That is why an index is a win at 1% selectivity and a loss at 40%: the random reads overwhelm the saving.

In [3]:
def scan_cost(pages):
    return pages


def seek_cost(rows_returned, index_height, cache_hit_rate=0.9):
    # A secondary index can touch a page per matching row;
    # the cache absorbs most repeated page reads.
    disk_reads = rows_returned * (1 - cache_hit_rate)
    return index_height + disk_reads


pages = 100_000
for selectivity in (0.001, 0.01, 0.1, 0.4):
    rows_returned = 10_000_000 * selectivity
    print(
        f"selectivity={selectivity:>5} scan={scan_cost(pages):>7.0f} "
        f"seek={seek_cost(rows_returned, 4):>10.0f}"
    )

assert seek_cost(10_000_000 * 0.001, 4) < scan_cost(pages)
assert seek_cost(10_000_000 * 0.4, 4) > scan_cost(pages)
print("the index wins when selective, the scan wins when not")

selectivity=0.001 scan= 100000 seek=      1004
selectivity= 0.01 scan= 100000 seek=     10004
selectivity=  0.1 scan= 100000 seek=    100004
selectivity=  0.4 scan= 100000 seek=    400004
the index wins when selective, the scan wins when not


### Statistics: the input to every decision

The optimiser never looks at the data. It looks at statistics: row count, page count, the number of distinct values, a histogram of the most common values, and sometimes correlation between columns. Sampling means very large tables get an approximate picture, and the most common values are exactly the ones that skew the estimate.

```sql
-- Refresh statistics after a bulk load
ANALYZE orders;

-- Postgres: collect statistics over correlated columns together
CREATE STATISTICS orders_city_state (dependencies, ndistinct)
ON city, state FROM customers;
```

### Plan stability and hints

A plan can flip between two shapes as data grows, changing a query from fast to slow overnight with no code change. Engines expose hints to force a plan - SQL Server `OPTION (HASH JOIN)`, Oracle and Snowflake hints, MySQL `STRAIGHT_JOIN`, Postgres via the `pg_hint_plan` extension. Treat hints as a last resort: they pin one plan forever and hide the underlying statistics problem. Fix the estimate when you can; hint only when you cannot.

### A worked plan walk-through

Suppose a report takes 90 seconds. The plan shows a nested loop whose outer operator returned 2 rows by estimate but 800,000 rows in reality, and whose inner side is a 5,000,000-row table with no index on the join key. The optimiser expected 2 lookups; it performed 800,000 full scans.

The fix has three parts. Refresh statistics so the estimate is closer. Add an index on the inner join key so each lookup is a seek even if the estimate stays wrong. And push the filter that produced 800,000 rows earlier in the plan if the business rules allow it. Re-measure after each change so you know which one earned the win.

### Interview follow-ups on plans

- What is the difference between `EXPLAIN` and `EXPLAIN ANALYZE`?
- Why can the same query be fast for one parameter and slow for another?
- What does a large gap between estimated and actual rows tell you?
- How do you decide between a scan and an index seek?
- When would you keep a plan hint in production?

## How do join strategies and join order work?

Joins are where most slow analytical queries spend their time. The engine has three physical algorithms - nested loop, hash, and merge - and it pairs them with a join order. The algorithm and the order are chosen from cardinality estimates, which is why bad statistics produce bad joins.

Mental model: a join is always between exactly two inputs at a time, even when you write five tables. The engine folds them pairwise into a tree. For each pair it picks an algorithm and a direction. The cost of the whole tree is the sum of the pairwise costs, so one bad pair dominates everything.

### Nested loop join

For every row of the outer (driving) input, look up matching rows in the inner input. If the inner side has an index on the join key, each lookup is a B-tree descent, so the cost is roughly `outer_rows * log(inner_rows)`. Without an index it is `outer_rows * inner_rows`, which explodes.

Nested loop wins when the outer side is small and the inner side is indexed. It also supports non-equality join predicates (`<`, `BETWEEN`, `ST_DWithin`) that hash and merge joins cannot express, and it can stop early when the query has a `LIMIT`.

```sql
-- Small outer side, indexed inner side: a nested loop is ideal
SELECT o.order_id, c.customer_name
FROM customers c
JOIN orders o ON o.customer_id = c.customer_id
WHERE c.customer_id = 42;
```

### Hash join

Build an in-memory hash table on the smaller input (the build side), then stream the larger input (the probe side) through it. Each probe is expected constant time, so the cost is linear: `build_rows + probe_rows`.

Hash join wins on large equi-joins with no useful index, which is the normal shape of a warehouse query. Its weakness is memory. If the build side does not fit in the memory grant, the engine partitions the hash table and spills some of it to disk, and performance drops sharply. A skewed join key can push one partition over the limit even when the total build side looks small.

### Merge join (sort-merge)

Sort both inputs on the join key, then walk them in lockstep. If both inputs already arrive sorted - for example, from an index scan or a previous merge - the sort is free and the merge is a single pass. Cost is `sort(outer) + sort(inner) + outer_rows + inner_rows`.

Merge join wins when both sides are already ordered, when the join key is a range predicate, and when a hash join would spill. It handles inequality joins once the data is sorted. It loses when the inputs must be sorted from scratch just for the join.

### Choosing a strategy: a decision table

| Situation | Best strategy | Why |
| --- | --- | --- |
| Tiny outer side, indexed inner, equality or range | Nested loop | Few B-tree descents |
| Two large inputs, equality, no useful order | Hash join | Linear, no sort |
| Both inputs already sorted on the key | Merge join | Single pass, no memory spike |
| Inequality or non-equi predicate | Nested loop or merge | Hash needs equality |
| Very small inner side | Broadcast hash | Replicate the small side, avoid a shuffle |
| Outer side huge, inner side small but unindexed | Hash with inner as build | One pass each |

### Join order and the optimiser

The order in which you write `JOIN` clauses is a hint at best, and most optimisers ignore it because SQL declares a result, not a sequence. The optimiser searches a space of join trees - usually left-deep trees, where each join consumes the previous result - and prunes it with dynamic programming above a row threshold, falling back to greedy or genetic search for very large queries.

The moving parts are cardinality estimates and cost. The optimiser wants the smallest intermediate results, so it prefers to apply selective joins first. That is why a query can be fast when you filter a dimension table down to ten rows and slow when you forget the filter.

### When statistics lie

Statistics are a summary, not the data. The common failure modes:

- **Stale statistics.** Rows were loaded or updated but the planner statistics were never refreshed. The estimate describes last week's table.
- **Uniformity.** The optimiser often assumes values are spread evenly. Real data is skewed: a few customers hold most orders.
- **Correlation.** It assumes columns are independent. `city = 'Springfield'` and `state = 'IL'` are almost the same fact, so multiplying their selectivities under-counts badly.
- **Functions and casts.** `WHERE UPPER(email) = 'A@B.COM'` with an index on `email` hides the value from the estimator and disables the index.
- **Bind variables.** A generic plan is planned for an average parameter, which no single parameter is.

A useful diagnostic: compare the estimated rows at the top of the plan with the actual row count. If the plan says 1,000 and the query returns 4,000,000, the estimator was wrong and the chosen algorithm probably is too. Refresh statistics first, then consider extended statistics for correlated columns.

### Fan-out and cardinality estimation

A join on a key that is unique on one side is safe: it preserves the row count of the other side. A join on a key that is not unique multiplies rows - this is fan-out. Join `orders` (many rows per customer) to `order_items` (many rows per order) and you get many rows per order; join that to another many-sided table without an aggregation and the result explodes.

The estimator handles this with the same arithmetic you would: `rows(a) * rows(b) / max(distinct_a, distinct_b)`. When the true distinct count is far from the statistic, the estimate is far off. When a query "returns too many rows" unexpectedly, fan-out is usually the reason - it is a data-model issue, not a tuning issue.

### Semi-joins and anti-joins

`WHERE EXISTS (...)` is a semi-join: it keeps a row from the left when at least one match exists and never duplicates it. `WHERE NOT EXISTS (...)` is an anti-join. Both are usually planned as a hash semi-join or hash anti-join, and both are the right tool when you mean "has a match" rather than "give me every matching pair".

```sql
-- Semi-join: one row per customer, no duplication
SELECT c.customer_id
FROM customers c
WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id);

-- Fan-out: one row per order, so customers appear many times
SELECT c.customer_id, o.order_id
FROM customers c
JOIN orders o ON o.customer_id = c.customer_id;
```

### Practical rules

1. Filter the small dimension tables early so the optimiser drives from a small input.
2. Join on equality where you can; it unlocks hash and merge joins.
3. Make sure the join key types match. An integer joined to a string forces casts and kills indexes.
4. Prefer `EXISTS` or `IN` when you only need existence, to avoid fan-out.
5. Refresh statistics after big loads, and consider extended statistics for correlated columns.
6. Watch the build side of a hash join: the small side should build, and skew can still break it.
7. Read the plan after every rewrite; the algorithm is a decision by the engine, not by you.

The cell below models the three strategies with a simple arithmetic cost model. It is not a database planner, but it shows why the same two tables can be cheap with an index and ruinous without one.

In [4]:
def nested_loop_cost(outer_rows, inner_rows, inner_indexed):
    probes = 1 if inner_indexed else inner_rows
    return outer_rows * probes


def hash_join_cost(build_rows, probe_rows):
    return build_rows + probe_rows


def merge_join_cost(left_rows, right_rows, already_sorted):
    sort_cost = 0 if already_sorted else (left_rows + right_rows)
    return sort_cost + left_rows + right_rows


facts = 1_000_000
dimensions = 50_000
print("nested loop, inner indexed:  ", nested_loop_cost(facts, dimensions, True))
print("nested loop, inner unindexed:", nested_loop_cost(facts, dimensions, False))
print("hash join:                  ", hash_join_cost(dimensions, facts))
print("merge join, already sorted: ", merge_join_cost(facts, dimensions, True))

assert nested_loop_cost(facts, dimensions, True) < hash_join_cost(dimensions, facts)
assert hash_join_cost(dimensions, facts) < nested_loop_cost(facts, dimensions, False)
print("an indexed nested loop is the cheapest here; an unindexed one is the worst")

nested loop, inner indexed:   1000000
nested loop, inner unindexed: 50000000000
hash join:                   1050000
merge join, already sorted:  1050000
an indexed nested loop is the cheapest here; an unindexed one is the worst


### Broadcast, shuffle, and co-location

In a distributed engine the join algorithm also needs a data-movement decision. A shuffle moves both sides so matching keys land on the same worker. A broadcast sends the small side to every worker and leaves the large side in place. If the tables are already partitioned on the join key - co-located - no movement is needed at all.

Broadcast is usually the cheapest when one side is genuinely small, because it avoids moving the huge side. The failure mode is broadcasting something that is not small: a 5 GB dimension copied to 200 workers is 1 TB of network traffic. Watch the plan for the estimated size of the broadcast side, not its row count.

### Data skew and the hot key

Hash joins assume keys spread evenly. One hot key - a null customer, a test account, a default warehouse - sends a disproportionate share of rows to one worker, which becomes the straggler that defines the query time. The plan may show one task taking ten times the median.

```sql
-- Salt the hot key so the load spreads across workers
SELECT ... FROM orders o
JOIN customers c
  ON c.customer_id = o.customer_id
 AND MOD(ABS(HASH(o.order_id)), 16) = 0   -- one shard of the salt
```

Alternatives are to filter or isolate the hot key, to broadcast the small side instead of hashing, or to pre-aggregate the skewed side before the join.

### Join keys must match in type and collation

Joining an integer column to a string column forces a cast on one side, which disables the index and can also disable partition pruning. Collations matter for text keys: two strings that look equal may compare unequal, or may compare case-insensitively, depending on the columns' collations. Align the types at load time, not in the join.

### Self-joins and hierarchy joins

A self-join matches a table to itself, usually to compare a row with its parent, its previous row, or another row in the same group. It needs two aliases, and the join key must be indexed or the cost is quadratic.

```sql
-- employee and manager name in one row
SELECT e.name AS employee, m.name AS manager
FROM employees e
LEFT JOIN employees m ON m.emp_id = e.manager_id;
```

When the depth is unbounded, a recursive CTE replaces repeated self-joins: one level per iteration instead of one join per level written by hand.

### Cardinality estimation: a worked example

Fact `orders` has 10,000,000 rows and 200,000 distinct customers. Dimension `customers` has 200,000 rows. An inner join on `customer_id` is many-to-one, so the output is still about 10,000,000 rows. Now filter customers to `country = 'US'`, which keeps 10% of them, and 10% of orders belong to those customers, so the output drops to about 1,000,000. This is exactly the arithmetic the optimiser performs.

In [5]:
def join_rows(left_rows, left_distinct, right_rows, right_distinct):
    return left_rows * right_rows / max(left_distinct, right_distinct)


orders = 10_000_000
order_customers = 200_000
customers = 200_000
us_customers = 20_000

full = join_rows(orders, order_customers, customers, customers)
filtered = join_rows(orders, order_customers, us_customers, us_customers)
print("full join rows:    ", round(full))
print("filtered join rows:", round(filtered))
assert round(full) == orders
assert round(filtered) == orders / 10
print("filtering the dimension by 10x shrinks the join by 10x")

full join rows:     10000000
filtered join rows: 1000000
filtering the dimension by 10x shrinks the join by 10x


### Join hints and when not to use them

Hints express what you saw in one plan on one day. They do not survive a data change, and they make the query brittle. Reach for a hint only after refreshing statistics failed and you have measured the forced plan beating the automatic one on representative data. Document the reason next to the hint, because the next engineer will want to delete it.

### Common join mistakes

1. Joining on a non-unique key and not noticing the fan-out.
2. Putting a right-side filter in `WHERE` and silently converting a left join to an inner join.
3. Joining columns with different types, which disables indexes.
4. Broadcast-joining a side that is not actually small.
5. Ignoring skew, so one worker dominates the run time.
6. Using `NOT IN` against a subquery that can return `NULL`.

### Interview follow-ups on joins

- When does a nested loop beat a hash join?
- What makes a hash join spill, and how do you fix it?
- How does the optimiser choose join order?
- What is fan-out and how do you detect it?
- How would you handle a skewed join key?

## How do indexes and predicates work together?

An index is only useful if the predicate can be turned into a range of keys to look up. Whether that happens is a property of how you wrote the predicate, not of the index alone. This is the difference between an index that exists and an index that is used.

Mental model: a B-tree stores keys in sorted order and keeps a short path from the root to any key. A lookup by equality is one descent. A range is one descent plus a walk along the linked leaves. Any predicate the engine can express as a key range is *sargable* - from Search ARGument ABLE.

### Sargability

A predicate is sargable when the indexed column appears alone on one side of the comparison and the other side is a constant or an expression that does not depend on the row.

```sql
-- Sargable: the index on hire_date can be used
WHERE hire_date >= DATE '2023-01-01'

-- Not sargable: the column is wrapped in a function
WHERE YEAR(hire_date) = 2023

-- Not sargable: a leading wildcard defeats the sorted order
WHERE name LIKE '%smith'

-- Sargable: an anchored prefix uses the index
WHERE name LIKE 'smith%'
```

The common non-sargable patterns are functions on the column (`UPPER`, `DATE`, `CAST`, `SUBSTRING`), arithmetic on the column (`salary * 12 > 100000`), implicit type casts, leading wildcards, and `OR` across different columns.

```sql
-- Rewrite the function on the column as a range on the column
WHERE hire_date >= DATE '2023-01-01' AND hire_date < DATE '2024-01-01'

-- Rewrite arithmetic so the column stands alone
WHERE salary > 100000 / 12.0

-- Rewrite a leading wildcard with a different access path,
-- for example a trigram index or a full-text index
```

Gotcha: an implicit cast is invisible in the SQL but fatal to the index. Joining a `VARCHAR` column to an integer literal, or comparing a timestamp column to a string in the wrong format, forces a cast on the column and turns a seek into a scan. Match the literal type to the column type.

### The leading column of a composite index

A composite index on `(a, b, c)` is sorted by `a`, then by `b` within equal `a`, then by `c` within equal `(a, b)`. The engine can use it only for a prefix of that order. Think of a phone book sorted by last name then first name: you can find all Smiths, and all John Smiths, but you cannot efficiently find every John across all last names.

```sql
CREATE INDEX idx_orders ON orders(customer_id, order_date, status);

-- Uses the index: prefix (customer_id)
WHERE customer_id = 7

-- Uses the index: full prefix (customer_id, order_date)
WHERE customer_id = 7 AND order_date >= DATE '2023-01-01'

-- Uses the index as a filter, not a seek: customer_id is missing
WHERE order_date >= DATE '2023-01-01'

-- Uses customer_id and order_date; status is checked after the seek
WHERE customer_id = 7 AND order_date = DATE '2023-06-01' AND status = 'shipped'
```

The classical ordering rule is: equality columns first, then the range column, then columns used only for ordering or covering. Once a range predicate is applied to a column, later columns in the index can no longer be used for a seek, though they can still be read from the index for filtering or coverage.

### Covering indexes

A covering index contains every column the query needs, so the engine can answer from the index alone and never touch the table. This is called an index-only scan. It is much faster because the index is smaller than the table and the lookups disappear.

```sql
-- Query needs customer_id, order_date, amount
CREATE INDEX idx_orders_covering
    ON orders(customer_id, order_date)
    INCLUDE (amount);      -- SQL Server; Postgres supports the same syntax

-- Postgres can also simply put amount in the key list
CREATE INDEX idx_orders_covering ON orders(customer_id, order_date, amount);
```

Gotcha: adding columns to an index to cover a query widens every index entry. A covering index speeds up one read pattern and slows down every write, and it costs disk. Cover the two or three hottest queries, not all of them.

### Selectivity

Selectivity is how much a predicate narrows the data. A column with many distinct values is highly selective: an equality match returns a small fraction of the rows. A column with few distinct values - `status`, `is_active`, `country` - is not, and the optimiser may correctly choose a full scan instead of the index.

The rule of thumb for an equality predicate is `rows / distinct_values`. If a table has 1,000,000 rows and `customer_id` has 1,000,000 distinct values, the predicate returns about one row: excellent. If `status` has two distinct values, it returns about 500,000 rows: the index would be slower than a scan, because a scan reads sequentially and the index hops randomly through the table.

```sql
-- Postgres: how selective is the column?
SELECT n_distinct FROM pg_stats
WHERE tablename = 'orders' AND attname = 'status';

-- Refresh statistics after a large load
ANALYZE orders;
```

### When an index will not be used

- The predicate is not sargable (a function on the column).
- The predicate is not selective enough, so a scan is genuinely cheaper.
- The types differ, forcing an implicit cast on the column.
- The table is small; a scan of a few pages beats any index overhead.
- The query returns most columns of most rows, so the index lookups would be more expensive than the scan.
- The index is invalid, or the statistics are so stale that the planner mis-prices it.
- A `NOT`, `<>`, or `IS NOT NULL` predicate in some engines cannot use the index at all.

### Partial, expression, and unique indexes

A partial index indexes only the rows that satisfy a predicate. It is smaller and cheaper to maintain, and it is perfect for a hot query over a small slice of the table.

```sql
-- Only open orders are indexed; closed orders never pay the write cost
CREATE INDEX idx_orders_open ON orders(customer_id)
WHERE status = 'open';

-- An expression index makes a function-based predicate sargable
CREATE INDEX idx_orders_month ON orders(EXTRACT(MONTH FROM order_date));
```

A unique index is a constraint as well as an access path. It lets the planner know that an equality predicate returns at most one row, which unlocks better join choices.

### The write cost of every index

Every index you add is maintained by every `INSERT`, `UPDATE`, and `DELETE` on the table, and it uses disk and cache. A table with ten indexes writes eleven structures per row. Index what you filter and join on, and drop indexes that no query uses. Unused indexes are pure cost.

### Common mistakes

1. Indexing a low-selectivity column such as `status` by itself and expecting a big win.
2. Putting the range column before the equality column in a composite index.
3. Wrapping the indexed column in a function and blaming the index.
4. Creating overlapping indexes - `(a)` and `(a, b)` - that duplicate work.
5. Forgetting that `LIKE '%x%'` cannot use a normal B-tree.
6. Never refreshing statistics after a big load.
7. Covering a query by indexing all of its columns and turning the index into a copy of the table.

The first cell below computes a selectivity factor for a few columns. The second cell writes a query in both a sargable and a non-sargable form against SQLite and shows the plan flip from a search to a full scan.

In [6]:
def rows_per_key(total_rows, distinct_values):
    if distinct_values <= 0:
        raise ValueError("distinct_values must be positive")
    return total_rows / distinct_values


def selectivity(total_rows, distinct_values):
    return distinct_values / total_rows


total = 1_000_000
columns = {
    "customer_id": 900_000,   # near unique
    "order_date": 1_095,      # one per day for three years
    "status": 4,              # open, shipped, cancelled, refunded
}

for name, distinct in columns.items():
    print(
        f"{name:12s} distinct={distinct:>7d} "
        f"selectivity={selectivity(total, distinct):.6f} "
        f"rows returned per equality ~ {rows_per_key(total, distinct):.1f}"
    )

most = max(columns, key=lambda c: selectivity(total, columns[c]))
least = min(columns, key=lambda c: selectivity(total, columns[c]))
assert most == "customer_id"
assert least == "status"
print(f"index {most} first; {least} alone is too coarse to help")

customer_id  distinct= 900000 selectivity=0.900000 rows returned per equality ~ 1.1
order_date   distinct=   1095 selectivity=0.001095 rows returned per equality ~ 913.2
status       distinct=      4 selectivity=0.000004 rows returned per equality ~ 250000.0
index customer_id first; status alone is too coarse to help


In [7]:
import sqlite3

con = sqlite3.connect(":memory:")
con.executescript(
    '''
    CREATE TABLE employees (
        emp_id INTEGER PRIMARY KEY,
        name TEXT,
        hire_date TEXT,
        salary REAL
    );
    INSERT INTO employees(emp_id, name, hire_date, salary) VALUES
        (1, 'Ada',   '2022-03-01', 120000),
        (2, 'Brett', '2023-01-15',  90000),
        (3, 'Chen',  '2023-07-01', 105000),
        (4, 'Dana',  '2024-02-20',  95000);
    CREATE INDEX idx_employees_hire ON employees(hire_date);
    '''
)


def plan(sql):
    return [row[3] for row in con.execute("EXPLAIN QUERY PLAN " + sql)]


non_sargable = "SELECT name FROM employees WHERE substr(hire_date, 1, 4) = '2023'"
sargable = (
    "SELECT name FROM employees "
    "WHERE hire_date >= '2023-01-01' AND hire_date < '2024-01-01'"
)

print("function on the column:", plan(non_sargable)[0])
print("range on the column:   ", plan(sargable)[0])
assert plan(non_sargable)[0].startswith("SCAN")
assert "SEARCH" in plan(sargable)[0] and "USING INDEX" in plan(sargable)[0]
print("same rows, different access path")

function on the column: SCAN employees
range on the column:    SEARCH employees USING INDEX idx_employees_hire (hire_date>? AND hire_date<?)
same rows, different access path


### Clustered versus secondary indexes

A clustered index defines the physical order of the table, so range scans are sequential and the index leaves are the rows themselves. There can be only one. Every other index is secondary and stores pointers back to the row, which costs an extra lookup for each match unless the index covers the query.

### Index-only scans

An index-only scan answers the query from the index and never visits the table. It requires every referenced column to be in the index and, in Postgres, the visibility map to show the pages as all-visible. A `SELECT *` almost never qualifies; selecting the three columns you need often does.

```sql
-- Needs only these three columns, so the index can cover it
SELECT customer_id, order_date, amount
FROM orders
WHERE customer_id = 42;

CREATE INDEX idx_orders_cover ON orders(customer_id, order_date, amount);
```

### Bitmap scans

When many rows match but each match is scattered, an engine may build a bitmap of matching pages and read them in physical order, converting random reads into sequential ones. Bitmap scans reduce the index-to-table penalty and are common on low-selectivity predicates. They cost memory proportional to the number of matching rows.

### Index bloat and maintenance

Updates and deletes leave gaps in a B-tree. Over time the index bloats, uses more pages, and reads more slowly than the data it describes. Postgres has `REINDEX` and `VACUUM`; MySQL InnoDB rebuilds via `OPTIMIZE TABLE`; SQL Server has `ALTER INDEX ... REBUILD`. Schedule maintenance for hot write tables, and watch index size versus table size.

### Correlation and multi-column statistics

When two columns move together, per-column statistics multiply two independent guesses and miss badly. Extended statistics tell the optimiser the columns are related. In Postgres, `CREATE STATISTICS ... (dependencies, ndistinct)`; in SQL Server, a multi-column statistic or a filtered statistic serves the same purpose.

### How many indexes is too many

Every index is a promise to maintain it on every write. A rough guide: index the primary and foreign keys, the columns in your hottest filters, and the sort keys of your top reports. Beyond five or six indexes on a write-heavy table, each new index costs more in write throughput than it returns in read speed. Audit for unused indexes and drop them.

### Leftmost prefix in practice

A composite index on `(a, b, c)` can answer `a`, `(a, b)`, and `(a, b, c)`, plus any of those with a range on the last listed column. It cannot seek on `b` alone or on `(b, c)`. Reordering the same three columns changes which of your queries it serves, so derive the order from the query patterns.

In [8]:
def can_seek(index_columns, predicate_columns):
    i = 0
    for column in predicate_columns:
        if i < len(index_columns) and index_columns[i] == column:
            i += 1
        else:
            return False
    return i > 0


index = ["customer_id", "order_date", "status"]
assert can_seek(index, ["customer_id"])
assert can_seek(index, ["customer_id", "order_date"])
assert can_seek(index, ["customer_id", "order_date", "status"])
assert not can_seek(index, ["order_date"])
assert not can_seek(index, ["status"])
assert not can_seek(index, ["order_date", "status"])
print("only leftmost prefixes of", index, "can seek")

only leftmost prefixes of ['customer_id', 'order_date', 'status'] can seek


### Indexing for ORDER BY and LIMIT

An index whose key order matches the `ORDER BY` can deliver rows already sorted, so a `LIMIT 10` reads ten rows instead of sorting the table. That only works if the same index also supplies the filter - otherwise the engine still has to read and discard. This is how a well-chosen index makes a "top 10 by date" report instant.

### Common index mistakes, condensed

1. A composite index in the wrong column order.
2. Functions or casts on the indexed column in the predicate.
3. Indexing a low-selectivity flag column on its own.
4. Duplicating a prefix that another index already covers.
5. Never refreshing statistics, so a good index is priced as a bad one.
6. Adding indexes for every query until writes crawl.

### Interview follow-ups on indexes

- Why is a composite index useless if you skip its first column?
- What is a covering index and when is it worth the write cost?
- Why can an index make a query slower?
- What is selectivity and how does it decide the access path?
- How do you find unused indexes in production?

## How do aggregation, window functions, and CTEs perform?

Aggregation, window functions, `DISTINCT`, and CTEs are the four things that make an analytical query use memory. They are also the four places where small changes in wording change the plan a lot. This section covers what they cost and how to keep them out of disk.

Mental model: the engine has two ways to group rows. A *hash aggregate* builds a hash table keyed by the group columns and updates the accumulator for each row; the result comes out in no particular order. A *sort aggregate* sorts the input by the group keys first, which lets it emit a group as soon as the key changes. The optimiser picks hash when there is memory and sort when it can reuse an existing order.

### GROUP BY mechanics

`GROUP BY` collapses rows. The cost is roughly one pass plus the memory for the hash table or the sort. A `HAVING` clause filters groups after they are built, so it cannot reduce the memory the grouping needs - only a `WHERE` clause on the input rows can do that.

```sql
-- Filter rows before grouping so the aggregate sees less data
SELECT customer_id, SUM(amount) AS revenue
FROM orders
WHERE status = 'completed'          -- reduces input
GROUP BY customer_id
HAVING SUM(amount) > 1000;          -- filters groups, after the fact
```

### DISTINCT versus GROUP BY

`SELECT DISTINCT a, b FROM t` and `SELECT a, b FROM t GROUP BY a, b` are the same operation and most engines plan them identically. The practical differences are stylistic and semantic: `GROUP BY` can also compute aggregates in the same pass, and `DISTINCT` applied to a whole wide row is often more expensive than grouping the two columns you actually care about.

Gotcha: `COUNT(DISTINCT x)` cannot be combined with a window frame in most engines, and it needs its own deduplication step. If you need a distinct count per group plus detail rows, compute it in a subquery or CTE and join back, rather than trying to force it into one window expression.

### Window functions and memory

A window function keeps every row. Before it can compute anything, the engine must arrange the rows into their partitions and order each partition by the window's `ORDER BY`. That means a sort (or a hash redistribution in a distributed engine) over the entire input, followed by a scan of each partition.

The memory is proportional to the largest partition, not the whole table, but a single skewed partition can be as large as the table. When a partition does not fit in the memory grant, the engine spills it to disk, and a query that was milliseconds becomes seconds.

```sql
-- One sort, one pass, two window results
SELECT
    order_id,
    customer_id,
    amount,
    SUM(amount) OVER (PARTITION BY customer_id ORDER BY order_date) AS running_total,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC) AS amount_rank
FROM orders;
```

### ROWS versus RANGE

The frame clause decides which rows each output row sees. `ROWS` counts physical rows. `RANGE` groups rows that are *peers* under the `ORDER BY` - rows with the same ordering value.

```sql
-- ROWS: each row advances by exactly one
SUM(amount) OVER (ORDER BY order_date
                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)

-- RANGE (the default): all rows with the same date are included together
SUM(amount) OVER (ORDER BY order_date
                  RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
```

With duplicate ordering values the two are different, and the difference is easy to miss in review. If two orders share a date, `ROWS` gives the first one a running total that excludes the second, while `RANGE` gives both the same total including both. Choose deliberately, and default to `ROWS` when you mean "one row at a time".

### Ranking functions

- `ROW_NUMBER()` numbers rows sequentially, breaking ties arbitrarily. Every row gets a distinct number.
- `RANK()` gives tied rows the same number and then skips: 1, 2, 2, 4.
- `DENSE_RANK()` gives tied rows the same number without skipping: 1, 2, 2, 3.

All three need an `ORDER BY` and a full sort of each partition. Use `ROW_NUMBER()` to deduplicate (keep the top row per key); use `RANK()` or `DENSE_RANK()` when the ties are meaningful, such as competition places or top-N with ties.

### CTE materialization

A CTE (`WITH name AS (...)`) is a named subquery. Whether the engine materialises it into a temporary result or inlines it into the main query changes performance a lot.

Postgres 12 and later inline non-recursive CTEs by default and let you force materialisation with `MATERIALIZED`. SQL Server and BigQuery tend to materialise. SQLite may do either. If a CTE is referenced several times, materialising it once can save repeated work; if it is selective, inlining lets the predicates push down.

```sql
-- Postgres: force one evaluation and reuse the result
WITH heavy AS MATERIALIZED (
    SELECT customer_id, SUM(amount) AS revenue
    FROM orders GROUP BY customer_id
)
SELECT * FROM heavy WHERE revenue > 10000;

-- Postgres: let the predicate push into the CTE (inline)
WITH heavy AS NOT MATERIALIZED (
    SELECT customer_id, SUM(amount) AS revenue
    FROM orders GROUP BY customer_id
)
SELECT * FROM heavy WHERE revenue > 10000;
```

Gotcha: a recursive CTE re-evaluates its recursive step once per level. A hierarchy that is ten levels deep runs the step ten times, and cycles without a guard can run forever. Terminate with a depth column and a `WHERE depth < n`, and make sure the join key inside the recursion is indexed.

### Set operations

`UNION` deduplicates, so it needs a sort or hash of the whole result. `UNION ALL` concatenates with no work. `INTERSECT` and `EXCEPT` are also set operations and also require deduplication. In ETL, `UNION ALL` is almost always what you want, because the rows are already distinct by construction and duplicate removal is pure overhead.

### Pushdown and pre-aggregation

The cheapest aggregation is the one over fewer rows. Two habits pay off:

1. Pre-aggregate before a join when both sides are large, so the join sees summarized rows.
2. Aggregate in stages - daily first, then monthly - instead of scanning raw events for a multi-year report.

```sql
-- Pre-aggregate each side, then join the small results
WITH order_totals AS (
    SELECT customer_id, SUM(amount) AS revenue, COUNT(*) AS order_count
    FROM orders GROUP BY customer_id
),
ticket_totals AS (
    SELECT customer_id, COUNT(*) AS ticket_count
    FROM support_tickets GROUP BY customer_id
)
SELECT c.customer_name, o.revenue, t.ticket_count
FROM customers c
LEFT JOIN order_totals o ON o.customer_id = c.customer_id
LEFT JOIN ticket_totals t ON t.customer_id = c.customer_id;
```

### Common mistakes

1. Filtering with `HAVING` when a `WHERE` would reduce the input.
2. Using `COUNT(DISTINCT ...) OVER (...)` and expecting it to be cheap.
3. Forgetting that the default window frame is `RANGE`, not `ROWS`.
4. Materialising a large CTE that is referenced once and would have been inlined.
5. `UNION` where `UNION ALL` is correct, paying for a useless deduplication.
6. Sorting the whole table when only the top partition matters - use `QUALIFY` or a filtered subquery where available.
7. Grouping by a wide column such as a full row or a long string, which makes the hash keys huge.

The cell below models the effect of duplicate ordering values on a running total. It is the exact difference between `ROWS` and `RANGE`, computed in plain Python.

In [9]:
events = [("A", 1, 10.0), ("A", 2, 20.0), ("A", 2, 30.0), ("A", 3, 40.0)]


def running_total(rows, frame):
    totals = {}
    for i, (key, order_value, amount) in enumerate(rows):
        if frame == "ROWS":
            peers = [r for r in rows[: i + 1] if r[0] == key]
        else:  # RANGE: all peers with the same ordering value, in either direction
            peers = [
                r
                for r in rows
                if r[0] == key and r[1] <= order_value
            ]
        totals[i] = round(sum(r[2] for r in peers), 2)
    return totals


rows_frame = running_total(events, "ROWS")
range_frame = running_total(events, "RANGE")
print("ROWS :", rows_frame)
print("RANGE:", range_frame)
assert rows_frame[1] == 30.0 and rows_frame[2] == 60.0
assert range_frame[1] == range_frame[2] == 60.0
assert rows_frame[3] == range_frame[3] == 100.0
print("duplicate dates make ROWS and RANGE disagree for the second row")

ROWS : {0: 10.0, 1: 30.0, 2: 60.0, 3: 100.0}
RANGE: {0: 10.0, 1: 60.0, 2: 60.0, 3: 100.0}
duplicate dates make ROWS and RANGE disagree for the second row


### Hash aggregate versus sort aggregate

A hash aggregate is usually faster because it is one pass with no ordering, but it needs memory for every distinct group. A sort aggregate uses less peak memory for small group counts but pays for the sort. If the group count is huge - grouping by a near-unique key - the hash table itself is the memory problem. That is when pre-aggregation or a coarser grain helps.

### Top-N per group without a full sort

`ROW_NUMBER()` plus a filtered subquery returns the top row per group. Engines with `QUALIFY` (Snowflake, BigQuery, Teradata) let you filter the window result directly, which is shorter and often plans the same.

```sql
SELECT * FROM (
    SELECT order_id, customer_id, amount,
           ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC) AS rn
    FROM orders
) ranked
WHERE rn <= 3;

-- Same idea where QUALIFY exists
SELECT order_id, customer_id, amount
FROM orders
QUALIFY ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC) <= 3;
```

### Window frame patterns worth memorising

- `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` - running total.
- `ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` - seven-row moving window.
- `ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING` - centred smoothing.
- `RANGE BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` - whole partition.
- `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` - whole partition, row by row.

The difference between the last two is subtle and real. `RANGE ... UNBOUNDED FOLLOWING` includes every row in the partition, and for a running metric it gives the grand total. The wrong frame is the most common window bug, because the default silently changes when an `ORDER BY` is present.

### Diagnosing a spill

A spill shows up as "disk" or "external" in the plan, or as a temporary-file counter in the engine's statistics. The causes are a large sort key, a large group count, a wide row carried through the sort, or a skewed partition. The fixes are to select fewer columns before the sort, reduce the rows with a `WHERE`, pre-aggregate, or increase the memory grant.

### Approximate aggregates

For very large distinct counts, exact `COUNT(DISTINCT ...)` needs memory proportional to the number of distinct values. HyperLogLog-style approximations (`approx_count_distinct`, `APPROX_COUNT_DISTINCT`, `HLL_COUNT`) trade a fraction of a percent of error for an order of magnitude less memory. They are usually the right choice for a dashboard and the wrong choice for an invoice.

### GROUPING SETS, ROLLUP, and CUBE

These compute several grouping levels in one pass instead of one query per level. `ROLLUP` walks a hierarchy from the finest grain to the total; `CUBE` produces every combination; `GROUPING SETS` lists exactly the levels you want. They are the SQL-native answer to "give me the totals too".

```sql
SELECT region, category, SUM(amount) AS revenue
FROM orders
GROUP BY ROLLUP (region, category)
ORDER BY region, category;
```

In [10]:
def window_sort_memory(rows, key_width_bytes, value_width_bytes):
    # Bytes the engine must hold while sorting one partition.
    return rows * (key_width_bytes + value_width_bytes)


for rows in (1_000, 1_000_000, 100_000_000):
    gigabytes = window_sort_memory(rows, 16, 32) / 1024 ** 3
    print(f"rows={rows:>12,} sort memory={gigabytes:7.2f} GiB")

assert window_sort_memory(100_000_000, 16, 32) > window_sort_memory(1_000_000, 16, 32)
assert round(window_sort_memory(1_000_000, 16, 32) / 1024 ** 3, 2) == 0.04
print("wide rows multiply the sort memory, so project only what you need")

rows=       1,000 sort memory=   0.00 GiB
rows=   1,000,000 sort memory=   0.04 GiB
rows= 100,000,000 sort memory=   4.47 GiB
wide rows multiply the sort memory, so project only what you need


### Common aggregation mistakes, condensed

1. `HAVING` where `WHERE` would reduce the input.
2. Relying on the default `RANGE` frame when `ROWS` was meant.
3. `COUNT(DISTINCT ...) OVER (...)` expected to be cheap.
4. Grouping by a wide text column, which inflates every hash key.
5. `UNION` instead of `UNION ALL`, paying for deduplication.
6. Materialising a CTE that is referenced once and would have inlined.

### Interview follow-ups on aggregation

- How do hash and sort aggregation differ, and when does each win?
- What is the difference between `ROWS` and `RANGE` in a window frame?
- When would you use `QUALIFY` rather than a subquery?
- How do you avoid spilling in a large window function?
- When is an approximate distinct count acceptable?

### A tuning story, end to end

A nightly report grew from 20 seconds to 12 minutes over six months. It joins a 400-million-row fact table to four dimensions, filters on a date range, and groups by region. Nothing in the SQL changed; the data grew.

The plan showed a nested loop with the fact table as the outer input and an unindexed dimension as the inner input. The estimate said 500 rows because the statistics were three months old. The actual outer count was 90 million, so the engine performed 90 million scans of the dimension.

The fix, in order: refresh statistics so the estimate became realistic; add an index on the dimension join key; filter the date range before the join instead of after; and switch the aggregation to a pre-aggregated CTE. The report came back to under 30 seconds, and the team scheduled `ANALYZE` after every nightly load so the estimates stayed honest.

### Quick reference

- Read the plan top-down, but diagnose at the operator where estimates and actuals diverge.
- Nested loop for small outer plus indexed inner; hash for large equi-joins; merge for sorted inputs.
- Sargable predicates, composite indexes ordered equality then range, covering indexes for hot queries.
- Reduce rows before joins, sorts, and windows; refresh statistics; re-measure every change.

Keep this chapter next to the coding questions. The same four ideas - read the plan, reduce rows early, choose the right join and access path, control memory - explain almost every slow query you will meet.

## SQL query tuning checklist

A repeatable order of operations beats guessing. Work top to bottom and stop when the query is fast enough.

**1. Measure before you touch anything.** Capture the plan and the wall-clock time. Write down the row count of each input table. A change without a baseline is not a tuning result.

**2. Read the plan.** Find the most expensive operator, not the most alarming name. Compare estimated rows against actual rows and locate the first operator where they diverge.

**3. Reduce rows as early as possible.** Push filters toward the tables, before joins and aggregations. Select only the columns you need, so covering indexes and column pruning can work.

**4. Fix the joins.** Check the algorithm against the table sizes, confirm the build side is the small side, and remove accidental fan-out. Use `EXISTS` for existence tests.

**5. Make predicates sargable and indexes useful.** No functions on indexed columns, matching types, composite indexes ordered equality first then range, covering indexes for the hottest queries.

**6. Tame memory.** Reduce the input to sorts and windows, pre-aggregate, avoid needless `DISTINCT` and `UNION`, and watch for spilled sorts and hashes.

**7. Refresh statistics.** After a large load, run `ANALYZE` (or the engine equivalent). Consider extended statistics for correlated columns.

**8. Verify, then stop.** Re-measure and confirm the plan changed the way you expected. Document the before and after, and roll back changes that did not help.

In an interview, give the checklist as a short story: measure, read the plan, reduce rows, fix the join, fix the predicate or index, control memory, re-measure. Then apply it to the query in front of you.

## Real world use cases for SQL

These scenarios show the full arc an interviewer wants: the business problem, the data volume, the table design, the query, and why that approach was chosen.

### 1. Daily revenue mart for a marketplace

Business context: finance needs a per-day, per-country revenue number by 07:00, and the number must reconcile with the payments ledger. Volume: 400 million orders, 3 billion order lines, 12 TB of raw data, one partition per day.

Design: `fact_order_lines` is partitioned by `order_date` and clustered by `country_code`; a small `dim_date` carries fiscal calendars; the mart is rebuilt incrementally for the last 2 days to absorb late-arriving refunds.

```sql
create table mart_daily_revenue as
select
  o.order_date,
  o.country_code,
  count(distinct o.order_id)              as orders,
  sum(oi.quantity * oi.unit_price)        as gross_revenue,
  sum(case when o.status = 'refunded'
           then -oi.quantity * oi.unit_price
           else oi.quantity * oi.unit_price end) as net_revenue
from fact_orders o
join fact_order_lines oi
  on oi.order_id = o.order_id
 and oi.order_date = o.order_date
where o.order_date between :run_date - interval '1 day' and :run_date
group by o.order_date, o.country_code;
```

Why this approach: the partition predicate prunes all but two days, so the daily job scans gigabytes instead of terabytes; `count(distinct order_id)` survives a join that multiplies order lines; and the explicit `case` keeps refunds negative instead of silently dropping them. Late-arriving data is handled by reprocessing a bounded window rather than by guessing.

### 2. Signup funnel and retention

Business context: the growth team wants conversion from landing to paid, and week-1 retention by signup cohort. Volume: 800 million clickstream events, 5 million users, 30 days of hot data.

Design: an `events` table partitioned by `event_date` with the user id in the clustering key; the funnel query counts distinct users per step rather than counting raw events.

```sql
with signups as (
  select user_id, min(event_date) as signup_date
  from events
  where event_name = 'signup'
  group by user_id
),
steps as (
  select
    e.user_id,
    s.signup_date,
    max(case when e.event_name = 'landing'      then 1 else 0 end) as saw_landing,
    max(case when e.event_name = 'trial_start'  then 1 else 0 end) as started_trial,
    max(case when e.event_name = 'purchase'     then 1 else 0 end) as purchased
  from events e
  join signups s on s.user_id = e.user_id
  where e.event_date >= date '2026-09-01'
  group by e.user_id, s.signup_date
)
select
  signup_date,
  count(*)                                        as signups,
  sum(saw_landing)                                as landed,
  sum(started_trial)                              as trials,
  sum(purchased)                                  as purchases,
  round(100.0 * sum(purchased) / count(*), 2)     as conversion_pct
from steps
group by signup_date
order by signup_date;
```

Why this approach: `max(case ...)` collapses each user's event history into one funnel row, so a user who triggers an event twice is counted once. Counting events instead of users is the classic funnel bug in interviews.

### 3. SCD2 history for a customer dimension

Business context: a warehouse must answer "what was this customer's segment on the date of the order?". Volume: 20 million customers, 300,000 attribute changes per day.

Design: `dim_customer_scd2` keeps one row per version with `valid_from`, `valid_to`, and `is_current`; a merge closes the old version and inserts the new one.

```sql
-- close the current version of changed customers
update dim_customer_scd2 as d
set valid_to = current_timestamp,
    is_current = false
where d.is_current
  and exists (
    select 1 from stg_customer_changes s
    where s.customer_id = d.customer_id
      and (s.segment <> d.segment or s.tier <> d.tier)
  );

-- insert the new versions
insert into dim_customer_scd2
  (customer_id, segment, tier, valid_from, valid_to, is_current)
select customer_id, segment, tier, current_timestamp, null, true
from stg_customer_changes;
```

Point-in-time join:

```sql
select o.order_id, o.order_date, c.segment
from fact_orders o
join dim_customer_scd2 c
  on c.customer_id = o.customer_id
 and o.order_date >= c.valid_from
 and o.order_date <  coalesce(c.valid_to, timestamp '9999-12-31');
```

Why this approach: an SCD2 join must be a range join on the validity window; joining on `customer_id` alone returns every historical version and multiplies the fact rows. A unique index on `(customer_id) where is_current` prevents two current rows per customer.

### 4. Deduplicating late-arriving events

Business context: mobile clients retry uploads, so the same event id can arrive up to 72 hours late and can arrive twice. Volume: 2 TB/day, 5% duplicate and late.

Design: land raw events append-only into `stg_events`, then deduplicate into `silver_events` by trust order: prefer the vendor's version, then the latest `ingested_at`.

```sql
insert into silver_events
select event_id, user_id, event_type, event_time, payload
from (
  select
    e.*,
    row_number() over (
      partition by event_id
      order by source_priority asc, ingested_at desc
    ) as rn
  from stg_events e
  where e.ingested_at >= current_timestamp - interval '72' hour
) ranked
where rn = 1;
```

Why this approach: `row_number` is deterministic because the ordering has a tiebreaker, and reprocessing a 72-hour window matches the late-arrival bound instead of rescanning all history. Writing reject counts to an audit table turns deduplication into a monitored data-quality metric.

### 5. Cohort retention by month

Business context: product wants the month-1, month-2, and month-3 retention of each signup cohort. Volume: 12 million users, 400 million activity rows.

Design: a `dim_user` table with `signup_month`, and `fact_activity` with one row per user per active month.

```sql
with activity as (
  select distinct
    u.signup_month,
    a.user_id,
    datediff(month, u.signup_date, a.activity_date) as month_number
  from fact_activity a
  join dim_user u on u.user_id = a.user_id
  where a.activity_date >= date '2026-01-01'
)
select
  signup_month,
  count(distinct case when month_number = 1 then user_id end) as month_1,
  count(distinct case when month_number = 2 then user_id end) as month_2,
  count(distinct case when month_number = 3 then user_id end) as month_3
from activity
group by signup_month
order by signup_month;
```

Why this approach: `datediff(month, ...)` creates the cohort index directly, and `count(distinct ...)` inside `case` returns each cohort's returning users without a self-join per period. Postgres and Snowflake both support `count(distinct ...) filter (where ...)`, which reads even cleaner.

### 6. Running balance and top-N per group in one report

Business context: operations wants each customer's running balance after every transaction, plus the three largest transactions per customer. Volume: 900 million ledger rows, queried for the last 90 days.

Design: `fact_ledger` is partitioned by `posting_date`; the report uses two window functions over the same scan.

```sql
with windowed as (
  select
    l.customer_id,
    l.posting_date,
    l.amount,
    sum(l.amount) over (
      partition by l.customer_id
      order by l.posting_date, l.ledger_id
      rows between unbounded preceding and current row
    ) as running_balance,
    row_number() over (
      partition by l.customer_id
      order by l.amount desc, l.ledger_id
    ) as amount_rank
  from fact_ledger l
  where l.posting_date >= current_date - interval '90' day
)
select
  customer_id,
  posting_date,
  amount,
  running_balance,
  case when amount_rank <= 3 then amount end as top3_amount
from windowed
order by customer_id, posting_date;
```

Why this approach: the explicit `rows between unbounded preceding and current row` frame makes the running total independent of ties in the ordering (the default `range` frame would include tied dates and produce a wrong balance). Both window functions share one scan of the partitioned range, which is why windowing beats a self-join per row here.

## SQL interview questions (practice notebook)

The self-contained practice notebook `notebooks/03_sql_interview_questions.ipynb` is merged below so this topic notebook stands alone: an in-memory SQLite setup followed by hands-on problems with asserts. Run it top to bottom.

---

# Practice notebook: SQL interview questions

The cells below are from `notebooks/03_sql_interview_questions.ipynb` and are merged here so this notebook is self-contained. Run them top to bottom; assertions confirm the expected behavior.

# SQL Interview Questions for Data Engineering

This notebook is a hands-on bank of the SQL questions that show up in data-engineering interviews. Everything
runs against a small **in-memory SQLite** database, so there is nothing to install or configure and no files are
written. It is deliberately granular: every idea gets its own explanation and its own small cell, so you can
stop, predict the output, and only then run the cell.

**How to use it**

1. Run the cells in order (`Shift+Enter`), or use *Run All*.
2. Try to predict each result before you run the cell, then read the verification `assert`s.
3. Outputs are saved in the notebook, so it reads like a document without a kernel.
4. The last section is a bank of theory questions with model answers.

The SQL uses features that SQLite has supported for years: window functions, CTEs, `FILTER`, `LAG`/`LEAD` and
`GROUP_CONCAT ... ORDER BY`. Everything is the standard library (`sqlite3`) and the results are deterministic.

Companion notebooks: `01_python_basics.ipynb` (Python), `02_python_coding_problems.ipynb` (algorithms),
`04_pyspark_concepts.ipynb` (Spark), `05_interview_questions.ipynb` (mixed question bank).

## Setup: an in-memory SQLite database

The setup is four small steps: open a connection, define two helpers, create and seed six tables, then check the
seed. Six tables keep the examples realistic:

| table | grain | notable columns |
| --- | --- | --- |
| `departments` | one row per department | `dept_id`, `dept_name`, `location` |
| `employees` | one row per employee | `manager_id` (self-reference), `salary`, `hire_date` |
| `employee_updates` | append-only history | `emp_id`, `salary`, `updated_at` (several rows per employee) |
| `customers` | one row per customer | `region`, `signup_date` |
| `orders` | one row per order | `status`, `amount`, `order_date` |
| `events` | one row per event | `user_id`, `event_ts`, `event_type` |

### The connection

`sqlite3.connect(":memory:")` opens a private, throwaway database that lives only for this kernel session, so
re-running the notebook always starts from the same clean state. The module is part of the standard library and
the bundled SQLite is recent enough for window functions.

In [1]:
import sqlite3

conn = sqlite3.connect(":memory:")
print("sqlite version:", sqlite3.sqlite_version)

sqlite version: 3.50.4


### Helper: `run`

`run(sql, params=())` executes one statement with optional bind parameters, commits, and returns every row as a
list of tuples. It is the helper the verification cells use, which keeps the `assert`s short and readable.

In [2]:
def run(sql, params=()):
    """Execute one SQL statement and return all rows as a list of tuples."""
    rows = conn.execute(sql, params).fetchall()
    conn.commit()
    return rows

### Helper: `show`

`show(sql, params=())` executes one query, prints it as a compact ASCII table, and also returns the rows. It is
what the answer cells call, so the answer is visible in the saved output instead of only existing inside an
`assert`.

In [3]:
def show(sql, params=(), max_rows=30):
    """Print the result of one query as a small table, and return the rows."""
    cur = conn.execute(sql, params)
    columns = [description[0] for description in cur.description]
    rows = cur.fetchall()
    widths = [len(name) for name in columns]
    for row in rows[:max_rows]:
        for i, value in enumerate(row):
            widths[i] = max(widths[i], len(str(value)))

    print(" | ".join(name.ljust(widths[i]) for i, name in enumerate(columns)))
    print("-+-".join("-" * width for width in widths))
    for row in rows[:max_rows]:
        print(" | ".join(str(value).ljust(widths[i]) for i, value in enumerate(row)))
    if len(rows) > max_rows:
        print(f"... {len(rows) - max_rows} more rows")
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})")
    return rows

### Seed data: six tables

Each table below is created and seeded in its own cell. `employees.emp_id` 1-14 covers four departments, with
two departments that have no employees below the average, one tied salary pair (Bob and Carol at 120000) and two
duplicate email pairs. `customers` 7 and 8 have never ordered, which the anti-join questions rely on.

#### `departments` (4 rows)

The small dimension table. `dept_id` is the primary key, `dept_name` is unique in practice, and `location` is a
plain attribute. Every employee references one of these rows, so this table must be seeded first.

In [4]:
run('''
CREATE TABLE departments (
    dept_id   INTEGER PRIMARY KEY,
    dept_name TEXT NOT NULL,
    location  TEXT NOT NULL
);
''')

run('''
INSERT INTO departments (dept_id, dept_name, location) VALUES
    (1, 'Engineering', 'New York'),
    (2, 'Sales',       'Chicago'),
    (3, 'Marketing',   'Austin'),
    (4, 'Support',     'Remote');
''')

show("SELECT * FROM departments ORDER BY dept_id")

dept_id | dept_name   | location
--------+-------------+---------
1       | Engineering | New York
2       | Sales       | Chicago 
3       | Marketing   | Austin  
4       | Support     | Remote  
(4 rows)


[(1, 'Engineering', 'New York'),
 (2, 'Sales', 'Chicago'),
 (3, 'Marketing', 'Austin'),
 (4, 'Support', 'Remote')]

#### `employees` (14 rows)

The workhorse table. `emp_id` is the primary key, `dept_id` references `departments`, and `manager_id` is a
**self-reference** to `employees.emp_id` - `NULL` marks the top of the hierarchy (Alice, Frank, Judy, Leo).
`salary` is stored as an integer and `hire_date` as an ISO-8601 string, so lexical and chronological order agree.
`email` is deliberately *not* unique: Dave/Kevin and Heidi/Nina share one.

In [5]:
run('''
CREATE TABLE employees (
    emp_id     INTEGER PRIMARY KEY,
    name       TEXT NOT NULL,
    dept_id    INTEGER NOT NULL REFERENCES departments(dept_id),
    manager_id INTEGER REFERENCES employees(emp_id),
    salary     INTEGER NOT NULL,
    hire_date  TEXT NOT NULL,
    email      TEXT NOT NULL
);
''')

run('''
INSERT INTO employees (emp_id, name, dept_id, manager_id, salary, hire_date, email) VALUES
    (1,  'Alice',   1, NULL, 150000, '2018-01-15', 'alice@example.com'),
    (2,  'Bob',     1, 1,    120000, '2019-03-01', 'bob@example.com'),
    (3,  'Carol',   1, 1,    120000, '2020-06-15', 'carol@example.com'),
    (4,  'Dave',    1, 2,     95000, '2021-09-01', 'dave@example.com'),
    (5,  'Eve',     1, 2,     90000, '2022-01-10', 'eve@example.com'),
    (6,  'Frank',   2, NULL, 130000, '2017-05-20', 'frank@example.com'),
    (7,  'Grace',   2, 6,     98000, '2020-02-14', 'grace@example.com'),
    (8,  'Heidi',   2, 6,     87000, '2021-07-01', 'heidi@example.com'),
    (9,  'Ivan',    2, 6,     87000, '2022-03-15', 'ivan@example.com'),
    (10, 'Judy',    3, NULL, 105000, '2019-11-01', 'judy@example.com'),
    (11, 'Kevin',   3, 10,    88000, '2021-04-01', 'dave@example.com'),
    (12, 'Leo',     4, NULL,  90000, '2018-08-08', 'leo@example.com'),
    (13, 'Mallory', 4, 12,    72000, '2022-05-05', 'mallory@example.com'),
    (14, 'Nina',    4, 12,    68000, '2023-02-01', 'heidi@example.com');
''')

print("employees seeded:", run("SELECT COUNT(*) FROM employees")[0][0])

employees seeded: 14


#### `employee_updates` (8 rows)

An **append-only history**: several rows per `emp_id`, each recording a new salary at an `updated_at` timestamp.
There is no primary key, so `rowid` is the only per-row identifier - which is why the deduplication questions use
it as a tie-breaker. Only employees 1, 2, 3, 6 and 7 appear here.

In [6]:
run('''
CREATE TABLE employee_updates (
    emp_id     INTEGER NOT NULL,
    dept_id    INTEGER NOT NULL,
    salary     INTEGER NOT NULL,
    updated_at TEXT NOT NULL
);
''')

run('''
INSERT INTO employee_updates (emp_id, dept_id, salary, updated_at) VALUES
    (1, 1, 155000, '2023-01-01'),
    (1, 1, 165000, '2023-06-01'),
    (2, 1, 122000, '2023-02-01'),
    (2, 1, 128000, '2023-05-01'),
    (3, 1, 125000, '2023-03-01'),
    (6, 2, 135000, '2023-01-15'),
    (6, 2, 140000, '2023-07-01'),
    (7, 2, 101000, '2023-04-01');
''')

show("SELECT * FROM employee_updates ORDER BY emp_id, updated_at")

emp_id | dept_id | salary | updated_at
-------+---------+--------+-----------
1      | 1       | 155000 | 2023-01-01
1      | 1       | 165000 | 2023-06-01
2      | 1       | 122000 | 2023-02-01
2      | 1       | 128000 | 2023-05-01
3      | 1       | 125000 | 2023-03-01
6      | 2       | 135000 | 2023-01-15
6      | 2       | 140000 | 2023-07-01
7      | 2       | 101000 | 2023-04-01
(8 rows)


[(1, 1, 155000, '2023-01-01'),
 (1, 1, 165000, '2023-06-01'),
 (2, 1, 122000, '2023-02-01'),
 (2, 1, 128000, '2023-05-01'),
 (3, 1, 125000, '2023-03-01'),
 (6, 2, 135000, '2023-01-15'),
 (6, 2, 140000, '2023-07-01'),
 (7, 2, 101000, '2023-04-01')]

#### `customers` (8 rows)

One row per customer with a `region` and a `signup_date`. Customers 7 (`Gamma SA`) and 8 (`Hotel BV`) never place
an order, so they are the fixed answer to every "who has no orders" question.

In [7]:
run('''
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    region      TEXT NOT NULL,
    signup_date TEXT NOT NULL
);
''')

run('''
INSERT INTO customers (customer_id, name, region, signup_date) VALUES
    (1, 'Acme Corp',   'North', '2022-01-05'),
    (2, 'Beta LLC',    'North', '2022-02-10'),
    (3, 'Cyan Inc',    'South', '2022-03-15'),
    (4, 'Delta Co',    'South', '2022-04-20'),
    (5, 'Echo Ltd',    'West',  '2022-05-25'),
    (6, 'Foxtrot GmbH','West',  '2022-06-30'),
    (7, 'Gamma SA',    'East',  '2023-01-10'),
    (8, 'Hotel BV',    'East',  '2023-02-15');
''')

show("SELECT * FROM customers ORDER BY customer_id")

customer_id | name         | region | signup_date
------------+--------------+--------+------------
1           | Acme Corp    | North  | 2022-01-05 
2           | Beta LLC     | North  | 2022-02-10 
3           | Cyan Inc     | South  | 2022-03-15 
4           | Delta Co     | South  | 2022-04-20 
5           | Echo Ltd     | West   | 2022-05-25 
6           | Foxtrot GmbH | West   | 2022-06-30 
7           | Gamma SA     | East   | 2023-01-10 
8           | Hotel BV     | East   | 2023-02-15 
(8 rows)


[(1, 'Acme Corp', 'North', '2022-01-05'),
 (2, 'Beta LLC', 'North', '2022-02-10'),
 (3, 'Cyan Inc', 'South', '2022-03-15'),
 (4, 'Delta Co', 'South', '2022-04-20'),
 (5, 'Echo Ltd', 'West', '2022-05-25'),
 (6, 'Foxtrot GmbH', 'West', '2022-06-30'),
 (7, 'Gamma SA', 'East', '2023-01-10'),
 (8, 'Hotel BV', 'East', '2023-02-15')]

#### `orders` (14 rows)

One row per order. `customer_id` references `customers`, `status` is one of `completed`, `pending` or
`cancelled`, `amount` is an integer, and `order_date` is an ISO-8601 date. The mix of statuses matters: most
questions filter to `completed`, and the pivot question counts all three.

In [8]:
run('''
CREATE TABLE orders (
    order_id    INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL REFERENCES customers(customer_id),
    status      TEXT NOT NULL,
    amount      INTEGER NOT NULL,
    order_date  TEXT NOT NULL
);
''')

run('''
INSERT INTO orders (order_id, customer_id, status, amount, order_date) VALUES
    (1,  1, 'completed', 100, '2023-01-05'),
    (2,  1, 'completed', 200, '2023-02-10'),
    (3,  1, 'completed', 150, '2023-03-15'),
    (4,  2, 'completed', 300, '2023-01-20'),
    (5,  2, 'cancelled', 250, '2023-02-05'),
    (6,  2, 'completed', 400, '2023-03-20'),
    (7,  3, 'completed', 120, '2023-01-25'),
    (8,  3, 'completed', 180, '2023-02-28'),
    (9,  4, 'pending',   220, '2023-02-14'),
    (10, 4, 'completed', 320, '2023-03-30'),
    (11, 5, 'completed', 500, '2023-01-02'),
    (12, 5, 'completed', 450, '2023-03-05'),
    (13, 6, 'completed', 275, '2023-02-18'),
    (14, 6, 'cancelled', 325, '2023-03-25');
''')

show("SELECT * FROM orders ORDER BY order_id")

order_id | customer_id | status    | amount | order_date
---------+-------------+-----------+--------+-----------
1        | 1           | completed | 100    | 2023-01-05
2        | 1           | completed | 200    | 2023-02-10
3        | 1           | completed | 150    | 2023-03-15
4        | 2           | completed | 300    | 2023-01-20
5        | 2           | cancelled | 250    | 2023-02-05
6        | 2           | completed | 400    | 2023-03-20
7        | 3           | completed | 120    | 2023-01-25
8        | 3           | completed | 180    | 2023-02-28
9        | 4           | pending   | 220    | 2023-02-14
10       | 4           | completed | 320    | 2023-03-30
11       | 5           | completed | 500    | 2023-01-02
12       | 5           | completed | 450    | 2023-03-05
13       | 6           | completed | 275    | 2023-02-18
14       | 6           | cancelled | 325    | 2023-03-25
(14 rows)


[(1, 1, 'completed', 100, '2023-01-05'),
 (2, 1, 'completed', 200, '2023-02-10'),
 (3, 1, 'completed', 150, '2023-03-15'),
 (4, 2, 'completed', 300, '2023-01-20'),
 (5, 2, 'cancelled', 250, '2023-02-05'),
 (6, 2, 'completed', 400, '2023-03-20'),
 (7, 3, 'completed', 120, '2023-01-25'),
 (8, 3, 'completed', 180, '2023-02-28'),
 (9, 4, 'pending', 220, '2023-02-14'),
 (10, 4, 'completed', 320, '2023-03-30'),
 (11, 5, 'completed', 500, '2023-01-02'),
 (12, 5, 'completed', 450, '2023-03-05'),
 (13, 6, 'completed', 275, '2023-02-18'),
 (14, 6, 'cancelled', 325, '2023-03-25')]

#### `events` (15 rows)

One row per user event with an ISO-8601 **timestamp** (`YYYY-MM-DD HH:MM:SS`) and an `event_type`. User 1 has
consecutive daily logins (a gaps-and-islands case), user 3 has tight timestamps then a 65-minute gap (a
sessionization case), and users 2-4 are small enough to verify by hand.

In [9]:
run('''
CREATE TABLE events (
    event_id   INTEGER PRIMARY KEY,
    user_id    INTEGER NOT NULL,
    event_ts   TEXT NOT NULL,
    event_type TEXT NOT NULL
);
''')

run('''
INSERT INTO events (event_id, user_id, event_ts, event_type) VALUES
    (1,  1, '2023-01-01 09:00:00', 'login'),
    (2,  1, '2023-01-02 09:05:00', 'login'),
    (3,  1, '2023-01-03 08:55:00', 'login'),
    (4,  1, '2023-01-05 09:00:00', 'login'),
    (5,  1, '2023-01-06 09:10:00', 'login'),
    (6,  2, '2023-01-01 10:00:00', 'login'),
    (7,  2, '2023-01-02 10:00:00', 'login'),
    (8,  2, '2023-01-04 10:00:00', 'login'),
    (9,  3, '2023-02-01 10:00:00', 'view'),
    (10, 3, '2023-02-01 10:05:00', 'click'),
    (11, 3, '2023-02-01 10:25:00', 'view'),
    (12, 3, '2023-02-01 11:30:00', 'purchase'),
    (13, 3, '2023-02-01 11:45:00', 'view'),
    (14, 4, '2023-02-01 10:00:00', 'login'),
    (15, 4, '2023-02-01 10:20:00', 'view');
''')

show("SELECT * FROM events ORDER BY event_id")

event_id | user_id | event_ts            | event_type
---------+---------+---------------------+-----------
1        | 1       | 2023-01-01 09:00:00 | login     
2        | 1       | 2023-01-02 09:05:00 | login     
3        | 1       | 2023-01-03 08:55:00 | login     
4        | 1       | 2023-01-05 09:00:00 | login     
5        | 1       | 2023-01-06 09:10:00 | login     
6        | 2       | 2023-01-01 10:00:00 | login     
7        | 2       | 2023-01-02 10:00:00 | login     
8        | 2       | 2023-01-04 10:00:00 | login     
9        | 3       | 2023-02-01 10:00:00 | view      
10       | 3       | 2023-02-01 10:05:00 | click     
11       | 3       | 2023-02-01 10:25:00 | view      
12       | 3       | 2023-02-01 11:30:00 | purchase  
13       | 3       | 2023-02-01 11:45:00 | view      
14       | 4       | 2023-02-01 10:00:00 | login     
15       | 4       | 2023-02-01 10:20:00 | view      
(15 rows)


[(1, 1, '2023-01-01 09:00:00', 'login'),
 (2, 1, '2023-01-02 09:05:00', 'login'),
 (3, 1, '2023-01-03 08:55:00', 'login'),
 (4, 1, '2023-01-05 09:00:00', 'login'),
 (5, 1, '2023-01-06 09:10:00', 'login'),
 (6, 2, '2023-01-01 10:00:00', 'login'),
 (7, 2, '2023-01-02 10:00:00', 'login'),
 (8, 2, '2023-01-04 10:00:00', 'login'),
 (9, 3, '2023-02-01 10:00:00', 'view'),
 (10, 3, '2023-02-01 10:05:00', 'click'),
 (11, 3, '2023-02-01 10:25:00', 'view'),
 (12, 3, '2023-02-01 11:30:00', 'purchase'),
 (13, 3, '2023-02-01 11:45:00', 'view'),
 (14, 4, '2023-02-01 10:00:00', 'login'),
 (15, 4, '2023-02-01 10:20:00', 'view')]

### Row-count sanity check

Before any question runs, confirm that every table holds exactly the number of rows it should. A broken seed
would otherwise surface as a confusing assertion failure twenty cells later.

In [10]:
expected_counts = {
    "departments": 4,
    "employees": 14,
    "employee_updates": 8,
    "customers": 8,
    "orders": 14,
    "events": 15,
}

for table, expected in expected_counts.items():
    actual = run(f"SELECT COUNT(*) FROM {table}")[0][0]
    assert actual == expected, (table, actual, expected)
    print(f"{table:<17} {actual:>3} rows")

departments         4 rows
employees          14 rows
employee_updates    8 rows
customers           8 rows
orders             14 rows
events             15 rows


### Structural invariants

Three shape checks that the questions depend on: four managers at the top of the hierarchy, four departments
represented by employees, and exactly two customers who have never ordered.

In [11]:
assert run("SELECT COUNT(*) FROM employees WHERE manager_id IS NULL") == [(4,)]
assert run("SELECT COUNT(DISTINCT dept_id) FROM employees") == [(4,)]
assert run(
    """
    SELECT COUNT(*) FROM customers c
    WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
    """
) == [(2,)]
print("seed verified")

seed verified


---
## Part 1 - Ranking and top-N

`ROW_NUMBER`, `RANK` and `DENSE_RANK` all number rows inside a `PARTITION BY`, but they break ties differently.
Almost every "top N per group" interview question is really a question about which one you need.

### Q1. Second-highest salary

Return the second-highest salary in `employees`. Repeated salaries count as one value.

**Expected result.** One row, `(130000,)`: Alice earns 150000 and Frank earns 130000, so 130000 is the runner-up.

In [12]:
show("""
SELECT MAX(salary) AS second_highest
FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees);
""")

second_highest
--------------
130000        
(1 row)


[(130000,)]

**Check 1 - the subquery answer.** `MAX(salary)` below the overall maximum is the second-highest value, because
`<` excludes every copy of the top salary.

In [13]:
assert run("""
SELECT MAX(salary) FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees)
""") == [(130000,)]
print("second-highest salary is 130000")

second-highest salary is 130000


**Check 2 - the window-function alternative.** Rank *distinct* salaries in descending order and keep rank 2;
`DENSE_RANK` gives ties the same rank, so no salary can be duplicated.

In [14]:
assert run("""
SELECT salary FROM (
    SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk
    FROM employees
) WHERE rnk = 2
""") == [(130000,)]
print("DENSE_RANK agrees")

DENSE_RANK agrees


**Approach and pitfalls.** `MAX(salary) WHERE salary < (SELECT MAX(salary) ...)` is the classic one-liner. The
trap is the naive `ORDER BY salary DESC LIMIT 1 OFFSET 1`, which returns the same salary again when the top salary
is tied. Subtracting the maximum also degrades gracefully: if only one salary exists the result is `NULL` rather
than an error.

**Follow-ups.**

1. How would you return the whole row of the second-highest earner, not just the salary?
2. What does `DENSE_RANK` return instead of `NULL` when the table has a single salary?

### Q2. Nth-highest salary

Return the Nth-highest distinct salary, with N passed as a bind parameter. Here N = 3.

**Expected result.** One row, `(120000,)`. The distinct salaries in descending order start 150000, 130000,
120000 - and 120000 is shared by Bob and Carol, which is exactly where naive ranking implementations go wrong.

In [15]:
n = 3  # the rank we want

show(
    """
    SELECT DISTINCT salary
    FROM employees
    ORDER BY salary DESC
    LIMIT 1 OFFSET ?;
    """,
    (n - 1,),
)

salary
------
120000
(1 row)


[(120000,)]

**Check 1 - the portable answer.** `SELECT DISTINCT salary ... LIMIT 1 OFFSET n-1` treats the ordered distinct
values as a zero-based list, so N = 3 means offset 2.

In [16]:
n = 3
assert run(
    "SELECT DISTINCT salary FROM employees ORDER BY salary DESC LIMIT 1 OFFSET ?",
    (n - 1,),
) == [(120000,)]
print("Nth-highest via LIMIT/OFFSET:", 120000)

Nth-highest via LIMIT/OFFSET: 120000


**Check 2 - the window answer.** `DENSE_RANK` assigns one rank per distinct salary, so `DISTINCT + rnk = n`
still yields exactly one value.

In [17]:
assert run(
    """
    SELECT DISTINCT salary FROM (
        SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk
        FROM employees
    ) WHERE rnk = ?
    """,
    (n,),
) == [(120000,)]
print("Nth-highest via DENSE_RANK:", 120000)

Nth-highest via DENSE_RANK: 120000


**Check 3 - why `DISTINCT` matters.** Dropping it exposes the tie: rank 3 holds two employees at 120000.

In [18]:
assert run(
    """
    SELECT salary FROM (
        SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk
        FROM employees
    ) WHERE rnk = ?
    """,
    (n,),
) == [(120000,), (120000,)]
print("rank 3 holds two tied rows")

rank 3 holds two tied rows


**Approach and pitfalls.** `LIMIT 1 OFFSET n-1` over `SELECT DISTINCT salary` is portable and cheap. The window
version reads better in a longer query but must be wrapped in `DISTINCT` (or `MIN`) when ties exist, because
`DENSE_RANK` numbers ties with the same rank and therefore emits several rows. `ROW_NUMBER` never emits ties but
also never gives you a stable definition of "Nth".

**Follow-ups.**

1. What should the query return when N is larger than the number of distinct salaries?
2. Which index would make `ORDER BY salary DESC` avoid a sort?

### Q3. Top-N per group: ROW_NUMBER vs RANK vs DENSE_RANK

Return the top 3 employees by salary *per department*, then compare how many rows each ranking function keeps.

**Expected result.** The top-3 list has 11 rows (department 2 has only three slots filled because Heidi and Ivan
tie at 87000). The comparison shows `ROW_NUMBER` keeping exactly 3 per department, `RANK` keeping 4 for Sales, and
`DENSE_RANK` keeping 4 for Sales *and* 4 for Engineering - the tie at 120000 lets rank 3 exist there.

In [19]:
show("""
SELECT dept_id, name, salary
FROM (
    SELECT dept_id, name, salary,
           ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY salary DESC, emp_id) AS rn
    FROM employees
) ranked
WHERE rn <= 3
ORDER BY dept_id, salary DESC, name;
""")

dept_id | name    | salary
--------+---------+-------
1       | Alice   | 150000
1       | Bob     | 120000
1       | Carol   | 120000
2       | Frank   | 130000
2       | Grace   | 98000 
2       | Heidi   | 87000 
3       | Judy    | 105000
3       | Kevin   | 88000 
4       | Leo     | 90000 
4       | Mallory | 72000 
4       | Nina    | 68000 
(11 rows)


[(1, 'Alice', 150000),
 (1, 'Bob', 120000),
 (1, 'Carol', 120000),
 (2, 'Frank', 130000),
 (2, 'Grace', 98000),
 (2, 'Heidi', 87000),
 (3, 'Judy', 105000),
 (3, 'Kevin', 88000),
 (4, 'Leo', 90000),
 (4, 'Mallory', 72000),
 (4, 'Nina', 68000)]

**Check 1 - the top-3 list.** `ROW_NUMBER` with a deterministic tie-breaker (`emp_id`) picks exactly three rows
per department, and the result is stable across runs.

In [20]:
n = 3
assert run(
    """
    SELECT dept_id, name, salary FROM (
        SELECT dept_id, name, salary,
               ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY salary DESC, emp_id) AS rn
        FROM employees
    ) WHERE rn <= ?
    ORDER BY dept_id, salary DESC, name
    """,
    (n,),
) == [
    (1, "Alice", 150000), (1, "Bob", 120000), (1, "Carol", 120000),
    (2, "Frank", 130000), (2, "Grace", 98000), (2, "Heidi", 87000),
    (3, "Judy", 105000), (3, "Kevin", 88000),
    (4, "Leo", 90000), (4, "Mallory", 72000), (4, "Nina", 68000),
]
print("top-3 per department:", 11, "rows")

top-3 per department: 11 rows


**Check 2 - how many rows each function keeps at the same threshold.** Count, per department, how many rows pass
`<= 3` under each ranking function.

In [21]:
assert run(
    """
    SELECT dept_id,
           COUNT(CASE WHEN rn   <= 3 THEN 1 END) AS by_row_number,
           COUNT(CASE WHEN rnk  <= 3 THEN 1 END) AS by_rank,
           COUNT(CASE WHEN drnk <= 3 THEN 1 END) AS by_dense_rank
    FROM (
        SELECT dept_id, salary,
               ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY salary DESC, emp_id) AS rn,
               RANK()       OVER (PARTITION BY dept_id ORDER BY salary DESC) AS rnk,
               DENSE_RANK() OVER (PARTITION BY dept_id ORDER BY salary DESC) AS drnk
        FROM employees
    )
    GROUP BY dept_id ORDER BY dept_id
    """
) == [(1, 3, 3, 4), (2, 3, 4, 4), (3, 2, 2, 2), (4, 3, 3, 3)]
print("ROW_NUMBER keeps 3, RANK/DENSE_RANK keep the ties")

ROW_NUMBER keeps 3, RANK/DENSE_RANK keep the ties


**Check 3 - the mechanism, seen directly.** Department 1 has the 120000 tie, so `RANK` skips to 4 while
`DENSE_RANK` moves to 3. The `WINDOW w AS (...)` clause names the shared window definition once.

In [22]:
assert run(
    """
    SELECT name, salary,
           ROW_NUMBER() OVER w AS rn,
           RANK()       OVER w AS rnk,
           DENSE_RANK() OVER w AS drnk
    FROM employees
    WHERE dept_id = 1
    WINDOW w AS (PARTITION BY dept_id ORDER BY salary DESC)
    ORDER BY salary DESC, emp_id
    """
) == [
    ("Alice", 150000, 1, 1, 1),
    ("Bob", 120000, 2, 2, 2),
    ("Carol", 120000, 3, 2, 2),
    ("Dave", 95000, 4, 4, 3),
    ("Eve", 90000, 5, 5, 4),
]
print("Q3 passed")

Q3 passed


**Approach and pitfalls.** Filter on the ranking in an outer query - you cannot reference a window alias in the
same `WHERE` clause, because window functions are evaluated after `WHERE`. `ROW_NUMBER` is the right choice for
"exactly N rows per group"; use `RANK`/`DENSE_RANK` when ties should all survive (for example "everyone in the top
3 salaries"). Always add a deterministic tie-breaker such as `emp_id` to `ROW_NUMBER`'s `ORDER BY`, or the row you
keep is arbitrary and non-reproducible.

**Follow-ups.**

1. How would you return only the single highest-paid employee per department?
2. Why is `WHERE rn <= 3` allowed in the outer query but `WHERE ROW_NUMBER() OVER (...) <= 3` is not?

---
## Part 2 - Deduplication and history

Real tables arrive with duplicates: append-only history, retried loads, and dirty source systems. These questions
are about collapsing many rows per key into one, safely and reproducibly.

### Q4. Deduplicate, keeping the latest row

`employee_updates` is append-only: several rows per `emp_id`, each with an `updated_at` timestamp. Return one row
per employee - the most recent.

**Expected result.** Five rows: `(1, 1, 165000, '2023-06-01')`, `(2, 1, 128000, '2023-05-01')`,
`(3, 1, 125000, '2023-03-01')`, `(6, 2, 140000, '2023-07-01')`, `(7, 2, 101000, '2023-04-01')`.

In [23]:
show("""
SELECT emp_id, dept_id, salary, updated_at
FROM (
    SELECT emp_id, dept_id, salary, updated_at,
           ROW_NUMBER() OVER (PARTITION BY emp_id ORDER BY updated_at DESC, rowid DESC) AS rn
    FROM employee_updates
) ranked
WHERE rn = 1
ORDER BY emp_id;
""")

emp_id | dept_id | salary | updated_at
-------+---------+--------+-----------
1      | 1       | 165000 | 2023-06-01
2      | 1       | 128000 | 2023-05-01
3      | 1       | 125000 | 2023-03-01
6      | 2       | 140000 | 2023-07-01
7      | 2       | 101000 | 2023-04-01
(5 rows)


[(1, 1, 165000, '2023-06-01'),
 (2, 1, 128000, '2023-05-01'),
 (3, 1, 125000, '2023-03-01'),
 (6, 2, 140000, '2023-07-01'),
 (7, 2, 101000, '2023-04-01')]

**Check 1 - one row per employee.** Partition by `emp_id`, order by `updated_at` descending, and keep `rn = 1`.
`rowid DESC` breaks a timestamp tie deterministically.

In [24]:
assert run(
    """
    SELECT emp_id, dept_id, salary, updated_at FROM (
        SELECT emp_id, dept_id, salary, updated_at,
               ROW_NUMBER() OVER (PARTITION BY emp_id ORDER BY updated_at DESC, rowid DESC) AS rn
        FROM employee_updates
    ) WHERE rn = 1 ORDER BY emp_id
    """
) == [
    (1, 1, 165000, "2023-06-01"),
    (2, 1, 128000, "2023-05-01"),
    (3, 1, 125000, "2023-03-01"),
    (6, 2, 140000, "2023-07-01"),
    (7, 2, 101000, "2023-04-01"),
]
print("dedupe kept the latest row per employee")

dedupe kept the latest row per employee


**Check 2 - the `GROUP BY MAX` alternative.** Aggregate to the newest timestamp per employee, then rejoin to get
the whole row. It works here because no employee has two updates with the same timestamp.

In [25]:
assert run(
    """
    SELECT u.emp_id, u.dept_id, u.salary, u.updated_at
    FROM employee_updates u
    JOIN (SELECT emp_id, MAX(updated_at) AS max_ts FROM employee_updates GROUP BY emp_id) m
      ON m.emp_id = u.emp_id AND m.max_ts = u.updated_at
    ORDER BY u.emp_id
    """
) == [
    (1, 1, 165000, "2023-06-01"),
    (2, 1, 128000, "2023-05-01"),
    (3, 1, 125000, "2023-03-01"),
    (6, 2, 140000, "2023-07-01"),
    (7, 2, 101000, "2023-04-01"),
]
print("Q4 passed")

Q4 passed


**Approach and pitfalls.** `ROW_NUMBER() ... ORDER BY updated_at DESC` inside a subquery, filtered by `rn = 1`
out in the outer query, is the general recipe. The tie-breaker (`rowid DESC` here) matters: without it, two rows
sharing the maximum timestamp give a non-deterministic answer. The `GROUP BY MAX` + self-join variant breaks when
two rows share the maximum timestamp - it returns both. For large tables, filtering to the latest partition or
using `QUALIFY` (BigQuery/Snowflake) avoids the shuffle.

**Follow-ups.**

1. How would you keep the *first* row per key instead of the latest?
2. What changes if the dedupe key is the combination `(emp_id, dept_id)`?

### Q5. Delete duplicate rows in place

Physically delete the stale rows from a copy of `employee_updates`, keeping one row per `emp_id`. We work on a
scratch table so the seed data stays intact for the rest of the notebook.

**Expected result.** 8 rows before the `DELETE`, 5 after, with only the latest timestamp per employee remaining.

**The scratch copy.** `CREATE TABLE ... AS SELECT * FROM employee_updates` clones both the rows and the shape, so
the rest of the notebook still sees the original 8 rows.

In [26]:
run("DROP TABLE IF EXISTS updates_scratch")
run("CREATE TABLE updates_scratch AS SELECT * FROM employee_updates")

print("before:", run("SELECT COUNT(*) FROM updates_scratch")[0][0], "rows")

before: 8 rows


**The delete.** Compute the surviving `rowid`s first (rank 1 per employee) and delete everything not in that set.
SQLite and Postgres both read the pre-delete snapshot inside the subquery, so the set does not shrink as rows go.

In [27]:
run("""
DELETE FROM updates_scratch
WHERE rowid NOT IN (
    SELECT rowid FROM (
        SELECT rowid,
               ROW_NUMBER() OVER (PARTITION BY emp_id ORDER BY updated_at DESC, rowid DESC) AS rn
        FROM updates_scratch
    ) WHERE rn = 1
)
""")

show("SELECT emp_id, salary, updated_at FROM updates_scratch ORDER BY emp_id")

emp_id | salary | updated_at
-------+--------+-----------
1      | 165000 | 2023-06-01
2      | 128000 | 2023-05-01
3      | 125000 | 2023-03-01
6      | 140000 | 2023-07-01
7      | 101000 | 2023-04-01
(5 rows)


[(1, 165000, '2023-06-01'),
 (2, 128000, '2023-05-01'),
 (3, 125000, '2023-03-01'),
 (6, 140000, '2023-07-01'),
 (7, 101000, '2023-04-01')]

**Check 1 - five rows survive, one per employee.** Both the count and the surviving values are asserted, so a
delete that removes the wrong rows cannot pass.

In [28]:
assert run("SELECT COUNT(*) FROM updates_scratch") == [(5,)]
assert run("SELECT emp_id, salary, updated_at FROM updates_scratch ORDER BY emp_id") == [
    (1, 165000, "2023-06-01"),
    (2, 128000, "2023-05-01"),
    (3, 125000, "2023-03-01"),
    (6, 140000, "2023-07-01"),
    (7, 101000, "2023-04-01"),
]
print("scratch table now holds 5 rows")

scratch table now holds 5 rows


**Check 2 - the same result without window functions.** A correlated `EXISTS` deletes a row when a strictly newer
version of the same employee exists, using `rowid` to break timestamp ties.

In [29]:
run("DROP TABLE IF EXISTS updates_scratch2")
run("CREATE TABLE updates_scratch2 AS SELECT * FROM employee_updates")
run("""
DELETE FROM updates_scratch2
WHERE EXISTS (
    SELECT 1 FROM employee_updates keep
    WHERE keep.emp_id = updates_scratch2.emp_id
      AND (keep.updated_at > updates_scratch2.updated_at
           OR (keep.updated_at = updates_scratch2.updated_at AND keep.rowid > updates_scratch2.rowid))
)
""")
assert run("SELECT COUNT(*) FROM updates_scratch2") == [(5,)]
print("Q5 passed")

Q5 passed


**Approach and pitfalls.** The safe pattern is "identify survivors, delete everything else" - expressed as
`rowid NOT IN (SELECT rowid FROM ... WHERE rn = 1)`. Two traps: (1) the parallel `DELETE FROM t WHERE t.ctid NOT
IN (SELECT MAX(ctid) ...)` style fails when the tie-breaker is not unique; (2) an unqualified subquery that reads
the same table you are deleting from can behave differently across databases - in SQLite and Postgres the inner
`SELECT` sees the pre-delete snapshot, which is what you want. In production, do the delete inside a transaction,
verify the row count, and consider `CREATE TABLE ... AS SELECT` + swap when the table is large.

**Follow-ups.**

1. How would you delete duplicates but keep the row with the largest surrogate key?
2. Why is a `plain DELETE ... WHERE rowid IN (...)` often preferred to `NOT IN (...)`, and what happens if the
   subquery returns a `NULL`?

### Q6. Find duplicate emails

List every email address used by more than one employee.

**Expected result.** Two rows: `('dave@example.com', 2, 'Dave,Kevin')` and `('heidi@example.com', 2,
'Heidi,Nina')`. Everything else is unique.

In [30]:
show("""
SELECT email,
       COUNT(*) AS copies,
       group_concat(name, ',' ORDER BY name) AS people
FROM employees
GROUP BY email
HAVING COUNT(*) > 1
ORDER BY email;
""")

email             | copies | people    
------------------+--------+-----------
dave@example.com  | 2      | Dave,Kevin
heidi@example.com | 2      | Heidi,Nina
(2 rows)


[('dave@example.com', 2, 'Dave,Kevin'), ('heidi@example.com', 2, 'Heidi,Nina')]

**Check 1 - the duplicated addresses.** `GROUP BY email HAVING COUNT(*) > 1` is the whole answer, and
`GROUP_CONCAT ... ORDER BY` makes the companion names deterministic.

In [31]:
assert run(
    """
    SELECT email, COUNT(*), group_concat(name, ',' ORDER BY name)
    FROM employees GROUP BY email HAVING COUNT(*) > 1 ORDER BY email
    """
) == [
    ("dave@example.com", 2, "Dave,Kevin"),
    ("heidi@example.com", 2, "Heidi,Nina"),
]
print("two duplicate emails found")

two duplicate emails found


**Check 2 - the offending row ids.** This is the shape a cleanup job needs: the primary keys to rewrite, not the
aggregated count.

In [32]:
assert run(
    """
    SELECT emp_id, email FROM employees
    WHERE email IN (SELECT email FROM employees GROUP BY email HAVING COUNT(*) > 1)
    ORDER BY email, emp_id
    """
) == [
    (4, "dave@example.com"), (11, "dave@example.com"),
    (8, "heidi@example.com"), (14, "heidi@example.com"),
]
print("Q6 passed")

Q6 passed


**Approach and pitfalls.** `GROUP BY email HAVING COUNT(*) > 1` is the whole answer. The pitfall is case and
whitespace: `Dave@Example.com ` and `dave@example.com` are different strings but the same mailbox, so real
dedupe pipelines normalise first with `lower(trim(email))`. `GROUP_CONCAT ... ORDER BY` is SQLite-specific;
Postgres uses `string_agg(name, ',' ORDER BY name)` and MySQL `GROUP_CONCAT(name ORDER BY name SEPARATOR ',')`.

**Follow-ups.**

1. How would you enforce uniqueness so the duplicate can never be inserted again?
2. How would you write the same query against a normalised email (lower-cased) column?

### Q7. UPDATE from another table

Apply the latest `employee_updates.salary` to a scratch copy of `employees`. Employees with no update row must be
left untouched.

**Expected result.** Exactly five salaries change: Alice 165000, Bob 128000, Carol 125000, Frank 140000 and Grace
101000. The other nine employees keep their original salaries.

**The scratch copy.** As in Q5, work on `employees_scratch` so `employees` keeps its seed values for the later
questions.

In [33]:
run("DROP TABLE IF EXISTS employees_scratch")
run("CREATE TABLE employees_scratch AS SELECT * FROM employees")

print("copy created:", run("SELECT COUNT(*) FROM employees_scratch")[0][0], "employees")

copy created: 14 employees


**The update.** A correlated scalar subquery with `ORDER BY ... LIMIT 1` picks the newest salary per employee,
and `WHERE EXISTS` guards the rows that have no update at all.

In [34]:
run("""
UPDATE employees_scratch
SET salary = (
    SELECT u.salary
    FROM employee_updates u
    WHERE u.emp_id = employees_scratch.emp_id
    ORDER BY u.updated_at DESC
    LIMIT 1
)
WHERE EXISTS (SELECT 1 FROM employee_updates u WHERE u.emp_id = employees_scratch.emp_id)
""")

show("SELECT emp_id, name, salary FROM employees_scratch ORDER BY emp_id")

emp_id | name    | salary
-------+---------+-------
1      | Alice   | 165000
2      | Bob     | 128000
3      | Carol   | 125000
4      | Dave    | 95000 
5      | Eve     | 90000 
6      | Frank   | 140000
7      | Grace   | 101000
8      | Heidi   | 87000 
9      | Ivan    | 87000 
10     | Judy    | 105000
11     | Kevin   | 88000 
12     | Leo     | 90000 
13     | Mallory | 72000 
14     | Nina    | 68000 
(14 rows)


[(1, 'Alice', 165000),
 (2, 'Bob', 128000),
 (3, 'Carol', 125000),
 (4, 'Dave', 95000),
 (5, 'Eve', 90000),
 (6, 'Frank', 140000),
 (7, 'Grace', 101000),
 (8, 'Heidi', 87000),
 (9, 'Ivan', 87000),
 (10, 'Judy', 105000),
 (11, 'Kevin', 88000),
 (12, 'Leo', 90000),
 (13, 'Mallory', 72000),
 (14, 'Nina', 68000)]

**Check 1 - the full salary list.** Employees without an update keep their original value; the five who have one
carry the newest amount.

In [35]:
assert run("SELECT emp_id, salary FROM employees_scratch ORDER BY emp_id") == [
    (1, 165000), (2, 128000), (3, 125000), (4, 95000), (5, 90000),
    (6, 140000), (7, 101000), (8, 87000), (9, 87000), (10, 105000),
    (11, 88000), (12, 90000), (13, 72000), (14, 68000),
]
print("14 salaries, five of them updated")

14 salaries, five of them updated


**Check 2 - count the changed rows.** Joining the copy to the original shows exactly five differences, which
proves no row was overwritten with `NULL`.

In [36]:
assert run(
    """
    SELECT COUNT(*) FROM employees_scratch e
    JOIN employees o ON o.emp_id = e.emp_id AND o.salary <> e.salary
    """
) == [(5,)]
print("Q7 passed")

Q7 passed


**Approach and pitfalls.** The correlated scalar subquery with `ORDER BY ... LIMIT 1` picks the newest update,
and the `WHERE EXISTS` guards against overwriting rows with `NULL` when no update exists. Forgetting that guard
silently nulls the salary of every employee missing from the source table - one of the most common production
bugs. Postgres/MySQL also support `UPDATE ... FROM`; SQLite 3.33+ has `UPDATE ... FROM` too, but the correlated
form above is the most portable.

**Follow-ups.**

1. How would you record the previous salary in an audit table as part of the update?
2. What happens if the subquery can match more than one row and you forget `LIMIT 1`?

---
## Part 3 - Running totals and period-over-period growth

Window functions turn "for each row, also show the aggregate so far" into a single pass. These are the queries
behind dashboards: cumulative revenue, growth rates, and share of total.

### Q8. Running total (overall)

For every order in date order, show the cumulative amount across all orders.

**Expected result.** 14 rows. The running total reaches 1020 at the end of January, 2145 at the end of February
and 3790 at the last order on 2023-03-30.

In [37]:
show("""
SELECT order_id, order_date, amount,
       SUM(amount) OVER (ORDER BY order_date, order_id) AS running_total
FROM orders
ORDER BY order_date, order_id;
""")

order_id | order_date | amount | running_total
---------+------------+--------+--------------
11       | 2023-01-02 | 500    | 500          
1        | 2023-01-05 | 100    | 600          
4        | 2023-01-20 | 300    | 900          
7        | 2023-01-25 | 120    | 1020         
5        | 2023-02-05 | 250    | 1270         
2        | 2023-02-10 | 200    | 1470         
9        | 2023-02-14 | 220    | 1690         
13       | 2023-02-18 | 275    | 1965         
8        | 2023-02-28 | 180    | 2145         
12       | 2023-03-05 | 450    | 2595         
3        | 2023-03-15 | 150    | 2745         
6        | 2023-03-20 | 400    | 3145         
14       | 2023-03-25 | 325    | 3470         
10       | 2023-03-30 | 320    | 3790         
(14 rows)


[(11, '2023-01-02', 500, 500),
 (1, '2023-01-05', 100, 600),
 (4, '2023-01-20', 300, 900),
 (7, '2023-01-25', 120, 1020),
 (5, '2023-02-05', 250, 1270),
 (2, '2023-02-10', 200, 1470),
 (9, '2023-02-14', 220, 1690),
 (13, '2023-02-18', 275, 1965),
 (8, '2023-02-28', 180, 2145),
 (12, '2023-03-05', 450, 2595),
 (3, '2023-03-15', 150, 2745),
 (6, '2023-03-20', 400, 3145),
 (14, '2023-03-25', 325, 3470),
 (10, '2023-03-30', 320, 3790)]

**Check 1 - the cumulative series.** `SUM(amount) OVER (ORDER BY order_date, order_id)` is evaluated after
`WHERE` and walks the ordered rows. Adding the unique `order_id` to the `ORDER BY` makes the total advance row by
row instead of lumping same-date orders together.

In [38]:
assert run(
    """
    SELECT order_id, order_date, amount,
           SUM(amount) OVER (ORDER BY order_date, order_id) AS running_total
    FROM orders ORDER BY order_date, order_id
    """
) == [
    (11, "2023-01-02", 500, 500),
    (1, "2023-01-05", 100, 600),
    (4, "2023-01-20", 300, 900),
    (7, "2023-01-25", 120, 1020),
    (5, "2023-02-05", 250, 1270),
    (2, "2023-02-10", 200, 1470),
    (9, "2023-02-14", 220, 1690),
    (13, "2023-02-18", 275, 1965),
    (8, "2023-02-28", 180, 2145),
    (12, "2023-03-05", 450, 2595),
    (3, "2023-03-15", 150, 2745),
    (6, "2023-03-20", 400, 3145),
    (14, "2023-03-25", 325, 3470),
    (10, "2023-03-30", 320, 3790),
]
print("running total ends at 3790")

running total ends at 3790


**Check 2 - the endpoint equals the plain total.** A cross-check that the final running total is just
`SUM(amount)` over the table.

In [39]:
assert run("SELECT SUM(amount) FROM orders") == [(3790,)]
print("Q8 passed")

Q8 passed


**Approach and pitfalls.** The default window frame for an ordered `SUM` is `RANGE BETWEEN UNBOUNDED PRECEDING
AND CURRENT ROW`, which means rows that share the same `ORDER BY` value all get the *same* running total. Adding
`order_id` (a unique key) to the `ORDER BY` makes the total advance row by row - the behaviour most people
actually want. Beware that the default `RANGE` frame also ignores peers correctly but can be slow; `ROWS BETWEEN
UNBOUNDED PRECEDING AND CURRENT ROW` is explicit and usually faster.

**Follow-ups.**

1. What is the difference between the `ROWS` and `RANGE` frames when two orders share a date?
2. How would you compute a 3-order moving average with a sliding frame?

### Q9. Running total per customer

Show the cumulative completed-order amount per customer, in order date order.

**Expected result.** 12 rows (non-`completed` orders are excluded). Customer 1 runs 100, 300, 450; customer 5
runs 500 then 950. Customer 4 has a single completed order of 320.

In [40]:
show("""
SELECT customer_id, order_date, amount,
       SUM(amount) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS running_total
FROM orders
WHERE status = 'completed'
ORDER BY customer_id, order_date, order_id;
""")

customer_id | order_date | amount | running_total
------------+------------+--------+--------------
1           | 2023-01-05 | 100    | 100          
1           | 2023-02-10 | 200    | 300          
1           | 2023-03-15 | 150    | 450          
2           | 2023-01-20 | 300    | 300          
2           | 2023-03-20 | 400    | 700          
3           | 2023-01-25 | 120    | 120          
3           | 2023-02-28 | 180    | 300          
4           | 2023-03-30 | 320    | 320          
5           | 2023-01-02 | 500    | 500          
5           | 2023-03-05 | 450    | 950          
6           | 2023-02-18 | 275    | 275          
(11 rows)


[(1, '2023-01-05', 100, 100),
 (1, '2023-02-10', 200, 300),
 (1, '2023-03-15', 150, 450),
 (2, '2023-01-20', 300, 300),
 (2, '2023-03-20', 400, 700),
 (3, '2023-01-25', 120, 120),
 (3, '2023-02-28', 180, 300),
 (4, '2023-03-30', 320, 320),
 (5, '2023-01-02', 500, 500),
 (5, '2023-03-05', 450, 950),
 (6, '2023-02-18', 275, 275)]

**Check 1 - the running total restarts per customer.** There are no non-completed rows in the result, and the
cumulative value resets at every new `customer_id` because of `PARTITION BY`.

In [41]:
assert run(
    """
    SELECT customer_id, order_date, amount,
           SUM(amount) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS running_total
    FROM orders WHERE status = 'completed'
    ORDER BY customer_id, order_date, order_id
    """
) == [
    (1, "2023-01-05", 100, 100),
    (1, "2023-02-10", 200, 300),
    (1, "2023-03-15", 150, 450),
    (2, "2023-01-20", 300, 300),
    (2, "2023-03-20", 400, 700),
    (3, "2023-01-25", 120, 120),
    (3, "2023-02-28", 180, 300),
    (4, "2023-03-30", 320, 320),
    (5, "2023-01-02", 500, 500),
    (5, "2023-03-05", 450, 950),
    (6, "2023-02-18", 275, 275),
]
print("Q9 passed")

Q9 passed


**Approach and pitfalls.** `PARTITION BY customer_id` restarts the running total for each customer; the frame
resets with the partition automatically. The subtle part is what you filter *before* the window: `WHERE status =
'completed'` removes cancelled and pending orders from the sequence entirely. If you instead filtered after the
window, the excluded amounts would still be included in the cumulative sum.

**Follow-ups.**

1. How would you add a column that shows each order's share of that customer's completed revenue?
2. What is the fourth order for customer 2, and why does it not appear here?

### Q10. Month-over-month growth

Using completed orders, show monthly revenue, the previous month's revenue and the month-over-month growth
percentage.

**Expected result.** January 1020 (no previous month, so `NULL`), February 655 with -35.78%, March 1320 with
+101.53%.

In [42]:
show("""
WITH monthly AS (
    SELECT substr(order_date, 1, 7) AS month,
           SUM(amount) AS revenue
    FROM orders
    WHERE status = 'completed'
    GROUP BY month
)
SELECT month,
       revenue,
       LAG(revenue) OVER (ORDER BY month) AS prev_revenue,
       ROUND(
           100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
           / LAG(revenue) OVER (ORDER BY month),
           2
       ) AS mom_pct
FROM monthly
ORDER BY month;
""")

month   | revenue | prev_revenue | mom_pct
--------+---------+--------------+--------
2023-01 | 1020    | None         | None   
2023-02 | 655     | 1020         | -35.78 
2023-03 | 1320    | 655          | 101.53 
(3 rows)


[('2023-01', 1020, None, None),
 ('2023-02', 655, 1020, -35.78),
 ('2023-03', 1320, 655, 101.53)]

**Check 1 - monthly revenue and growth.** The aggregate runs first (one row per month), then `LAG` looks at the
previous *month*. January has no predecessor, so its previous value and growth are `NULL`.

In [43]:
assert run(
    """
    WITH monthly AS (
        SELECT substr(order_date, 1, 7) AS month, SUM(amount) AS revenue
        FROM orders WHERE status = 'completed' GROUP BY month
    )
    SELECT month, revenue, LAG(revenue) OVER (ORDER BY month),
           ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
                 / LAG(revenue) OVER (ORDER BY month), 2)
    FROM monthly ORDER BY month
    """
) == [
    ("2023-01", 1020, None, None),
    ("2023-02", 655, 1020, -35.78),
    ("2023-03", 1320, 655, 101.53),
]
print("mom growth: -35.78%, +101.53%")

mom growth: -35.78%, +101.53%


**Check 2 - the same growth as a ratio.** Expressing the change as `revenue / prev_revenue` avoids the
subtraction and makes the direction explicit: February is 0.642x January, March is 2.015x February.

In [44]:
assert run(
    """
    WITH monthly AS (
        SELECT substr(order_date, 1, 7) AS month, SUM(amount) AS revenue
        FROM orders WHERE status = 'completed' GROUP BY month
    )
    SELECT month, ROUND(revenue * 1.0 / LAG(revenue) OVER (ORDER BY month), 3)
    FROM monthly ORDER BY month
    """
) == [("2023-01", None), ("2023-02", 0.642), ("2023-03", 2.015)]
print("Q10 passed")

Q10 passed


**Approach and pitfalls.** Aggregate first (one row per month), then apply `LAG` - applying `LAG` to raw rows
gives the previous *order*, not the previous month. Guard against division by zero (a month with revenue 0) and
decide how to treat the first period: `NULL` is usually correct, `0` hides the distinction between "no data" and
"no growth". Don't forget that `100.0` (not `100`) is needed to avoid integer division in some engines; SQLite
returns a float here because of the explicit `100.0`.

**Follow-ups.**

1. How would you compute a trailing 3-month average revenue with a window frame?
2. How would you fill in months with no orders so the series has no gaps?

### Q11. Percent of total with a window SUM

Show each customer's completed revenue and its share of all completed revenue.

**Expected result.** Six customers. Customer 5 is the largest at 950 (31.72%); the shares sum to 100.00.

In [45]:
show("""
SELECT customer_id,
       SUM(amount) AS total,
       ROUND(100.0 * SUM(amount) / SUM(SUM(amount)) OVER (), 2) AS pct_of_total
FROM orders
WHERE status = 'completed'
GROUP BY customer_id
ORDER BY customer_id;
""")

customer_id | total | pct_of_total
------------+-------+-------------
1           | 450   | 15.03       
2           | 700   | 23.37       
3           | 300   | 10.02       
4           | 320   | 10.68       
5           | 950   | 31.72       
6           | 275   | 9.18        
(6 rows)


[(1, 450, 15.03),
 (2, 700, 23.37),
 (3, 300, 10.02),
 (4, 320, 10.68),
 (5, 950, 31.72),
 (6, 275, 9.18)]

**Check 1 - revenue and share per customer.** `SUM(SUM(amount)) OVER ()` is an aggregate inside a window: the
inner `SUM` collapses each customer group and the outer `SUM(...) OVER ()` totals those group values.

In [46]:
assert run(
    """
    SELECT customer_id, SUM(amount),
           ROUND(100.0 * SUM(amount) / SUM(SUM(amount)) OVER (), 2)
    FROM orders WHERE status = 'completed'
    GROUP BY customer_id ORDER BY customer_id
    """
) == [
    (1, 450, 15.03),
    (2, 700, 23.37),
    (3, 300, 10.02),
    (4, 320, 10.68),
    (5, 950, 31.72),
    (6, 275, 9.18),
]
print("six customers, customer 5 largest at 31.72%")

six customers, customer 5 largest at 31.72%


**Check 2 - the rounded shares add up to 100.** A sanity check on the denominator: this passes here, but
independent rounding is not guaranteed to sum to exactly 100.00 in general.

In [47]:
assert run(
    """
    SELECT ROUND(SUM(pct), 2) FROM (
        SELECT ROUND(100.0 * SUM(amount) / SUM(SUM(amount)) OVER (), 2) AS pct
        FROM orders WHERE status = 'completed' GROUP BY customer_id
    )
    """
) == [(100.0,)]
print("Q11 passed")

Q11 passed


**Approach and pitfalls.** `SUM(SUM(amount)) OVER ()` is an aggregate *inside* a window: the inner `SUM` collapses
each customer group, and the outer `SUM(...) OVER ()` totals those group values across the whole result. This is
why the query can compute a grand total without a self-join or a second pass. Watch out for integer division
(multiply by `100.0`), and remember rounded percentages may not sum to exactly 100 - the sanity check passes here
but is not guaranteed in general.

**Follow-ups.**

1. How would you compute each customer's share *within its region* instead of globally?
2. How would you rewrite this without window functions?

---
## Part 4 - Joins and set logic

Self-joins, anti-joins and `LEFT JOIN` + `IS NULL` are the bread and butter of dimensional queries. The question
almost always hides a `NULL` or an `EXISTS`/`IN` decision.

### Q12. Self-join for manager and employee

List every employee together with their manager's name. Employees without a manager must still appear.

**Expected result.** 14 rows. Alice, Frank, Judy and Leo show `NULL` as manager; the rest report their manager's
name.

In [48]:
show("""
SELECT e.emp_id, e.name AS employee, m.name AS manager
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.emp_id
ORDER BY e.emp_id;
""")

emp_id | employee | manager
-------+----------+--------
1      | Alice    | None   
2      | Bob      | Alice  
3      | Carol    | Alice  
4      | Dave     | Bob    
5      | Eve      | Bob    
6      | Frank    | None   
7      | Grace    | Frank  
8      | Heidi    | Frank  
9      | Ivan     | Frank  
10     | Judy     | None   
11     | Kevin    | Judy   
12     | Leo      | None   
13     | Mallory  | Leo    
14     | Nina     | Leo    
(14 rows)


[(1, 'Alice', None),
 (2, 'Bob', 'Alice'),
 (3, 'Carol', 'Alice'),
 (4, 'Dave', 'Bob'),
 (5, 'Eve', 'Bob'),
 (6, 'Frank', None),
 (7, 'Grace', 'Frank'),
 (8, 'Heidi', 'Frank'),
 (9, 'Ivan', 'Frank'),
 (10, 'Judy', None),
 (11, 'Kevin', 'Judy'),
 (12, 'Leo', None),
 (13, 'Mallory', 'Leo'),
 (14, 'Nina', 'Leo')]

**Check 1 - the full roster survives.** A self-join needs two aliases for the same table, and `LEFT JOIN` keeps
the four top-of-hierarchy rows whose `manager_id` is `NULL`. An inner join would silently drop them.

In [49]:
assert run(
    """
    SELECT e.emp_id, e.name, m.name
    FROM employees e LEFT JOIN employees m ON e.manager_id = m.emp_id
    ORDER BY e.emp_id
    """
) == [
    (1, "Alice", None), (2, "Bob", "Alice"), (3, "Carol", "Alice"),
    (4, "Dave", "Bob"), (5, "Eve", "Bob"),
    (6, "Frank", None), (7, "Grace", "Frank"), (8, "Heidi", "Frank"), (9, "Ivan", "Frank"),
    (10, "Judy", None), (11, "Kevin", "Judy"),
    (12, "Leo", None), (13, "Mallory", "Leo"), (14, "Nina", "Leo"),
]
print("14 employees, 4 with no manager")

14 employees, 4 with no manager


**Check 2 - the other direction: employees who manage nobody.** Reversing the self-join and filtering
`COUNT(e.emp_id) = 0` lists the individual contributors; `COUNT(e.emp_id)` is used instead of `COUNT(*)` because
the `LEFT JOIN` would otherwise count the all-`NULL` placeholder row.

In [50]:
assert run(
    """
    SELECT m.emp_id, m.name, COUNT(e.emp_id) AS reports
    FROM employees m LEFT JOIN employees e ON e.manager_id = m.emp_id
    GROUP BY m.emp_id, m.name HAVING COUNT(e.emp_id) = 0 ORDER BY m.emp_id
    """
) == [
    (3, "Carol", 0), (4, "Dave", 0), (5, "Eve", 0),
    (7, "Grace", 0), (8, "Heidi", 0), (9, "Ivan", 0), (11, "Kevin", 0),
    (13, "Mallory", 0), (14, "Nina", 0),
]
print("Q12 passed")

Q12 passed


**Approach and pitfalls.** A self-join needs two aliases so the same table appears twice in the `FROM` clause. Use
`LEFT JOIN` when top-of-hierarchy rows must survive; an inner join silently drops the CEO/root, which is a classic
interview trap. To walk more than one level, use a recursive CTE:

```sql
WITH RECURSIVE chain(emp_id, name, level) AS (
    SELECT emp_id, name, 1 FROM employees WHERE manager_id IS NULL
    UNION ALL
    SELECT e.emp_id, e.name, c.level + 1
    FROM employees e JOIN chain c ON e.manager_id = c.emp_id
)
SELECT * FROM chain ORDER BY level, emp_id;
```

**Follow-ups.**

1. How would you find every employee who ultimately reports to Alice?
2. How would you find employees who earn more than their own manager?

### Q13. Anti-join: customers with no orders

Find customers that have never placed an order.

**Expected result.** Two rows: `(7, 'Gamma SA')` and `(8, 'Hotel BV')`.

In [51]:
show("""
SELECT c.customer_id, c.name
FROM customers c
WHERE NOT EXISTS (
    SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id
)
ORDER BY c.customer_id;
""")

customer_id | name    
------------+---------
7           | Gamma SA
8           | Hotel BV
(2 rows)


[(7, 'Gamma SA'), (8, 'Hotel BV')]

**Check 1 - `NOT EXISTS`.** The anti-join short-circuits on the first match and is `NULL`-safe, which is why it
is the default answer.

In [52]:
assert run(
    """
    SELECT c.customer_id, c.name FROM customers c
    WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
    ORDER BY c.customer_id
    """
) == [(7, "Gamma SA"), (8, "Hotel BV")]
print("NOT EXISTS finds Gamma SA and Hotel BV")

NOT EXISTS finds Gamma SA and Hotel BV


**Check 2 - the `LEFT JOIN ... IS NULL` equivalent.** It returns the same rows because `orders.customer_id` is
non-nullable here; that assumption is exactly what to check before rewriting the query.

In [53]:
assert run(
    """
    SELECT c.customer_id, c.name FROM customers c
    LEFT JOIN orders o ON o.customer_id = c.customer_id
    WHERE o.order_id IS NULL
    ORDER BY c.customer_id
    """
) == [(7, "Gamma SA"), (8, "Hotel BV")]
print("Q13 passed")

Q13 passed


**Approach and pitfalls.** `NOT EXISTS` is the safest anti-join: it is `NULL`-proof, short-circuits on the first
match, and optimisers handle it well. `LEFT JOIN ... WHERE right_key IS NULL` is equivalent but only when the
right-hand key is non-nullable - if `orders.customer_id` could be `NULL`, matching "no order" and "an order with a
`NULL` customer" become indistinguishable. `NOT IN (SELECT ...)` is the dangerous one: if the subquery returns a
single `NULL`, the whole predicate becomes unknown and you get zero rows.

**Follow-ups.**

1. Why does `NOT IN` return no rows when the subquery contains a `NULL`?
2. How would you find customers with orders but no *completed* orders?

### Q14. LEFT JOIN and NULL handling

For every customer, count their orders and total order amount, including customers with no orders.

**Expected result.** 8 rows. Gamma SA and Hotel BV show `order_count = 0` and `total_amount = 0` thanks to
`COUNT(o.order_id)` and `COALESCE`. The second check shows what `COUNT(*)` would have produced instead: 1 for
those customers, because the `LEFT JOIN` still emits one all-`NULL` row.

In [54]:
show("""
SELECT c.customer_id, c.name,
       COUNT(o.order_id) AS order_count,
       COALESCE(SUM(o.amount), 0) AS total_amount
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.name
ORDER BY c.customer_id;
""")

customer_id | name         | order_count | total_amount
------------+--------------+-------------+-------------
1           | Acme Corp    | 3           | 450         
2           | Beta LLC     | 3           | 950         
3           | Cyan Inc     | 2           | 300         
4           | Delta Co     | 2           | 540         
5           | Echo Ltd     | 2           | 950         
6           | Foxtrot GmbH | 2           | 600         
7           | Gamma SA     | 0           | 0           
8           | Hotel BV     | 0           | 0           
(8 rows)


[(1, 'Acme Corp', 3, 450),
 (2, 'Beta LLC', 3, 950),
 (3, 'Cyan Inc', 2, 300),
 (4, 'Delta Co', 2, 540),
 (5, 'Echo Ltd', 2, 950),
 (6, 'Foxtrot GmbH', 2, 600),
 (7, 'Gamma SA', 0, 0),
 (8, 'Hotel BV', 0, 0)]

**Check 1 - all eight customers.** `COUNT(o.order_id)` skips the `NULL` produced by the outer join, and
`COALESCE(SUM(...), 0)` turns the empty sum into a real zero.

In [55]:
assert run(
    """
    SELECT c.customer_id, c.name, COUNT(o.order_id), COALESCE(SUM(o.amount), 0)
    FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.name ORDER BY c.customer_id
    """
) == [
    (1, "Acme Corp", 3, 450),
    (2, "Beta LLC", 3, 950),
    (3, "Cyan Inc", 2, 300),
    (4, "Delta Co", 2, 540),
    (5, "Echo Ltd", 2, 950),
    (6, "Foxtrot GmbH", 2, 600),
    (7, "Gamma SA", 0, 0),
    (8, "Hotel BV", 0, 0),
]
print("zero-order customers report 0, not NULL")

zero-order customers report 0, not NULL


**Check 2 - the `COUNT(*)` pitfall.** Same join, but counting rows instead of non-null keys gives 1 for the two
customers with no orders.

In [56]:
assert run(
    """
    SELECT c.customer_id, COUNT(*)
    FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id HAVING COUNT(*) = 1 ORDER BY c.customer_id
    """
) == [(7, 1), (8, 1)]
print("Q14 passed")

Q14 passed


**Approach and pitfalls.** `COUNT(column)` skips `NULL`s, `COUNT(*)` does not - that single distinction is the
difference between 0 and 1 for customers with no orders. Wrap `SUM` in `COALESCE` because `SUM` over zero rows is
`NULL`, and `NULL` breaks downstream arithmetic. Filtering on a left-joined column in the `WHERE` clause
(`WHERE o.status = 'completed'`) silently turns the outer join back into an inner join; push such predicates into
the `ON` clause when you want to keep the unmatched rows.

**Follow-ups.**

1. How would you keep all customers but count only completed orders?
2. Why does `WHERE o.status = 'completed'` remove customers with no orders?

### Q15. EXISTS vs IN

Find customers that have at least one order, first with `EXISTS` and then with `IN`.

**Expected result.** Both queries return the same six customers, 1 through 6.

In [57]:
show("""
SELECT c.customer_id, c.name
FROM customers c
WHERE EXISTS (
    SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id
)
ORDER BY c.customer_id;
""")

customer_id | name        
------------+-------------
1           | Acme Corp   
2           | Beta LLC    
3           | Cyan Inc    
4           | Delta Co    
5           | Echo Ltd    
6           | Foxtrot GmbH
(6 rows)


[(1, 'Acme Corp'),
 (2, 'Beta LLC'),
 (3, 'Cyan Inc'),
 (4, 'Delta Co'),
 (5, 'Echo Ltd'),
 (6, 'Foxtrot GmbH')]

**Check 1 - the `EXISTS` semi-join.** The engine can stop at the first matching order per customer; `expected` is
then reused by the next check so both forms are compared against one literal.

In [58]:
expected = [
    (1, "Acme Corp"), (2, "Beta LLC"), (3, "Cyan Inc"),
    (4, "Delta Co"), (5, "Echo Ltd"), (6, "Foxtrot GmbH"),
]

assert run(
    """
    SELECT c.customer_id, c.name FROM customers c
    WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
    ORDER BY c.customer_id
    """
) == expected
print("EXISTS returns 6 customers")

EXISTS returns 6 customers


**Check 2 - the `IN` form returns the identical set.** `IN` materialises the subquery list; on a unique,
non-nullable key the optimiser may even produce the same plan.

In [59]:
assert run(
    """
    SELECT customer_id, name FROM customers
    WHERE customer_id IN (SELECT customer_id FROM orders)
    ORDER BY customer_id
    """
) == expected
print("IN returns the same 6 customers")

IN returns the same 6 customers


**Setup for the `NULL` trap.** A two-row table where one `customer_id` is `NULL` is enough to make `NOT IN`
misbehave.

In [60]:
run("DROP TABLE IF EXISTS tiny_orders")
run("CREATE TABLE tiny_orders (customer_id INTEGER)")
run("INSERT INTO tiny_orders VALUES (1), (NULL)")
print("tiny_orders holds a 1 and a NULL")

tiny_orders holds a 1 and a NULL


**Check 3 - `NOT IN` with a `NULL` returns no rows.** `x <> NULL` is `UNKNOWN`, so *no* customer can satisfy the
predicate, not even the ones who did not order.

In [61]:
assert run(
    "SELECT customer_id FROM customers WHERE customer_id NOT IN (SELECT customer_id FROM tiny_orders)"
) == []
print("NOT IN silently matched nothing")

NOT IN silently matched nothing


**Check 4 - `NOT EXISTS` handles the same data correctly.** It compares row by row, so the `NULL` row simply
matches nobody and the true anti-join (customers 2-8) comes back.

In [62]:
assert run(
    "SELECT customer_id FROM customers c WHERE NOT EXISTS "
    "(SELECT 1 FROM tiny_orders t WHERE t.customer_id = c.customer_id) ORDER BY customer_id"
) == [(2,), (3,), (4,), (5,), (6,), (7,), (8,)]
print("Q15 passed")

Q15 passed


**Approach and pitfalls.** `EXISTS` is a semi-join: the engine can stop at the first matching row, and it is
`NULL`-safe. `IN` materialises (or hashes) the subquery list and is fine for small, non-nullable lists, but
`NOT IN` with a `NULL` in the list returns *no rows at all*, because `x <> NULL` evaluates to `UNKNOWN`. Modern
optimisers often rewrite `IN` and `EXISTS` into the same plan, so readability and `NULL` safety, not speed, are
the deciding factors.

**Follow-ups.**

1. How do the plans for `EXISTS` and `IN` differ in `EXPLAIN QUERY PLAN`?
2. When does `IN` beat `EXISTS` on performance?

---
## Part 5 - Aggregation, pivots and medians

The next questions combine grouping with window functions: comparing rows to their group, reshaping rows into
columns, and computing order statistics that SQL has no built-in `MEDIAN` for.

### Q16. Employees above their department average

Return employees whose salary is strictly above the average salary of their own department.

**Expected result.** Six rows: Alice, Bob and Carol in Engineering (average 115000), Frank in Sales (average
100500), Judy in Marketing (average 96500) and Leo in Support (average 76666.67). Both the correlated-subquery
and window versions agree.

In [63]:
show("""
SELECT e.name, e.dept_id, e.salary
FROM employees e
WHERE e.salary > (
    SELECT AVG(salary) FROM employees x WHERE x.dept_id = e.dept_id
)
ORDER BY e.dept_id, e.salary DESC, e.name;
""")

name  | dept_id | salary
------+---------+-------
Alice | 1       | 150000
Bob   | 1       | 120000
Carol | 1       | 120000
Frank | 2       | 130000
Judy  | 3       | 105000
Leo   | 4       | 90000 
(6 rows)


[('Alice', 1, 150000),
 ('Bob', 1, 120000),
 ('Carol', 1, 120000),
 ('Frank', 2, 130000),
 ('Judy', 3, 105000),
 ('Leo', 4, 90000)]

**Check 1 - the correlated subquery answer.** The subquery is conceptually re-evaluated for every employee, each
time restricted to that employee's department. The comparison is a *strict* `>`.

In [64]:
expected = [
    ("Alice", 1, 150000),
    ("Bob", 1, 120000),
    ("Carol", 1, 120000),
    ("Frank", 2, 130000),
    ("Judy", 3, 105000),
    ("Leo", 4, 90000),
]

assert run(
    """
    SELECT e.name, e.dept_id, e.salary FROM employees e
    WHERE e.salary > (SELECT AVG(salary) FROM employees x WHERE x.dept_id = e.dept_id)
    ORDER BY e.dept_id, e.salary DESC, e.name
    """
) == expected
print("correlated subquery finds 6 employees")

correlated subquery finds 6 employees


**Check 2 - the window version agrees.** `AVG(salary) OVER (PARTITION BY dept_id)` computes each department
average once, and an outer query filters on it because windows cannot appear in the same `WHERE`.

In [65]:
assert run(
    """
    SELECT name, dept_id, salary FROM (
        SELECT name, dept_id, salary,
               AVG(salary) OVER (PARTITION BY dept_id) AS dept_avg
        FROM employees
    ) WHERE salary > dept_avg
    ORDER BY dept_id, salary DESC, name
    """
) == expected
print("window version matches")

window version matches


**Check 3 - the averages, for reference.** These are the thresholds every comparison above used.

In [66]:
assert run(
    """
    SELECT dept_id, ROUND(AVG(salary), 2) FROM employees GROUP BY dept_id ORDER BY dept_id
    """
) == [(1, 115000.0), (2, 100500.0), (3, 96500.0), (4, 76666.67)]
print("Q16 passed")

Q16 passed


**Approach and pitfalls.** The correlated subquery is the textbook answer and reads clearly, but it is
conceptually one aggregate per row. The window version computes each department average once per partition and is
usually the faster plan on large tables. Both compare against a *strict* `>`; using `>=` would pull in everyone in
a department where every salary equals the average. Note the window function cannot appear in the same `WHERE`,
which is why the window version needs the extra subquery level.

**Follow-ups.**

1. How would you return each employee's salary as a percentage of their department total?
2. Which version would you choose for a 1-billion-row fact table, and why?

### Q17. Pivot with CASE

Turn the `status` values into columns: one row per customer with the number of completed, pending and cancelled
orders.

**Expected result.** 8 rows. Beta LLC has 2 completed and 1 cancelled; Delta Co has 1 completed and 1 pending; the
two customers with no orders show zeros.

In [67]:
show("""
SELECT c.customer_id, c.name,
       COUNT(CASE WHEN o.status = 'completed' THEN 1 END) AS completed,
       COUNT(CASE WHEN o.status = 'pending'   THEN 1 END) AS pending,
       COUNT(CASE WHEN o.status = 'cancelled' THEN 1 END) AS cancelled
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.name
ORDER BY c.customer_id;
""")

customer_id | name         | completed | pending | cancelled
------------+--------------+-----------+---------+----------
1           | Acme Corp    | 3         | 0       | 0        
2           | Beta LLC     | 2         | 0       | 1        
3           | Cyan Inc     | 2         | 0       | 0        
4           | Delta Co     | 1         | 1       | 0        
5           | Echo Ltd     | 2         | 0       | 0        
6           | Foxtrot GmbH | 1         | 0       | 1        
7           | Gamma SA     | 0         | 0       | 0        
8           | Hotel BV     | 0         | 0       | 0        
(8 rows)


[(1, 'Acme Corp', 3, 0, 0),
 (2, 'Beta LLC', 2, 0, 1),
 (3, 'Cyan Inc', 2, 0, 0),
 (4, 'Delta Co', 1, 1, 0),
 (5, 'Echo Ltd', 2, 0, 0),
 (6, 'Foxtrot GmbH', 1, 0, 1),
 (7, 'Gamma SA', 0, 0, 0),
 (8, 'Hotel BV', 0, 0, 0)]

**Check 1 - pivoting counts.** `COUNT(CASE WHEN ... THEN 1 END)` has no `ELSE`, so unmatched statuses produce
`NULL`, which `COUNT` ignores - that is what yields a clean 0 for customers with no orders.

In [68]:
assert run(
    """
    SELECT c.customer_id, c.name,
           COUNT(CASE WHEN o.status = 'completed' THEN 1 END),
           COUNT(CASE WHEN o.status = 'pending'   THEN 1 END),
           COUNT(CASE WHEN o.status = 'cancelled' THEN 1 END)
    FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.name ORDER BY c.customer_id
    """
) == [
    (1, "Acme Corp", 3, 0, 0),
    (2, "Beta LLC", 2, 0, 1),
    (3, "Cyan Inc", 2, 0, 0),
    (4, "Delta Co", 1, 1, 0),
    (5, "Echo Ltd", 2, 0, 0),
    (6, "Foxtrot GmbH", 1, 0, 1),
    (7, "Gamma SA", 0, 0, 0),
    (8, "Hotel BV", 0, 0, 0),
]
print("count pivot: 8 customers, 3 status columns")

count pivot: 8 customers, 3 status columns


**Check 2 - pivoting amounts instead of counts.** With `SUM` the `ELSE 0` is mandatory: without it the whole sum
for a customer with no matching rows would be `NULL`.

In [69]:
assert run(
    """
    SELECT c.customer_id,
           SUM(CASE WHEN o.status = 'completed' THEN o.amount ELSE 0 END),
           SUM(CASE WHEN o.status = 'pending'   THEN o.amount ELSE 0 END),
           SUM(CASE WHEN o.status = 'cancelled' THEN o.amount ELSE 0 END)
    FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id ORDER BY c.customer_id
    """
) == [
    (1, 450, 0, 0),
    (2, 700, 0, 250),
    (3, 300, 0, 0),
    (4, 320, 220, 0),
    (5, 950, 0, 0),
    (6, 275, 0, 325),
    (7, 0, 0, 0),
    (8, 0, 0, 0),
]
print("Q17 passed")

Q17 passed


**Approach and pitfalls.** For counts, `COUNT(CASE WHEN ... THEN 1 END)` is the safe form: the missing `ELSE`
yields `NULL`, which `COUNT` ignores, so unmatched statuses contribute 0. With `SUM` you need `ELSE 0`, otherwise
the whole sum is `NULL` for customers with no matching rows. SQLite also supports the `FILTER (WHERE ...)` clause,
so `COUNT(*) FILTER (WHERE o.status = 'completed')` is a cleaner spelling; the `CASE` form is more portable.

**Follow-ups.**

1. Rewrite the count pivot using `FILTER (WHERE ...)`.
2. How would you produce the same pivot dynamically when the status values are unknown?

**Pivoting turns rows into columns.** SQL has no portable `PIVOT`, so the standard trick is a `CASE` inside an aggregate, one expression per output column. The aggregate collapses the rows of each group, and the `CASE` routes the right measure into the right column.

```sql
SELECT region,
       SUM(CASE WHEN status = 'completed' THEN amount ELSE 0 END) AS completed,
       SUM(CASE WHEN status = 'cancelled' THEN amount ELSE 0 END) AS cancelled
FROM orders
GROUP BY region;
```

The columns are fixed by the query text, so a new status needs a query change - that is the main limitation. Some engines add `PIVOT` (SQL Server, Snowflake, BigQuery) or `crosstab` (Postgres), but they hide the same grouping. The reverse operation, unpivoting, is a `UNION ALL` of one `SELECT` per column.

### Q18. Median salary

SQL has no portable `MEDIAN`. Compute the median of `employees.salary` with two different techniques.

**Expected result.** Both return `92500.0`. With 14 salaries the median is the average of the 7th and 8th values,
90000 and 95000.

In [70]:
show("""
WITH ranked AS (
    SELECT salary,
           ROW_NUMBER() OVER (ORDER BY salary) AS rn,
           COUNT(*) OVER () AS n
    FROM employees
)
SELECT AVG(salary) AS median_salary
FROM ranked
WHERE rn IN ((n + 1) / 2, (n + 2) / 2);
""")

median_salary
-------------
92500.0      
(1 row)


[(92500.0,)]

**Check 1 - the portable window version.** For odd `n` the two computed ranks land on the same row, so the
average of one value is that value; for even `n` they bracket the two middle rows. The integer division is
intentional.

In [71]:
assert run(
    """
    WITH ranked AS (
        SELECT salary, ROW_NUMBER() OVER (ORDER BY salary) AS rn, COUNT(*) OVER () AS n
        FROM employees
    )
    SELECT AVG(salary) FROM ranked WHERE rn IN ((n + 1) / 2, (n + 2) / 2)
    """
) == [(92500.0,)]
print("median via ROW_NUMBER:", 92500.0)

median via ROW_NUMBER: 92500.0


**Check 2 - the `LIMIT`/`OFFSET` variant.** `LIMIT 2 - n % 2` keeps one row for odd `n` and two for even `n`,
starting at the middle offset; the surrounding `AVG` then does the rest.

In [72]:
assert run(
    """
    SELECT AVG(salary) FROM (
        SELECT salary FROM employees ORDER BY salary
        LIMIT 2 - (SELECT COUNT(*) FROM employees) % 2
        OFFSET (SELECT (COUNT(*) - 1) / 2 FROM employees)
    )
    """
) == [(92500.0,)]
print("median via LIMIT/OFFSET:", 92500.0)

median via LIMIT/OFFSET: 92500.0


**Check 3 - the two middle values.** Confirms that the median is the average of 90000 and 95000 rather than one
of them.

In [73]:
assert run(
    """
    SELECT salary FROM (
        SELECT salary, ROW_NUMBER() OVER (ORDER BY salary) AS rn, COUNT(*) OVER () AS n
        FROM employees
    ) WHERE rn IN ((n + 1) / 2, (n + 2) / 2) ORDER BY salary
    """
) == [(90000,), (95000,)]
print("Q18 passed")

Q18 passed


**Approach and pitfalls.** The `ROW_NUMBER` + `COUNT(*) OVER ()` version is the portable one: for odd `n` both
`(n+1)/2` and `(n+2)/2` land on the same row so the `AVG` of one value is that value; for even `n` they bracket the
two middle rows. Integer division is intentional - in SQLite, `/` on integers truncates. The `LIMIT 2 - n % 2`
trick works but is harder to read. For percentiles in general, prefer the engine's native function
(`PERCENTILE_CONT` in BigQuery/Snowflake/Postgres, `PERCENTILE` in SQLite's percentile extension).

**Follow-ups.**

1. How would you compute the median per department?
2. How would you compute the 90th percentile salary?

---
## Part 6 - Time, sequences and cohorts

These are the hardest questions to improvise: gaps and islands, sessionization, and cohort retention all reduce to
"compare each row with the previous row in order" - which is `LAG` plus a cumulative `SUM`.

### Q19. Consecutive-day streak (gaps and islands)

For each user in `events`, find their longest run of consecutive calendar days with at least one event.

**Expected result.** User 1 has a 3-day streak (Jan 1-3) plus a 2-day streak (Jan 5-6); user 2 has a 2-day streak;
users 3 and 4 each have a single day.

In [74]:
show("""
WITH daily AS (
    SELECT DISTINCT user_id, date(event_ts) AS d
    FROM events
),
grouped AS (
    SELECT user_id, d,
           date(d, '-' || ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY d) || ' day') AS island
    FROM daily
)
SELECT user_id, MIN(d) AS start_day, MAX(d) AS end_day, COUNT(*) AS streak_len
FROM grouped
GROUP BY user_id, island
ORDER BY user_id, start_day;
""")

user_id | start_day  | end_day    | streak_len
--------+------------+------------+-----------
1       | 2023-01-01 | 2023-01-03 | 3         
1       | 2023-01-05 | 2023-01-06 | 2         
2       | 2023-01-01 | 2023-01-02 | 2         
2       | 2023-01-04 | 2023-01-04 | 1         
3       | 2023-02-01 | 2023-02-01 | 1         
4       | 2023-02-01 | 2023-02-01 | 1         
(6 rows)


[(1, '2023-01-01', '2023-01-03', 3),
 (1, '2023-01-05', '2023-01-06', 2),
 (2, '2023-01-01', '2023-01-02', 2),
 (2, '2023-01-04', '2023-01-04', 1),
 (3, '2023-02-01', '2023-02-01', 1),
 (4, '2023-02-01', '2023-02-01', 1)]

**Check 1 - every island.** Deduplicating to one row per day *first* keeps `date - row_number` constant inside a
run, so grouping by that constant recovers the islands.

In [75]:
assert run(
    """
    WITH daily AS (
        SELECT DISTINCT user_id, date(event_ts) AS d FROM events
    ), grouped AS (
        SELECT user_id, d,
               date(d, '-' || ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY d) || ' day') AS island
        FROM daily
    )
    SELECT user_id, MIN(d) AS start_day, MAX(d), COUNT(*)
    FROM grouped GROUP BY user_id, island ORDER BY user_id, start_day
    """
) == [
    (1, "2023-01-01", "2023-01-03", 3),
    (1, "2023-01-05", "2023-01-06", 2),
    (2, "2023-01-01", "2023-01-02", 2),
    (2, "2023-01-04", "2023-01-04", 1),
    (3, "2023-02-01", "2023-02-01", 1),
    (4, "2023-02-01", "2023-02-01", 1),
]
print("6 islands across 4 users")

6 islands across 4 users


**Check 2 - the longest streak per user.** Wrapping the island query and taking `MAX(streak_len)` gives the
one-number answer, 3 for user 1 and 2 for user 2.

In [76]:
assert run(
    """
    WITH daily AS (
        SELECT DISTINCT user_id, date(event_ts) AS d FROM events
    ), grouped AS (
        SELECT user_id, d,
               date(d, '-' || ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY d) || ' day') AS island
        FROM daily
    )
    SELECT user_id, MAX(streak_len) FROM (
        SELECT user_id, island, COUNT(*) AS streak_len FROM grouped GROUP BY user_id, island
    ) GROUP BY user_id ORDER BY user_id
    """
) == [(1, 3), (2, 2), (3, 1), (4, 1)]
print("Q19 passed")

Q19 passed


**Approach and pitfalls.** The gaps-and-islands trick: within a consecutive run, `date - row_number` is constant,
because both advance by one each day. Group by that constant "island key" and you have your runs. Deduplicate to
one row per day *first* (`SELECT DISTINCT`), or multiple events on the same day inflate the count and shift the
row numbers. `date(d, '-' || n || ' day')` is SQLite syntax for date arithmetic; Postgres would use
`d - n * INTERVAL '1 day'` or `d - n`, and BigQuery `DATE_SUB(d, INTERVAL n DAY)`. Streaks across month or year
boundaries work automatically because real dates are used, not string parts.

**Follow-ups.**

1. How would you adapt this to count consecutive *orders* (by order id) rather than calendar days?
2. How would you list every user whose longest streak is at least 3?

### Q20. Sessionization with LAG

Assign a session id to each event: a new session starts when the gap from the same user's previous event exceeds
30 minutes.

**Expected result.** User 3 gets two sessions - events at 10:00, 10:05 and 10:25 are one session, then 11:30
starts a second (the 65-minute gap) and 11:45 joins it. User 4's two events are 20 minutes apart, so one session.
Across all users the session counts are 5, 3, 2 and 1.

In [77]:
show("""
WITH ordered AS (
    SELECT event_id, user_id, event_ts, event_type,
           LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS prev_ts
    FROM events
),
flagged AS (
    SELECT event_id, user_id, event_ts, event_type,
           CASE WHEN prev_ts IS NULL
                  OR (julianday(event_ts) - julianday(prev_ts)) * 24 * 60 > 30
                THEN 1 ELSE 0 END AS is_new_session
    FROM ordered
)
SELECT user_id, event_ts, event_type,
       SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS session_id
FROM flagged
WHERE user_id IN (3, 4)
ORDER BY user_id, event_ts;
""")

user_id | event_ts            | event_type | session_id
--------+---------------------+------------+-----------
3       | 2023-02-01 10:00:00 | view       | 1         
3       | 2023-02-01 10:05:00 | click      | 1         
3       | 2023-02-01 10:25:00 | view       | 1         
3       | 2023-02-01 11:30:00 | purchase   | 2         
3       | 2023-02-01 11:45:00 | view       | 2         
4       | 2023-02-01 10:00:00 | login      | 1         
4       | 2023-02-01 10:20:00 | view       | 1         
(7 rows)


[(3, '2023-02-01 10:00:00', 'view', 1),
 (3, '2023-02-01 10:05:00', 'click', 1),
 (3, '2023-02-01 10:25:00', 'view', 1),
 (3, '2023-02-01 11:30:00', 'purchase', 2),
 (3, '2023-02-01 11:45:00', 'view', 2),
 (4, '2023-02-01 10:00:00', 'login', 1),
 (4, '2023-02-01 10:20:00', 'view', 1)]

**Check 1 - the session ids for users 3 and 4.** `LAG` finds the previous event, the `CASE` flags a boundary
(including the first event, whose `prev_ts` is `NULL`), and the cumulative `SUM` numbers the sessions per user.

In [78]:
assert run(
    """
    WITH ordered AS (
        SELECT event_id, user_id, event_ts, event_type,
               LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS prev_ts
        FROM events
    ), flagged AS (
        SELECT event_id, user_id, event_ts, event_type,
               CASE WHEN prev_ts IS NULL
                      OR (julianday(event_ts) - julianday(prev_ts)) * 24 * 60 > 30
                    THEN 1 ELSE 0 END AS is_new_session
        FROM ordered
    )
    SELECT user_id, event_ts, event_type,
           SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY event_ts, event_id)
    FROM flagged WHERE user_id IN (3, 4) ORDER BY user_id, event_ts
    """
) == [
    (3, "2023-02-01 10:00:00", "view", 1),
    (3, "2023-02-01 10:05:00", "click", 1),
    (3, "2023-02-01 10:25:00", "view", 1),
    (3, "2023-02-01 11:30:00", "purchase", 2),
    (3, "2023-02-01 11:45:00", "view", 2),
    (4, "2023-02-01 10:00:00", "login", 1),
    (4, "2023-02-01 10:20:00", "view", 1),
]
print("user 3 has 2 sessions, user 4 has 1")

user 3 has 2 sessions, user 4 has 1


**Check 2 - session count per user.** The largest session id per user *is* the number of sessions; grouping by
`user_id` and taking `MAX(session_id)` returns 5, 3, 2 and 1.

In [79]:
assert run(
    """
    WITH ordered AS (
        SELECT event_id, user_id, event_ts,
               LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS prev_ts
        FROM events
    ), flagged AS (
        SELECT event_id, user_id, event_ts,
               CASE WHEN prev_ts IS NULL
                      OR (julianday(event_ts) - julianday(prev_ts)) * 24 * 60 > 30
                    THEN 1 ELSE 0 END AS is_new_session
        FROM ordered
    ), numbered AS (
        SELECT user_id,
               SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS session_id
        FROM flagged
    )
    SELECT user_id, MAX(session_id) FROM numbered GROUP BY user_id ORDER BY user_id
    """
) == [(1, 5), (2, 3), (3, 2), (4, 1)]
print("Q20 passed")

Q20 passed


**Approach and pitfalls.** Sessionization is "flag the boundaries, then cumulative-sum the flags". `LAG` needs a
deterministic `ORDER BY` (tie-break with `event_id`) or events in the same second can be reordered. `julianday`
returns days as a float, so `* 24 * 60` converts the difference to minutes; other engines subtract timestamps or
use `TIMESTAMP_DIFF`. The first event per user has `prev_ts IS NULL` and must be flagged as a new session
explicitly. Session ids are per user and start at 1 because the `SUM` is partitioned.

**Follow-ups.**

1. How would you also start a new session when the event type changes (for example a `purchase` closes a session)?
2. How would you materialise a `sessions` table with start time, end time and event count?

### Q21. Cohort and retention by month

Group customers into cohorts by the month of their first order, then count how many customers from each cohort
were active in each subsequent month (including the acquisition month).

**Expected result.** The January 2023 cohort has 4 customers and retains 4, 3, 3 across January, February and
March. The February cohort has 2 customers, retained 2 and 2.

In [80]:
show("""
WITH first_order AS (
    SELECT customer_id, MIN(substr(order_date, 1, 7)) AS cohort_month
    FROM orders
    GROUP BY customer_id
),
activity AS (
    SELECT DISTINCT customer_id, substr(order_date, 1, 7) AS activity_month
    FROM orders
)
SELECT f.cohort_month,
       a.activity_month,
       COUNT(DISTINCT a.customer_id) AS active_customers
FROM first_order f
JOIN activity a ON a.customer_id = f.customer_id
GROUP BY f.cohort_month, a.activity_month
ORDER BY f.cohort_month, a.activity_month;
""")

cohort_month | activity_month | active_customers
-------------+----------------+-----------------
2023-01      | 2023-01        | 4               
2023-01      | 2023-02        | 3               
2023-01      | 2023-03        | 3               
2023-02      | 2023-02        | 2               
2023-02      | 2023-03        | 2               
(5 rows)


[('2023-01', '2023-01', 4),
 ('2023-01', '2023-02', 3),
 ('2023-01', '2023-03', 3),
 ('2023-02', '2023-02', 2),
 ('2023-02', '2023-03', 2)]

**Check 1 - the retention triangle.** Three steps: find each customer's first month, list the distinct
(customer, month) activity pairs, then join and count. `COUNT(DISTINCT customer_id)` stops a customer with
several orders in one month from inflating the count.

In [81]:
assert run(
    """
    WITH first_order AS (
        SELECT customer_id, MIN(substr(order_date, 1, 7)) AS cohort_month FROM orders GROUP BY customer_id
    ), activity AS (
        SELECT DISTINCT customer_id, substr(order_date, 1, 7) AS activity_month FROM orders
    )
    SELECT f.cohort_month, a.activity_month, COUNT(DISTINCT a.customer_id)
    FROM first_order f JOIN activity a ON a.customer_id = f.customer_id
    GROUP BY f.cohort_month, a.activity_month ORDER BY f.cohort_month, a.activity_month
    """
) == [
    ("2023-01", "2023-01", 4),
    ("2023-01", "2023-02", 3),
    ("2023-01", "2023-03", 3),
    ("2023-02", "2023-02", 2),
    ("2023-02", "2023-03", 2),
]
print("January cohort retains 4, 3, 3")

January cohort retains 4, 3, 3


**Check 2 - cohort sizes.** The denominator for any retention rate: 4 customers acquired in January and 2 in
February.

In [82]:
assert run(
    """
    WITH first_order AS (
        SELECT customer_id, MIN(substr(order_date, 1, 7)) AS cohort_month FROM orders GROUP BY customer_id
    )
    SELECT cohort_month, COUNT(*) FROM first_order GROUP BY cohort_month ORDER BY cohort_month
    """
) == [("2023-01", 4), ("2023-02", 2)]
print("Q21 passed")

Q21 passed


**Approach and pitfalls.** Three steps: (1) one row per customer with the first period (the cohort), (2) the
distinct (customer, period) pairs that were active, (3) join and count. `COUNT(DISTINCT customer_id)` is the
safety net - without it a customer with three orders in one month inflates retention above 100%. The month offset
is usually more useful than the raw month label:

```sql
(CAST(substr(a.activity_month, 1, 4) AS INTEGER) * 12 + CAST(substr(a.activity_month, 6, 2) AS INTEGER))
- (CAST(substr(f.cohort_month,   1, 4) AS INTEGER) * 12 + CAST(substr(f.cohort_month,   6, 2) AS INTEGER))
```

Also decide whether "active" means any order or only completed orders; here it is any order.

**Follow-ups.**

1. Turn the counts into a retention rate using the cohort sizes.
2. How would you pivot the result into a classic cohort matrix (one row per cohort, one column per month offset)?

### Q22. Cumulative distinct count

For each order date, show how many distinct customers had placed an order on or before that date.

**Expected result.** 14 rows. The count reaches 4 by 2023-01-25, 5 on 2023-02-14 and 6 on 2023-02-18, then stays
at 6 - all six ordering customers have been seen.

In [83]:
show("""
SELECT o.order_date,
       (
           SELECT COUNT(DISTINCT o2.customer_id)
           FROM orders o2
           WHERE o2.order_date <= o.order_date
       ) AS cum_distinct_customers
FROM (SELECT DISTINCT order_date FROM orders) o
ORDER BY o.order_date;
""")

order_date | cum_distinct_customers
-----------+-----------------------
2023-01-02 | 1                     
2023-01-05 | 2                     
2023-01-20 | 3                     
2023-01-25 | 4                     
2023-02-05 | 4                     
2023-02-10 | 4                     
2023-02-14 | 5                     
2023-02-18 | 6                     
2023-02-28 | 6                     
2023-03-05 | 6                     
2023-03-15 | 6                     
2023-03-20 | 6                     
2023-03-25 | 6                     
2023-03-30 | 6                     
(14 rows)


[('2023-01-02', 1),
 ('2023-01-05', 2),
 ('2023-01-20', 3),
 ('2023-01-25', 4),
 ('2023-02-05', 4),
 ('2023-02-10', 4),
 ('2023-02-14', 5),
 ('2023-02-18', 6),
 ('2023-02-28', 6),
 ('2023-03-05', 6),
 ('2023-03-15', 6),
 ('2023-03-20', 6),
 ('2023-03-25', 6),
 ('2023-03-30', 6)]

**Check 1 - the correlated-subquery answer.** Correct but O(dates x rows): for each distinct date, count the
distinct customers that had ordered by then.

In [84]:
assert run(
    """
    SELECT o.order_date,
           (SELECT COUNT(DISTINCT o2.customer_id) FROM orders o2 WHERE o2.order_date <= o.order_date)
    FROM (SELECT DISTINCT order_date FROM orders) o ORDER BY o.order_date
    """
) == [
    ("2023-01-02", 1),
    ("2023-01-05", 2),
    ("2023-01-20", 3),
    ("2023-01-25", 4),
    ("2023-02-05", 4),
    ("2023-02-10", 4),
    ("2023-02-14", 5),
    ("2023-02-18", 6),
    ("2023-02-28", 6),
    ("2023-03-05", 6),
    ("2023-03-15", 6),
    ("2023-03-20", 6),
    ("2023-03-25", 6),
    ("2023-03-30", 6),
]
print("cumulative distinct reaches 6")

cumulative distinct reaches 6


**Check 2 - the scalable window alternative.** Instead of recounting history, find each customer's first order
date and cumulate the count of *new* customers with `SUM(COUNT(*)) OVER (ORDER BY first_date)`.

In [85]:
assert run(
    """
    WITH first_order AS (
        SELECT customer_id, MIN(order_date) AS first_date FROM orders GROUP BY customer_id
    )
    SELECT first_date, COUNT(*), SUM(COUNT(*)) OVER (ORDER BY first_date)
    FROM first_order GROUP BY first_date ORDER BY first_date
    """
) == [
    ("2023-01-02", 1, 1),
    ("2023-01-05", 1, 2),
    ("2023-01-20", 1, 3),
    ("2023-01-25", 1, 4),
    ("2023-02-14", 1, 5),
    ("2023-02-18", 1, 6),
]
print("Q22 passed")

Q22 passed


**Approach and pitfalls.** "Cumulative distinct" is not a window function - `COUNT(DISTINCT ...) OVER (...)` is
not supported in SQLite, and in engines that allow it the frame semantics are subtle. Two safe patterns: a
correlated subquery over distinct dates (correct, but O(dates x rows)), or "count the first appearance of each
customer and cumulate" via `MIN(order_date)` + `SUM(COUNT(*)) OVER (ORDER BY first_date)`. The second pattern is
the one that scales, and it naturally answers "how many customers had we acquired by date X". A common wrong
answer is `SUM(COUNT(DISTINCT customer_id))`, which double-counts customers who order in several periods.

**Follow-ups.**

1. How would you produce a daily active-user count alongside the cumulative distinct count?
2. How would you compute a 7-day rolling distinct customer count?

---
## Interview questions

Short theory questions that interviewers ask alongside the coding exercises. Each one is split into the question
and its model answer - say the answer out loud before you read the next cell.

### T1. What is an index, and what does a B-tree index actually do?

*Answer it out loud before reading on.*

An index is a separate, ordered data structure that maps key values to the physical location of rows. The default
in most relational databases (and in SQLite) is a B-tree: a balanced, high-fanout tree that keeps keys sorted, so
a lookup is O(log n) page reads instead of a full table scan. Leaves are linked, which also makes range scans
(`BETWEEN`, `>`, `ORDER BY`) cheap.

The costs matter as much as the benefits: every index adds space and makes `INSERT`/`UPDATE`/`DELETE` slower
because the index must be maintained. Index the columns you filter and join on most often, and prefer composite
indexes whose column order matches the query's leading predicates (leftmost-prefix rule). An index on
`(dept_id, salary)` can serve `WHERE dept_id = ?` and `WHERE dept_id = ? ORDER BY salary`, but not
`WHERE salary = ?` alone.

### T2. How do you read `EXPLAIN QUERY PLAN`, and what do you look for?

*Answer it out loud before reading on.*

`EXPLAIN QUERY PLAN` shows the access path the optimiser chose for a statement. The two things to look for:

- `SCAN <table>` means the engine reads every row. Fine for small tables, a red flag for large ones.
- `SEARCH <table> USING INDEX ...` means an index restricted the rows. This is what you want on a big table.

Also watch for `USE TEMP B-TREE FOR ORDER BY` (the sort is not covered by an index), nested-loop join order
(wrong order can multiply the work), and a large estimated row count feeding a join. The plan is a *plan*, not a
measurement: use it to find scans, then measure with timing and row counts. Optimisers are cost-based, so stale
statistics (`ANALYZE`) can produce a bad plan.

**Demo 1 - the plan for the duplicate-email query.** An unindexed `GROUP BY email` has no choice but to scan the
whole table, which the plan says explicitly.

In [86]:
print("-- duplicate-email query --")
for row in run("EXPLAIN QUERY PLAN " + """
    SELECT email, COUNT(*) FROM employees GROUP BY email HAVING COUNT(*) > 1
"""):
    print(row)

-- duplicate-email query --
(6, 0, 216, 'SCAN employees')
(8, 0, 0, 'USE TEMP B-TREE FOR GROUP BY')


**Demo 2 - create an index and re-check the plan.** With `(dept_id, salary)` in place, the lookup by `dept_id`
becomes a `SEARCH ... USING INDEX`, and the `ORDER BY salary DESC` can be served by the same index.

In [87]:
run("CREATE INDEX IF NOT EXISTS idx_employees_dept ON employees(dept_id, salary)")
print("-- indexed lookup --")
for row in run("EXPLAIN QUERY PLAN " + """
    SELECT name, salary FROM employees WHERE dept_id = 2 ORDER BY salary DESC
"""):
    print(row)

-- indexed lookup --
(4, 0, 61, 'SEARCH employees USING INDEX idx_employees_dept (dept_id=?)')


**Demo 3 - confirm the index exists.** `sqlite_master` is the catalogue of every created object, which is how you
check what the setup actually built.

In [88]:
assert run(
    "SELECT name FROM sqlite_master WHERE type = 'index' AND name = 'idx_employees_dept'"
) == [("idx_employees_dept",)]
print("index registered")

index registered


### T3. What is normalisation? Explain 1NF, 2NF and 3NF, and when you would denormalise.

*Answer it out loud before reading on.*

Normalisation organises columns and tables to remove redundancy and update anomalies.

- **1NF**: every column holds a single atomic value (no repeating groups or comma-separated lists) and rows are
  unique.
- **2NF**: 1NF plus no partial dependency - every non-key column depends on the *whole* primary key, which only
  matters for composite keys.
- **3NF**: 2NF plus no transitive dependency - non-key columns depend on the key, not on another non-key column
  (for example storing `dept_name` next to `emp_id` when `dept_id` already determines it).

**BCNF** is a stricter 3NF for tables with overlapping candidate keys. You denormalise deliberately for read
performance or simplicity: star schemas keep a dimension's attributes together to avoid joins, and analytics
tables pre-join facts with dimensions. The trade-off is redundant storage and the risk of inconsistent updates,
which is why denormalised layers are usually rebuilt from a normalised source of truth.

**Follow-up.** Where does a star schema sit relative to 3NF? (Answer: a dimension is typically 2NF/3NF-ish, and
the fact table is intentionally denormalised.)

### T4. What does ACID mean, and what are the isolation levels?

*Answer it out loud before reading on.*

- **Atomicity**: all statements in a transaction commit or none do.
- **Consistency**: a transaction moves the database from one valid state to another (constraints hold).
- **Isolation**: concurrent transactions do not see each other's uncommitted work.
- **Durability**: once committed, data survives a crash.

The SQL standard defines four isolation levels, each permitting fewer anomalies:

| level | dirty read | non-repeatable read | phantom read |
| --- | --- | --- | --- |
| Read uncommitted | yes | yes | yes |
| Read committed | no | yes | yes |
| Repeatable read | no | no | yes |
| Serializable | no | no | no |

Most engines default to Read Committed (Postgres, Oracle) or Repeatable Read (MySQL InnoDB); SQLite is
serializable by default and locks at the database or page level. In practice you also meet **snapshot isolation**
and **write skew** (MVCC): Postgres' Repeatable Read is a snapshot isolation level that prevents write skew by
aborting conflicting transactions. Isolation is a correctness-versus-throughput dial: stronger isolation means
more locking or more aborts.

### T5. `GROUP BY` vs window functions: what is the difference?

*Answer it out loud before reading on.*

`GROUP BY` collapses rows: the result has one row per group, so you lose the individual rows. Window functions
keep every row and compute the aggregate (or ranking) across a defined window, which is why they can add
"department average" or "running total" as extra columns.

```sql
-- one row per department
SELECT dept_id, AVG(salary) FROM employees GROUP BY dept_id;

-- every employee keeps their row and gains the department average
SELECT name, salary, AVG(salary) OVER (PARTITION BY dept_id) FROM employees;
```

Windows are evaluated after `WHERE`, `GROUP BY` and `HAVING`, which is why you cannot filter on a window alias in
`WHERE` (use an outer query or `QUALIFY` in BigQuery/Snowflake). Window functions can also rank
(`ROW_NUMBER`/`RANK`/`DENSE_RANK`) and access other rows (`LAG`/`LEAD`, `FIRST_VALUE`), which `GROUP BY` cannot do
without a self-join.

### T6. `UNION` vs `UNION ALL`

*Answer it out loud before reading on.*

`UNION` concatenates two result sets and removes duplicates, which requires a sort or hash and is therefore slower
and memory-hungry. `UNION ALL` simply concatenates, keeping duplicates. The rule of thumb: use `UNION ALL` unless
you specifically need deduplication, which is almost never in ETL where duplicates usually signal a bug rather
than something to hide. Both require compatible column counts and types; only the first query's column names are
used. `INTERSECT` and `EXCEPT` (minus) are the set-difference counterparts.

### T7. Explain `NULL` semantics, and the traps around `NOT IN` and `COUNT`.

*Answer it out loud before reading on.*

`NULL` means "unknown", not "empty". Any comparison with `NULL` yields `UNKNOWN`, and `WHERE` keeps only rows
where the predicate is `TRUE`. So `salary = NULL` and `salary <> NULL` both return nothing; use `IS NULL` /
`IS NOT NULL`. Three-valued logic propagates through `AND`/`OR`: `TRUE OR NULL` is `TRUE`, but `TRUE AND NULL` is
`NULL`.

Traps:

- `NOT IN (subquery)` returns **no rows** if the subquery yields even one `NULL`, because `x <> NULL` is unknown.
  Prefer `NOT EXISTS`.
- `COUNT(*)` counts rows; `COUNT(col)` counts non-null values; `COUNT(DISTINCT col)` ignores nulls too.
- `SUM`/`AVG`/`MIN`/`MAX` skip nulls, so `AVG` over a column with nulls divides by the non-null count. `SUM` of
  no rows is `NULL`, not 0 - wrap it in `COALESCE`.
- `NULL = NULL` is unknown, so joins on nullable keys behave surprisingly.
- Many engines index nulls (Oracle does not by default), and `GROUP BY` treats all nulls as one group.

### T8. `HAVING` vs `WHERE`

*Answer it out loud before reading on.*

`WHERE` filters individual rows *before* grouping and aggregation; `HAVING` filters groups *after* aggregation.
`WHERE status = 'completed'` removes rows, `HAVING COUNT(*) > 1` removes whole groups. Because of that order,
`HAVING` may reference aggregates and `WHERE` may not; conversely, filtering in `WHERE` is usually cheaper
because fewer rows reach the aggregation step. A common interview follow-up is "can you use an aggregate in
`WHERE`?" - no, and the fix is either `HAVING` or a subquery/CTE that computes the aggregate first.

### T9. CTE vs subquery: when do you use each?

*Answer it out loud before reading on.*

A CTE (`WITH name AS (...)`) is a named subquery. It improves readability, lets you reference the same result
more than once, and can be recursive. Whether it is materialised or inlined depends on the engine: Postgres 12+
inlines non-recursive CTEs unless marked `MATERIALIZED`, SQLite may materialise, and BigQuery/Snowflake treat
CTEs as named subqueries. Historically, CTEs were an "optimisation fence" in older Postgres, which is why the old
advice was "CTEs are slow". Use a CTE for clarity or reuse; use a plain subquery for a one-off inline filter. For
hierarchies (org charts, bill of materials, graph traversal), a recursive CTE is the tool:

```sql
WITH RECURSIVE subordinates AS (
    SELECT emp_id, manager_id FROM employees WHERE emp_id = 1
    UNION ALL
    SELECT e.emp_id, e.manager_id
    FROM employees e JOIN subordinates s ON e.manager_id = s.emp_id
)
SELECT * FROM subordinates;
```

### T10. Star vs snowflake schema

*Answer it out loud before reading on.*

A **star schema** has one central fact table (one row per business event, with foreign keys and measures) joined
directly to denormalised dimension tables. It is simple, requires few joins, and is what most BI tools assume. A
**snowflake schema** normalises dimensions into sub-dimensions (product -> category -> department), saving
storage and removing redundancy but requiring more joins.

In practice, star wins for analytics because storage is cheap and joins cost time; snowflaking is worth it when a
dimension is enormous or shared across many marts. The related idea is **one big table (OBT)** / wide denormalised
tables, which push the join cost to the transformation layer. Fact-table design also distinguishes
transaction facts (one row per event), periodic snapshots (one row per entity per period) and accumulating
snapshots (one row per process instance, updated as it moves through stages).

### T11. What is a slowly changing dimension (SCD), and how does Type 2 work?

*Answer it out loud before reading on.*

An SCD is a dimension attribute that changes over time rather than staying fixed. The types:

- **Type 0**: never changes (a true natural key).
- **Type 1**: overwrite in place - no history, simplest, loses the old value.
- **Type 2**: add a new row per version, with `valid_from`/`valid_to` (or `start_date`/`end_date`) and often a
  `is_current` flag and a new surrogate key. History is preserved and facts can be joined to the version that
  was current when the event happened.
- **Type 3**: keep a limited number of previous values as extra columns (`current_region`, `prior_region`).
- **Type 6**: a hybrid of 1, 2 and 3.

Type 2 is the standard interview answer. The implementation closes the current row (`valid_to = today`,
`is_current = false`) and inserts a new row with `valid_from = today`. Getting the boundaries right matters:
`valid_to` is usually exclusive (`[valid_from, valid_to)`), and a common bug is overlapping or gapped ranges.
Loading Type 2 with a merge/incremental model (`dbt snapshot` is the classic tool) is the day-to-day reality.

### T12. What is partitioning, and how does it help?

*Answer it out loud before reading on.*

Partitioning physically splits a large table into smaller pieces by a key - usually a date range, or a hash or
list of keys - while queries still see one logical table. The main win is **partition pruning**: a query filtered
on the partition key reads only the relevant partitions instead of the whole table. It also enables partition-wise
maintenance (drop an old partition instead of `DELETE`ing rows) and parallel scans.

Things to know: the partition key should appear in the `WHERE` clause of the important queries, or pruning cannot
happen; too many small partitions hurt (metadata overhead); and partitioning is not the same as clustering or
bucketing, which order rows *within* a partition. Many warehouses combine both: partition by day, cluster by
customer id.

### T13. How do you approach query optimisation?

*Answer it out loud before reading on.*

A repeatable checklist beats guessing:

1. **Measure first.** Get the actual plan and timings, not a hunch.
2. **Read the plan** (`EXPLAIN` / `EXPLAIN ANALYZE`, `EXPLAIN QUERY PLAN` in SQLite). Look for full scans on big
   tables, bad join orders, and spills/sorts.
3. **Reduce rows early.** Push `WHERE` predicates down, filter before joining, and select only the columns you
   need - `SELECT *` blocks covering indexes and moves data you will never read.
4. **Index for the access pattern** you actually measured: composite indexes matching the leading predicates and
   the `ORDER BY`; avoid indexing low-cardinality columns on their own.
5. **Fix the join strategy** where needed: statistics (`ANALYZE`), correct data types, and explicit join keys.
6. **Pre-aggregate or materialise** expensive repeated work; incremental models instead of full rebuilds.
7. **Partition and prune** large tables.
8. **Verify with the new plan and timings**, and watch the write cost you added.

Common anti-patterns: functions on the indexed column (`WHERE date(created_at) = ...`), implicit type casts,
`SELECT DISTINCT` to paper over a bad join, and `NOT IN` with nullable subqueries.

### T14. Clustered vs non-clustered indexes, and what is a covering index?

*Answer it out loud before reading on.*

A **clustered** index defines the physical order of the table's rows - there can be only one, and in InnoDB and
SQL Server the primary key is clustered by default. Because rows are stored in key order, range scans on the
clustered key are very fast, and the index *is* the data (no extra lookup). A **non-clustered** index is a
separate structure: it stores the key plus a pointer (row id / primary key) back to the row, so a lookup may need
a second read, called a **key lookup** or **bookmark lookup**.

A **covering index** contains all the columns a query needs, so the engine can answer it from the index alone -
this is the "index-only scan". For example, an index on `(dept_id, salary)` covers
`SELECT salary FROM employees WHERE dept_id = ?` without touching the table. In Postgres you make this explicit
with `INCLUDE` columns, and in InnoDB the primary key is implicitly appended to every secondary index, which is
why keeping the primary key narrow matters. Trade-offs are the same as any index: extra storage and slower
writes.

---
## Where to go next

You have now seen the 22 coding questions, the `EXPLAIN QUERY PLAN` walkthrough and the 14 theory questions.
To make the practice stick: re-run the notebook after changing the seed data (add a third duplicate email, or a
customer with only cancelled orders) and see which assertions fail - that is exactly the edge-case thinking an
interviewer is probing for.

## SQL exercises

Eight short exercises on planning, cost estimation, and semantics. Each one gives the task, a hint, and the expected result, then a runnable Python cell that checks the answer with `assert`. Only the standard library is used, so each cell runs on its own.

### Try it 1. Choose a join strategy from table sizes

**Task.** Write `choose_join(outer_rows, inner_rows, inner_indexed, equi, pre_sorted)` that returns `"nested loop"`, `"hash"`, or `"merge"`.

**Hint.** A small outer side with an indexed inner side favours a nested loop. Already-sorted inputs favour a merge. Everything else large and equal favours a hash join. A non-equi join cannot hash, so fall back to nested loop or merge.

**Expected result.** A 100-row outer join to a 5,000,000-row indexed inner is a nested loop. A 5,000,000-row outer join to a 50,000-row unindexed inner is a hash join. Sorted inputs and non-equi joins return merge.

In [99]:
def choose_join(outer_rows, inner_rows, inner_indexed, equi=True, pre_sorted=False):
    if pre_sorted:
        return "merge"
    if not equi:
        return "nested loop" if inner_indexed and outer_rows <= 10_000 else "merge"
    if outer_rows <= 1_000 and inner_indexed:
        return "nested loop"
    return "hash"


assert choose_join(100, 5_000_000, True) == "nested loop"
assert choose_join(5_000_000, 50_000, False) == "hash"
assert choose_join(1_000_000, 1_000_000, False, equi=False) == "merge"
assert choose_join(2_000_000, 2_000_000, False, pre_sorted=True) == "merge"
print("join strategies chosen for four shapes")

join strategies chosen for four shapes


### Try it 2. Estimate rows after a multi-join

**Task.** Start from a 10,000,000-row fact table. A date filter keeps 20% of the rows. Join a country dimension that keeps 10% of its rows and a category dimension that keeps 5%. Estimate the result size.

**Hint.** For inner joins on unique dimension keys, multiply the fact rows by every filter's selectivity: `rows * s_fact * s_country * s_category`. Filtering the dimensions first shrinks the join inputs.

**Expected result.** `10_000_000 * 0.2 * 0.1 * 0.05 = 10_000` rows.

In [100]:
def estimate_rows(fact_rows, fact_selectivity, dimension_selectivities):
    rows = fact_rows * fact_selectivity
    for s in dimension_selectivities:
        rows *= s
    return int(rows)


estimate = estimate_rows(10_000_000, 0.2, [0.1, 0.05])
print("estimated result rows:", estimate)
assert estimate == 10_000

unfiltered = estimate_rows(10_000_000, 1.0, [1.0, 1.0])
assert unfiltered == 10_000_000
print("filters cut the estimate from", unfiltered, "to", estimate)

estimated result rows: 10000
filters cut the estimate from 10000000 to 10000


### Try it 3. Decide sargable versus non-sargable

**Task.** Write `is_sargable(predicate)` that returns `True` when the predicate can use an index on the column, and classify eight example predicates.

**Hint.** Reject predicates that wrap the column in a function, start a `LIKE` with `%`, or do arithmetic between the column and a constant on the comparison side.

**Expected result.** Ranges, anchored prefixes, `IN`, and comparisons against arithmetic on the constant side are sargable. `YEAR(col) = ...`, `UPPER(col) = ...`, `col * 12 > ...`, and `LIKE '%...'` are not.

In [101]:
import re

NON_SARGABLE_FUNCS = (
    "year", "month", "day", "date", "upper", "lower",
    "substr", "substring", "cast", "coalesce", "strftime",
)


def is_sargable(predicate):
    p = predicate.lower()
    if re.search(r"\b(" + "|".join(NON_SARGABLE_FUNCS) + r")\s*\(", p):
        return False
    if re.search(r"like\s*'%", p):
        return False
    if re.search(r"[a-z_]+\s*[*+/-]\s*\d", p):
        return False
    return True


cases = [
    ("hire_date >= '2023-01-01'", True),
    ("YEAR(hire_date) = 2023", False),
    ("email LIKE 'a%@example.com'", True),
    ("email LIKE '%@example.com'", False),
    ("salary > 100000 / 12.0", True),
    ("salary * 12 > 100000", False),
    ("UPPER(country) = 'US'", False),
    ("customer_id IN (1, 2, 3)", True),
]

for predicate, expected in cases:
    got = is_sargable(predicate)
    print(f"{'sargable' if got else 'blocked ':9s} {predicate}")
    assert got is expected, predicate

print("all eight predicates classified correctly")

sargable  hire_date >= '2023-01-01'
blocked   YEAR(hire_date) = 2023
sargable  email LIKE 'a%@example.com'
blocked   email LIKE '%@example.com'
sargable  salary > 100000 / 12.0
blocked   salary * 12 > 100000
blocked   UPPER(country) = 'US'
sargable  customer_id IN (1, 2, 3)
all eight predicates classified correctly


### Try it 4. Compute index selectivity from distinct counts

**Task.** For a 1,000,000-row table, compute the selectivity and average rows returned per equality for three candidate columns, then name the most selective one.

**Hint.** Selectivity is `distinct / rows`; average rows per key is `rows / distinct`. Higher selectivity means a smaller and better index candidate for an equality predicate.

**Expected result.** `customer_id` (900,000 distinct) is the most selective. `status` (4 distinct) is the least and would return about 250,000 rows per value, so an index on it alone is rarely worth it.

In [102]:
def selectivity(rows, distinct_values):
    return distinct_values / rows


def rows_per_key(rows, distinct_values):
    return rows / distinct_values


rows = 1_000_000
columns = {"customer_id": 900_000, "order_date": 1_095, "status": 4}

for name, distinct in columns.items():
    print(
        f"{name:12s} selectivity={selectivity(rows, distinct):.6f} "
        f"rows per key={rows_per_key(rows, distinct):.1f}"
    )

best = max(columns, key=lambda name: selectivity(rows, columns[name]))
worst = min(columns, key=lambda name: selectivity(rows, columns[name]))
assert best == "customer_id"
assert worst == "status"
assert rows_per_key(rows, columns["status"]) == 250_000.0
print("index", best, "first; skip", worst, "until a covering index can help")

customer_id  selectivity=0.900000 rows per key=1.1
order_date   selectivity=0.001095 rows per key=913.2
status       selectivity=0.000004 rows per key=250000.0
index customer_id first; skip status until a covering index can help


### Try it 5. Order a composite index from query patterns

**Task.** Write `index_for_query(equality_cols, range_cols, order_cols)` that returns the column order for a composite index, then apply it to a query that filters `customer_id` and `status`, ranges on `order_date`, and orders by `amount`.

**Hint.** Equality columns first, then the range column, then columns used only for ordering or covering. Deduplicate while preserving order.

**Expected result.** `['customer_id', 'status', 'order_date', 'amount']`.

In [103]:
def index_for_query(equality_cols, range_cols, order_cols):
    ordered = []
    for column in list(equality_cols) + list(range_cols) + list(order_cols):
        if column not in ordered:
            ordered.append(column)
    return ordered


index = index_for_query(["customer_id", "status"], ["order_date"], ["amount"])
print("composite index order:", index)
assert index == ["customer_id", "status", "order_date", "amount"]

# A range column placed before an equality column is a common mistake.
wrong = index_for_query(["customer_id"], ["order_date", "status"], [])
assert wrong == ["customer_id", "order_date", "status"]
assert index[:2] == ["customer_id", "status"]
print("equality columns lead, then the single range column")

composite index order: ['customer_id', 'status', 'order_date', 'amount']
equality columns lead, then the single range column


### Try it 6. Detect a fan-out from join keys

**Task.** Build a small `customers` and `orders` table in SQLite. Decide whether joining multiplies rows, and prove it by comparing the joined row count with the base tables.

**Hint.** A join key that is unique on the right side preserves the left row count. If the right side has fewer distinct keys than rows, the join fans out. Compare `COUNT(*)` with `COUNT(DISTINCT key)`.

**Expected result.** `customer_id` is unique in `customers` but not in `orders`, so joining fans out: four orders become four joined rows even though there are only three customers.

In [104]:
import sqlite3

con = sqlite3.connect(":memory:")
con.executescript(
    '''
    CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT);
    CREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER);
    INSERT INTO customers VALUES (1, 'Ada'), (2, 'Bret'), (3, 'Chen');
    INSERT INTO orders VALUES (10, 1), (11, 1), (12, 2), (13, 3);
    '''
)


def is_unique(table, column):
    rows = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    distinct = con.execute(
        f"SELECT COUNT(DISTINCT {column}) FROM {table}"
    ).fetchone()[0]
    return rows == distinct


joined = con.execute(
    "SELECT COUNT(*) FROM customers c JOIN orders o ON o.customer_id = c.customer_id"
).fetchone()[0]
print("customers:", 3, "orders:", 4, "joined rows:", joined)
assert is_unique("customers", "customer_id")
assert not is_unique("orders", "customer_id")
assert joined == 4 > 3
print("the join fans out because customer_id is not unique in orders")

customers: 3 orders: 4 joined rows: 4
the join fans out because customer_id is not unique in orders


### Try it 7. Plan a window-function alternative to a self-join

**Task.** Compute a running total per customer two ways: with an O(n squared) self-join, and with a single ordered pass that plays the role of a window function. Prove the results match and compare the work.

**Hint.** The self-join compares every row with every earlier row in the group. The window alternative sorts once and keeps one accumulator per group, which is a single pass.

**Expected result.** Both approaches return the same running totals. The self-join touches `n * n` pairs; the window touches `n` rows.

In [105]:
rows = [
    ("C1", "2023-01-01", 10.0),
    ("C2", "2023-01-01", 5.0),
    ("C1", "2023-01-02", 20.0),
    ("C1", "2023-01-03", 30.0),
    ("C2", "2023-01-04", 15.0),
]


def running_total_self_join(data):
    pairs = 0
    result = []
    for customer, day, amount in data:
        total = 0.0
        for other_customer, other_day, other_amount in data:
            pairs += 1
            if other_customer == customer and other_day <= day:
                total += other_amount
        result.append((customer, day, total))
    return result, pairs


def running_total_window(data):
    touched = 0
    totals = {}
    result = []
    for customer, day, amount in sorted(data, key=lambda row: row[1]):
        touched += 1
        totals[customer] = totals.get(customer, 0.0) + amount
        result.append((customer, day, totals[customer]))
    return result, touched


self_result, pairs = running_total_self_join(rows)
window_result, touched = running_total_window(rows)
print("self-join pairs:", pairs, "window rows:", touched)
assert self_result == window_result
assert pairs == len(rows) ** 2
assert touched == len(rows)
print("same answer,", pairs, "comparisons versus", touched, "updates")

self-join pairs: 25 window rows: 5
same answer, 25 comparisons versus 5 updates


### Try it 8. Compute NULL semantics of a predicate

**Task.** Implement three-valued `AND`, `OR`, and `NOT` for `True`, `False`, and `None`, then show that a single NULL in a `NOT IN` list makes the whole predicate unknown.

**Hint.** `NULL AND FALSE` is `FALSE`, but `NULL AND TRUE` is `NULL`. `NULL OR TRUE` is `TRUE`, but `NULL OR FALSE` is `NULL`. `NOT NULL` stays `NULL`. Only rows where the predicate is `TRUE` survive a `WHERE`.

**Expected result.** The truth tables behave as above, and `x NOT IN (1, NULL)` returns no rows even though `x NOT IN (1)` returns the non-matching rows.

In [106]:
import sqlite3


def sql_and(a, b):
    if a is False or b is False:
        return False
    if a is None or b is None:
        return None
    return True


def sql_or(a, b):
    if a is True or b is True:
        return True
    if a is None or b is None:
        return None
    return False


def sql_not(a):
    return None if a is None else (not a)


assert sql_and(None, False) is False
assert sql_and(None, True) is None
assert sql_or(None, False) is None
assert sql_or(None, True) is True
assert sql_not(None) is None
print("three-valued logic checks passed")

con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE t (x INTEGER)")
con.executemany("INSERT INTO t VALUES (?)", [(1,), (2,), (None,)])
safe = con.execute("SELECT x FROM t WHERE x NOT IN (1)").fetchall()
unsafe = con.execute("SELECT x FROM t WHERE x NOT IN (1, NULL)").fetchall()
print("NOT IN (1)      ->", safe)
print("NOT IN (1, NULL)->", unsafe)
assert safe == [(2,)]
assert unsafe == []
print("a NULL in the list turns every comparison unknown")

three-valued logic checks passed
NOT IN (1)      -> [(2,)]
NOT IN (1, NULL)-> []
a NULL in the list turns every comparison unknown
